In [1]:
# Cell 1: Initialize the S3 environment, retrieve Hugging Face credentials,
# and locate the Phase 1 Architecture C checkpoint and tokenizer.

import os
import random
from pathlib import Path

import numpy as np
import torch

# --------------------------------------------------
# 1. Reproducibility
# --------------------------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# --------------------------------------------------
# 2. Select the execution device
# --------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 65)
print("S3: ARCHITECTURE C (LSTM) — ENVIRONMENT INITIALIZATION")
print("=" * 65)
print("PyTorch version:", torch.__version__)
print("Execution device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU unavailable; execution will use the CPU.")

# --------------------------------------------------
# 3. Retrieve the Kaggle Hugging Face secret
# --------------------------------------------------
try:
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF")

    if not HF_TOKEN or not HF_TOKEN.strip():
        raise ValueError("The Kaggle secret 'HF' is empty.")

    # Make the token available to Hugging Face Hub clients.
    # Never print or display the token.
    os.environ["HF_TOKEN"] = HF_TOKEN

    print("\nHugging Face token: retrieved successfully.")

except Exception as exc:
    raise RuntimeError(
        "Could not retrieve the Kaggle secret named 'HF'. "
        "In Kaggle, add your Hugging Face token under Secrets "
        "with the exact name HF and enable notebook access."
    ) from exc

# --------------------------------------------------
# 4. Locate the mounted Phase 1 Architecture C dataset
# --------------------------------------------------
INPUT_ROOT = Path("/kaggle/input")

if not INPUT_ROOT.exists():
    raise FileNotFoundError(
        "The Kaggle input directory /kaggle/input does not exist."
    )

checkpoint_matches = sorted(
    INPUT_ROOT.rglob("model_C_final.pt")
)

if not checkpoint_matches:
    raise FileNotFoundError(
        "Could not find model_C_final.pt under /kaggle/input. "
        "Confirm that the Phase 1 Architecture C dataset is attached "
        "to this notebook."
    )

if len(checkpoint_matches) > 1:
    preferred = [
        path for path in checkpoint_matches
        if path.parent.name.lower() == "architecture_c"
    ]

    if len(preferred) == 1:
        checkpoint_matches = preferred
    else:
        raise RuntimeError(
            "Multiple Architecture C final checkpoints were found. "
            "Inspect the paths before selecting one:\n"
            + "\n".join(str(p) for p in checkpoint_matches)
        )

ARCH_C_FINAL_CHECKPOINT = checkpoint_matches[0]
ARCH_C_ROOT = ARCH_C_FINAL_CHECKPOINT.parent

# Locate the exact tokenizer artifact attached in Kaggle Inputs.
tokenizer_matches = sorted(
    path for path in INPUT_ROOT.rglob("tokenizer.json")
    if path.parent.name.lower() == "tokenizer"
    and "cs781-phase1-tokenized-corpus" in str(path).lower()
)

if len(tokenizer_matches) != 1:
    raise RuntimeError(
        "Expected exactly one tokenizer.json inside the attached "
        "cs781-phase1-tokenized-corpus/tokenizer directory, but found "
        f"{len(tokenizer_matches)}:\n"
        + "\n".join(str(p) for p in tokenizer_matches)
    )

PHASE1_TOKENIZER_PATH = tokenizer_matches[0]

# --------------------------------------------------
# 5. Display verified paths without loading or changing weights
# --------------------------------------------------
print("\nPhase 1 Architecture C directory:", ARCH_C_ROOT)
print("Final checkpoint:", ARCH_C_FINAL_CHECKPOINT)
print("Phase 1 tokenizer:", PHASE1_TOKENIZER_PATH)

print("\nCheckpoint size:",
      f"{ARCH_C_FINAL_CHECKPOINT.stat().st_size:,} bytes")
print("Tokenizer size:",
      f"{PHASE1_TOKENIZER_PATH.stat().st_size:,} bytes")

print("\nCell 1 completed. No model training has started.")

S3: ARCHITECTURE C (LSTM) — ENVIRONMENT INITIALIZATION
PyTorch version: 2.11.0+cu128
Execution device: cuda
GPU: Tesla T4

Hugging Face token: retrieved successfully.

Phase 1 Architecture C directory: /kaggle/input/datasets/rohitbatra2004/phase133/architecture_C
Final checkpoint: /kaggle/input/datasets/rohitbatra2004/phase133/architecture_C/model_C_final.pt
Phase 1 tokenizer: /kaggle/input/datasets/rohitbatra2004/cs781-phase1-tokenized-corpus/tokenizer/tokenizer.json

Checkpoint size: 413,624,316 bytes
Tokenizer size: 3,587,785 bytes

Cell 1 completed. No model training has started.


In [2]:
# Cell 2: Download the Phase 2 L-NER dataset from Hugging Face
# using the Kaggle HF token, then verify every required input file.

from pathlib import Path
from huggingface_hub import snapshot_download

# --------------------------------------------------
# 1. Define the dataset and local download directory
# --------------------------------------------------
PHASE2_REPO_ID = "Exploration-Lab/CS-781-Capstone"
PHASE2_ROOT = Path("/kaggle/working/phase2_lner")
PHASE2_DATA_DIR = PHASE2_ROOT / "Phase_2_LNER"

PHASE2_ROOT.mkdir(parents=True, exist_ok=True)

print("=" * 65)
print("S3: PHASE 2 L-NER DATASET DOWNLOAD")
print("=" * 65)
print("Repository:", PHASE2_REPO_ID)
print("Destination:", PHASE2_ROOT)

# --------------------------------------------------
# 2. Download only the Phase 2 L-NER directory
# --------------------------------------------------
snapshot_download(
    repo_id=PHASE2_REPO_ID,
    repo_type="dataset",
    token=HF_TOKEN,
    local_dir=str(PHASE2_ROOT),
    allow_patterns=["Phase_2_LNER/**"],
)

print("\nDataset download completed.")

# --------------------------------------------------
# 3. Verify the required dataset files
# --------------------------------------------------
required_files = [
    PHASE2_DATA_DIR / "Train" / "fold1-text.json",
    PHASE2_DATA_DIR / "Train" / "fold1-gold.json",
    PHASE2_DATA_DIR / "Train" / "fold2-text.json",
    PHASE2_DATA_DIR / "Train" / "fold2-gold.json",
    PHASE2_DATA_DIR / "Train" / "labels.txt",
    PHASE2_DATA_DIR / "Test" / "fold3-text.json",
]

missing_files = [
    path for path in required_files
    if not path.is_file() or path.stat().st_size == 0
]

if missing_files:
    raise FileNotFoundError(
        "The following required files are missing or empty:\n"
        + "\n".join(str(path) for path in missing_files)
    )

print("\nVerified files:")
for path in required_files:
    print(
        f"OK | {path.relative_to(PHASE2_ROOT)} "
        f"| {path.stat().st_size:,} bytes"
    )

# --------------------------------------------------
# 4. Report the verified training and test locations
# --------------------------------------------------
TRAIN_DIR = PHASE2_DATA_DIR / "Train"
TEST_DIR = PHASE2_DATA_DIR / "Test"

print("\nTraining directory:", TRAIN_DIR)
print("Test directory:", TEST_DIR)
print("\nCell 2 completed. No model training has started.")

S3: PHASE 2 L-NER DATASET DOWNLOAD
Repository: Exploration-Lab/CS-781-Capstone
Destination: /kaggle/working/phase2_lner


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]


Dataset download completed.

Verified files:
OK | Phase_2_LNER/Train/fold1-text.json | 1,106,635 bytes
OK | Phase_2_LNER/Train/fold1-gold.json | 460,581 bytes
OK | Phase_2_LNER/Train/fold2-text.json | 1,124,295 bytes
OK | Phase_2_LNER/Train/fold2-gold.json | 475,489 bytes
OK | Phase_2_LNER/Train/labels.txt | 71 bytes
OK | Phase_2_LNER/Test/fold3-text.json | 1,186,648 bytes

Training directory: /kaggle/working/phase2_lner/Phase_2_LNER/Train
Test directory: /kaggle/working/phase2_lner/Phase_2_LNER/Test

Cell 2 completed. No model training has started.


In [3]:
# Cell 3: Load and audit Phase 2 L-NER data, entity labels,
# document IDs, annotation structure, span boundaries, and offset conventions.

import json
from collections import Counter
from pathlib import Path

# --------------------------------------------------
# 1. Load all required JSON files
# --------------------------------------------------
def load_json_file(path):
    """Load a UTF-8 JSON file and report a clear error if parsing fails."""
    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(f"Required file not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


fold1_text = load_json_file(TRAIN_DIR / "fold1-text.json")
fold1_gold = load_json_file(TRAIN_DIR / "fold1-gold.json")
fold2_text = load_json_file(TRAIN_DIR / "fold2-text.json")
fold2_gold = load_json_file(TRAIN_DIR / "fold2-gold.json")
fold3_text = load_json_file(TEST_DIR / "fold3-text.json")

with (TRAIN_DIR / "labels.txt").open("r", encoding="utf-8") as f:
    LABELS = [line.strip() for line in f if line.strip()]

# --------------------------------------------------
# 2. Validate top-level types and label inventory
# --------------------------------------------------
for variable_name, data in [
    ("fold1_text", fold1_text),
    ("fold1_gold", fold1_gold),
    ("fold2_text", fold2_text),
    ("fold2_gold", fold2_gold),
    ("fold3_text", fold3_text),
]:
    if not isinstance(data, dict):
        raise TypeError(
            f"{variable_name} should be a dictionary keyed by document ID; "
            f"got {type(data).__name__}."
        )

if not LABELS:
    raise ValueError("labels.txt contains no entity labels.")

if len(LABELS) != len(set(LABELS)):
    raise ValueError(f"Duplicate labels found in labels.txt: {LABELS}")

# --------------------------------------------------
# 3. Verify document IDs and document text types
# --------------------------------------------------
def verify_text_documents(name, documents):
    bad_docs = [
        doc_id for doc_id, text_value in documents.items()
        if not isinstance(text_value, str)
    ]

    if bad_docs:
        raise TypeError(
            f"{name} contains non-string document text. "
            f"Examples: {bad_docs[:5]}"
        )

    if not documents:
        raise ValueError(f"{name} contains no documents.")

    print(f"{name}: {len(documents)} documents; all texts are strings.")


verify_text_documents("Fold 1", fold1_text)
verify_text_documents("Fold 2", fold2_text)
verify_text_documents("Fold 3 test", fold3_text)

if set(fold1_text) != set(fold1_gold):
    raise ValueError(
        "Fold 1 text and gold document IDs do not match. "
        f"Text-only IDs: {list(set(fold1_text) - set(fold1_gold))[:5]}; "
        f"gold-only IDs: {list(set(fold1_gold) - set(fold1_text))[:5]}"
    )

if set(fold2_text) != set(fold2_gold):
    raise ValueError(
        "Fold 2 text and gold document IDs do not match. "
        f"Text-only IDs: {list(set(fold2_text) - set(fold2_gold))[:5]}; "
        f"gold-only IDs: {list(set(fold2_gold) - set(fold2_text))[:5]}"
    )

if set(fold1_text) & set(fold2_text):
    raise ValueError("Fold 1 and Fold 2 contain overlapping document IDs.")

print("\nEntity labels:", LABELS)
print("Number of entity labels:", len(LABELS))

# --------------------------------------------------
# 4. Audit gold annotation structure and offsets
# Expected candidate format: [start, end, label, annotated_text]
# We inspect offset conventions rather than assuming them.
# --------------------------------------------------
def audit_annotations(name, documents, gold):
    label_counts = Counter()
    annotation_count = 0
    invalid_count = 0
    unknown_label_count = 0
    out_of_bounds_count = 0
    annotated_text_matches_exclusive = 0
    annotated_text_matches_inclusive = 0
    examples = []

    for doc_id, text_value in documents.items():
        annotations = gold[doc_id]

        if not isinstance(annotations, list):
            raise TypeError(
                f"{name}, document {doc_id}: annotations must be a list; "
                f"got {type(annotations).__name__}."
            )

        for index, annotation in enumerate(annotations):
            annotation_count += 1

            if not isinstance(annotation, (list, tuple)) or len(annotation) < 3:
                invalid_count += 1
                if invalid_count <= 5:
                    print(
                        f"Malformed annotation in {name}, document {doc_id}, "
                        f"index {index}: {repr(annotation)[:200]}"
                    )
                continue

            start, end, label = annotation[:3]

            if (
                isinstance(start, bool)
                or isinstance(end, bool)
                or not isinstance(start, int)
                or not isinstance(end, int)
                or not isinstance(label, str)
            ):
                invalid_count += 1
                if invalid_count <= 5:
                    print(
                        f"Invalid field types in {name}, document {doc_id}: "
                        f"{repr(annotation)[:200]}"
                    )
                continue

            label_counts[label] += 1

            if label not in LABELS:
                unknown_label_count += 1

            if not (0 <= start <= end <= len(text_value)):
                out_of_bounds_count += 1

            if len(annotation) >= 4 and isinstance(annotation[3], str):
                annotated_text = annotation[3]

                if text_value[start:end] == annotated_text:
                    annotated_text_matches_exclusive += 1

                if end < len(text_value) and text_value[start:end + 1] == annotated_text:
                    annotated_text_matches_inclusive += 1

            if len(examples) < 5:
                examples.append({
                    "doc_id": doc_id,
                    "annotation": annotation,
                    "text_slice_start_end": text_value[start:end],
                    "text_slice_start_end_plus_one": (
                        text_value[start:end + 1]
                        if end < len(text_value) else None
                    ),
                })

    print(f"\n{name} annotation audit")
    print("-" * 55)
    print("Documents:", len(documents))
    print("Total annotations:", annotation_count)
    print("Malformed/invalid annotations:", invalid_count)
    print("Annotations with labels outside labels.txt:", unknown_label_count)
    print("Out-of-bounds spans:", out_of_bounds_count)
    print("Annotated text matches text[start:end]:", annotated_text_matches_exclusive)
    print("Annotated text matches text[start:end+1]:", annotated_text_matches_inclusive)
    print("Annotation counts by label:", dict(label_counts))

    if examples:
        print("\nRepresentative annotation examples:")
        for example in examples[:3]:
            print(json.dumps(example, ensure_ascii=False, default=str))

    return {
        "documents": len(documents),
        "annotations": annotation_count,
        "label_counts": label_counts,
        "invalid": invalid_count,
        "unknown_labels": unknown_label_count,
        "out_of_bounds": out_of_bounds_count,
        "exclusive_matches": annotated_text_matches_exclusive,
        "inclusive_matches": annotated_text_matches_inclusive,
    }


AUDIT_FOLD1 = audit_annotations("Fold 1", fold1_text, fold1_gold)
AUDIT_FOLD2 = audit_annotations("Fold 2", fold2_text, fold2_gold)

# --------------------------------------------------
# 5. Summarize dataset sizes and combined label counts
# --------------------------------------------------
COMBINED_LABEL_COUNTS = (
    AUDIT_FOLD1["label_counts"] + AUDIT_FOLD2["label_counts"]
)

print("\n" + "=" * 65)
print("S3 DATA AUDIT SUMMARY")
print("=" * 65)
print("Fold 1 documents:", len(fold1_text))
print("Fold 2 documents:", len(fold2_text))
print("Fold 3 test documents:", len(fold3_text))
print("Combined training documents:", len(fold1_text) + len(fold2_text))
print("Unique entity labels:", len(LABELS))
print("Combined annotations:", AUDIT_FOLD1["annotations"] + AUDIT_FOLD2["annotations"])
print("Combined counts by label:", dict(COMBINED_LABEL_COUNTS))

print("\nCell 3 completed. No model training has started.")

Fold 1: 35 documents; all texts are strings.
Fold 2: 35 documents; all texts are strings.
Fold 3 test: 35 documents; all texts are strings.

Entity labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Number of entity labels: 12

Fold 1 annotation audit
-------------------------------------------------------
Documents: 35
Total annotations: 4120
Malformed/invalid annotations: 0
Annotations with labels outside labels.txt: 0
Out-of-bounds spans: 0
Annotated text matches text[start:end]: 4120
Annotated text matches text[start:end+1]: 0
Annotation counts by label: {'RESP': 191, 'DATE': 813, 'COURT': 455, 'WIT': 67, 'APP': 207, 'STAT': 525, 'A.COUNSEL': 70, 'PREC': 624, 'JUDGE': 123, 'R.COUNSEL': 87, 'CASENO': 423, 'AUTH': 535}

Representative annotation examples:
{"doc_id": "115651329", "annotation": [137, 153, "RESP", "STATE OF HARYANA"], "text_slice_start_end": "STATE OF HARYANA", "text_slice_start_end_plus_one": "STATE OF H

In [4]:
# Cell 4: Create a reproducible document-level train/validation split,
# load the Phase 1 tokenizer, and audit token-to-character boundary alignment.

from collections import Counter
from pathlib import Path

import numpy as np
from sklearn.model_selection import train_test_split
from tokenizers import Tokenizer

# --------------------------------------------------
# 1. Combine the two labeled training folds
# --------------------------------------------------
ALL_TEXTS = {**fold1_text, **fold2_text}
ALL_GOLD = {**fold1_gold, **fold2_gold}

if set(ALL_TEXTS) != set(ALL_GOLD):
    raise ValueError("Combined training text and annotation IDs do not match.")

ALL_DOC_IDS = sorted(ALL_TEXTS.keys())

# Split by document, never by individual sentence or annotation.
TRAIN_DOC_IDS, VAL_DOC_IDS = train_test_split(
    ALL_DOC_IDS,
    test_size=0.20,
    random_state=SEED,
    shuffle=True,
)

TRAIN_DOC_IDS = sorted(TRAIN_DOC_IDS)
VAL_DOC_IDS = sorted(VAL_DOC_IDS)

if set(TRAIN_DOC_IDS) & set(VAL_DOC_IDS):
    raise RuntimeError("Train/validation document leakage detected.")

print("=" * 65)
print("S3: DOCUMENT SPLIT")
print("=" * 65)
print("All labeled documents:", len(ALL_DOC_IDS))
print("Training documents:", len(TRAIN_DOC_IDS))
print("Validation documents:", len(VAL_DOC_IDS))
print("Training annotations:", sum(len(ALL_GOLD[d]) for d in TRAIN_DOC_IDS))
print("Validation annotations:", sum(len(ALL_GOLD[d]) for d in VAL_DOC_IDS))

# --------------------------------------------------
# 2. Load the exact Phase 1 tokenizer
# --------------------------------------------------
tokenizer = Tokenizer.from_file(str(PHASE1_TOKENIZER_PATH))

TOKENIZER_VOCAB_SIZE = tokenizer.get_vocab_size(with_added_tokens=True)

print("\n" + "=" * 65)
print("S3: PHASE 1 TOKENIZER")
print("=" * 65)
print("Tokenizer path:", PHASE1_TOKENIZER_PATH)
print("Vocabulary size:", TOKENIZER_VOCAB_SIZE)

if TOKENIZER_VOCAB_SIZE != 50_000:
    print(
        "WARNING: Tokenizer vocabulary is not 50,000. "
        "We must verify the checkpoint configuration before building the model."
    )

# --------------------------------------------------
# 3. Measure token lengths and inspect tokenizer offsets
# --------------------------------------------------
token_lengths = []
invalid_offsets = 0
tokens_with_empty_offsets = 0

for doc_id in ALL_DOC_IDS:
    text_value = ALL_TEXTS[doc_id]
    encoding = tokenizer.encode(text_value, add_special_tokens=False)

    token_lengths.append(len(encoding.ids))

    for start, end in encoding.offsets:
        if start == end:
            tokens_with_empty_offsets += 1
        elif not (0 <= start < end <= len(text_value)):
            invalid_offsets += 1

print("\nTokenized document length statistics:")
print("Minimum:", int(np.min(token_lengths)))
print("Median:", int(np.median(token_lengths)))
print("90th percentile:", int(np.percentile(token_lengths, 90)))
print("95th percentile:", int(np.percentile(token_lengths, 95)))
print("Maximum:", int(np.max(token_lengths)))
print("Tokens with empty offsets:", tokens_with_empty_offsets)
print("Invalid non-empty offsets:", invalid_offsets)

if invalid_offsets:
    raise ValueError(
        "Tokenizer returned invalid non-empty character offsets. "
        "Do not proceed with NER target construction."
    )

# --------------------------------------------------
# 4. Measure gold entity-boundary alignment
# A span is fully aligned when its start and end exactly
# match boundaries represented by tokenizer offsets.
# --------------------------------------------------
boundary_totals = Counter()
unaligned_examples = []

for doc_id in ALL_DOC_IDS:
    text_value = ALL_TEXTS[doc_id]
    encoding = tokenizer.encode(text_value, add_special_tokens=False)

    token_spans = [
        (start, end)
        for start, end in encoding.offsets
        if start < end
    ]

    token_starts = {start for start, _ in token_spans}
    token_ends = {end for _, end in token_spans}

    for annotation in ALL_GOLD[doc_id]:
        start, end, label = annotation[:3]

        boundary_totals["entities"] += 1

        start_aligned = start in token_starts
        end_aligned = end in token_ends

        boundary_totals["start_aligned"] += int(start_aligned)
        boundary_totals["end_aligned"] += int(end_aligned)
        boundary_totals["both_aligned"] += int(
            start_aligned and end_aligned
        )

        if not (start_aligned and end_aligned) and len(unaligned_examples) < 8:
            unaligned_examples.append({
                "doc_id": doc_id,
                "label": label,
                "gold_text": text_value[start:end],
                "gold_span": (start, end),
                "start_aligned": start_aligned,
                "end_aligned": end_aligned,
            })

n_entities = boundary_totals["entities"]

if n_entities == 0:
    raise ValueError("No gold entities were found during tokenizer alignment audit.")

print("\n" + "=" * 65)
print("S3: GOLD SPAN / TOKENIZER ALIGNMENT")
print("=" * 65)
print("Gold entities audited:", n_entities)
print(
    "Start-boundary alignment:",
    f"{100 * boundary_totals['start_aligned'] / n_entities:.2f}%"
)
print(
    "End-boundary alignment:",
    f"{100 * boundary_totals['end_aligned'] / n_entities:.2f}%"
)
print(
    "Both boundaries aligned:",
    f"{100 * boundary_totals['both_aligned'] / n_entities:.2f}%"
)

print("\nExamples with at least one unaligned boundary:")
for example in unaligned_examples:
    print(example)

print("\nCell 4 completed. No model training has started.")

S3: DOCUMENT SPLIT
All labeled documents: 70
Training documents: 56
Validation documents: 14
Training annotations: 6532
Validation annotations: 1861

S3: PHASE 1 TOKENIZER
Tokenizer path: /kaggle/input/datasets/rohitbatra2004/cs781-phase1-tokenized-corpus/tokenizer/tokenizer.json
Vocabulary size: 50000

Tokenized document length statistics:
Minimum: 2108
Median: 5483
90th percentile: 9914
95th percentile: 12139
Maximum: 23559
Tokens with empty offsets: 0
Invalid non-empty offsets: 0

S3: GOLD SPAN / TOKENIZER ALIGNMENT
Gold entities audited: 8393
Start-boundary alignment: 3.78%
End-boundary alignment: 98.19%
Both boundaries aligned: 3.66%

Examples with at least one unaligned boundary:
{'doc_id': '1011232', 'label': 'APP', 'gold_text': 'ACHUTANANDA PUROHIT', 'gold_span': (12, 31), 'start_aligned': False, 'end_aligned': True}
{'doc_id': '1011232', 'label': 'JUDGE', 'gold_text': 'KRISHNAIYER, V.R.', 'gold_span': (111, 128), 'start_aligned': False, 'end_aligned': True}
{'doc_id': '1011232

In [5]:
# Cell 5: Diagnose tokenizer offset alignment by measuring exact and
# shifted entity boundaries, character coverage, and representative token offsets.

from collections import Counter

# --------------------------------------------------
# 1. Count boundary matches and character coverage
# --------------------------------------------------
offset_stats = Counter()
unaligned_start_examples = []

for doc_id in ALL_DOC_IDS:
    text_value = ALL_TEXTS[doc_id]
    encoding = tokenizer.encode(text_value, add_special_tokens=False)

    offsets = [
        (s, e) for s, e in encoding.offsets
        if s < e
    ]
    token_starts = {s for s, _ in offsets}
    token_ends = {e for _, e in offsets}

    for annotation in ALL_GOLD[doc_id]:
        start, end, label = annotation[:3]
        offset_stats["entities"] += 1

        # Candidate start-boundary shifts.
        for shift in (-2, -1, 0, 1, 2):
            if start + shift in token_starts:
                offset_stats[f"start_shift_{shift:+d}"] += 1

        # Candidate end-boundary shifts.
        for shift in (-2, -1, 0, 1, 2):
            if end + shift in token_ends:
                offset_stats[f"end_shift_{shift:+d}"] += 1

        # Determine whether the first and last entity characters
        # are covered by at least one non-empty token offset.
        start_covered = any(s <= start < e for s, e in offsets)
        end_char_covered = any(s <= end - 1 < e for s, e in offsets)

        offset_stats["start_char_covered"] += int(start_covered)
        offset_stats["end_char_covered"] += int(end_char_covered)

        # Count entities whose whole character span is covered.
        # This is a diagnostic only; it does not construct training labels.
        if start_covered and end_char_covered:
            offset_stats["both_endpoint_chars_covered"] += 1

        if not (start in token_starts) and len(unaligned_start_examples) < 5:
            unaligned_start_examples.append(
                (doc_id, start, end, label, text_value[start:end])
            )

# --------------------------------------------------
# 2. Print aggregate alignment statistics
# --------------------------------------------------
n = offset_stats["entities"]

print("=" * 65)
print("S3: TOKENIZER OFFSET DIAGNOSTICS")
print("=" * 65)
print("Entities checked:", n)

print("\nGold start vs tokenizer token starts:")
for shift in (-2, -1, 0, 1, 2):
    count = offset_stats[f"start_shift_{shift:+d}"]
    print(
        f"Gold start + ({shift:+d}) in token starts: "
        f"{count:,} ({100 * count / n:.2f}%)"
    )

print("\nGold end vs tokenizer token ends:")
for shift in (-2, -1, 0, 1, 2):
    count = offset_stats[f"end_shift_{shift:+d}"]
    print(
        f"Gold end + ({shift:+d}) in token ends: "
        f"{count:,} ({100 * count / n:.2f}%)"
    )

print("\nCharacter coverage:")
for key, description in [
    ("start_char_covered", "First entity character covered"),
    ("end_char_covered", "Last entity character covered"),
    ("both_endpoint_chars_covered", "Both endpoint characters covered"),
]:
    count = offset_stats[key]
    print(f"{description}: {count:,}/{n:,} ({100 * count / n:.2f}%)")

# --------------------------------------------------
# 3. Show token offsets near several unaligned starts
# --------------------------------------------------
print("\n" + "=" * 65)
print("REPRESENTATIVE TOKEN OFFSETS AROUND ENTITY STARTS")
print("=" * 65)

for doc_id, start, end, label, entity_text in unaligned_start_examples:
    text_value = ALL_TEXTS[doc_id]
    encoding = tokenizer.encode(text_value, add_special_tokens=False)

    print(f"\nDocument: {doc_id} | Label: {label}")
    print(f"Gold span: [{start}, {end}) | Entity: {entity_text!r}")
    print("Context:", repr(text_value[max(0, start - 8):min(len(text_value), start + 20)]))

    nearby = [
        (token, (s, e))
        for token, (s, e) in zip(encoding.tokens, encoding.offsets)
        if s < start + 5 and e > start - 5
    ]

    for token, span in nearby[:15]:
        print(f"  offset={span!s:>12} token={token!r}")

print("\nCell 5 completed. No model training has started.")

S3: TOKENIZER OFFSET DIAGNOSTICS
Entities checked: 8393

Gold start vs tokenizer token starts:
Gold start + (-2) in token starts: 1,561 (18.60%)
Gold start + (-1) in token starts: 8,152 (97.13%)
Gold start + (+0) in token starts: 317 (3.78%)
Gold start + (+1) in token starts: 1,214 (14.46%)
Gold start + (+2) in token starts: 2,428 (28.93%)

Gold end vs tokenizer token ends:
Gold end + (-2) in token ends: 567 (6.76%)
Gold end + (-1) in token ends: 598 (7.12%)
Gold end + (+0) in token ends: 8,241 (98.19%)
Gold end + (+1) in token ends: 3,021 (35.99%)
Gold end + (+2) in token ends: 962 (11.46%)

Character coverage:
First entity character covered: 8,393/8,393 (100.00%)
Last entity character covered: 8,393/8,393 (100.00%)
Both endpoint characters covered: 8,393/8,393 (100.00%)

REPRESENTATIVE TOKEN OFFSETS AROUND ENTITY STARTS

Document: 1011232 | Label: APP
Gold span: [12, 31) | Entity: 'ACHUTANANDA PUROHIT'
Context: 'TIONER: ACHUTANANDA PUROHIT '
  offset=     (0, 10) token='ĠPETITIONER'


In [6]:
# Cell 6: Inspect Architecture C checkpoint metadata, model configuration,
# state-dictionary shapes, and training status before building the NER model.

import gc
import json
from pathlib import Path

import torch

# --------------------------------------------------
# 1. Show available model configuration and metadata files
# --------------------------------------------------
print("=" * 70)
print("S3: ARCHITECTURE C CHECKPOINT AUDIT")
print("=" * 70)
print("Architecture C root:", ARCH_C_ROOT)

for filename in [
    "model_C_metadata.json",
    "configs",
    "metrics",
    "checkpoints",
]:
    path = ARCH_C_ROOT / filename
    if path.is_file() and path.suffix.lower() == ".json":
        print(f"\n{filename}:")
        try:
            with path.open("r", encoding="utf-8") as f:
                metadata = json.load(f)
            print(json.dumps(metadata, indent=2, default=str)[:8000])
        except Exception as exc:
            print("Could not read JSON metadata:", repr(exc))
    elif path.is_dir():
        print(f"\n{filename}/ contents:")
        for child in sorted(path.iterdir()):
            if child.is_file():
                print(f"  {child.name} ({child.stat().st_size:,} bytes)")

# --------------------------------------------------
# 2. Select the checkpoints to inspect
# --------------------------------------------------
checkpoint_candidates = [
    ARCH_C_ROOT / "model_C_final.pt",
    ARCH_C_ROOT / "checkpoints" / "best.pt",
    ARCH_C_ROOT / "checkpoints" / "latest.pt",
]

checkpoint_candidates = [
    path for path in checkpoint_candidates if path.is_file()
]

if not checkpoint_candidates:
    raise FileNotFoundError(
        "No Architecture C checkpoint files were found."
    )

# --------------------------------------------------
# 3. Inspect checkpoints one by one to limit peak memory
# --------------------------------------------------
def is_tensor_mapping(obj):
    return (
        isinstance(obj, dict)
        and len(obj) > 0
        and all(torch.is_tensor(v) for v in obj.values())
    )


def inspect_checkpoint(path):
    print("\n" + "=" * 70)
    print("Checkpoint:", path)

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    if not isinstance(checkpoint, dict):
        print("Unexpected checkpoint type:", type(checkpoint).__name__)
        del checkpoint
        gc.collect()
        return

    print("Top-level keys:", list(checkpoint.keys())[:60])

    # Display scalar training metadata when present.
    metadata_keys = [
        "experiment_id",
        "architecture_name",
        "architecture_version",
        "run_mode",
        "step",
        "tokens_seen",
        "train_loss",
        "best_validation_step",
        "best_validation_loss",
        "best_validation_perplexity",
        "status",
        "config_fingerprint",
        "model_type",
        "format_version",
    ]

    for key in metadata_keys:
        if key in checkpoint:
            value = checkpoint[key]
            if isinstance(value, (str, int, float, bool)) or value is None:
                print(f"{key}: {value}")
            else:
                print(f"{key}: <{type(value).__name__}>")

    for config_key in ["model_config", "config", "model_args"]:
        config_value = checkpoint.get(config_key)
        if isinstance(config_value, dict):
            print(f"\n{config_key}:")

S3: ARCHITECTURE C CHECKPOINT AUDIT
Architecture C root: /kaggle/input/datasets/rohitbatra2004/phase133/architecture_C

model_C_metadata.json:
{
  "artifact_type": "CS781_Phase1_Legal_Language_Model",
  "architecture_name": "Architecture_C_Deep_LSTM_Language_Model",
  "architecture_version": "C_v1",
  "experiment_id": "CS781_Phase1_C_v1_production",
  "model_type": "stacked_lstm_language_model",
  "parameter_count": 103404880,
  "parameter_millions": 103.40488,
  "model_config": {
    "vocab_size": 50000,
    "context_length": 1024,
    "embedding_dim": 1280,
    "hidden_size": 1280,
    "num_layers": 3,
    "dropout": 0.1,
    "tie_embeddings": true,
    "use_bias": true
  },
  "checkpoint_used": "/kaggle/working/architecture_C/checkpoints/best.pt",
  "training_step": 12000,
  "tokens_seen": 98304000,
  "best_validation_loss": 3.8705093705654146,
  "best_validation_perplexity": 47.96681274170745,
  "early_stopping": {
    "enabled": true,
    "patience_evaluations": 4,
    "min_delta"

In [7]:
# Cell 7: Verify the Architecture C final checkpoint, configuration,
# LSTM weight shapes, tokenizer compatibility, and parameter count.

import gc
import json
import torch

# --------------------------------------------------
# 1. Load the final model artifact
# --------------------------------------------------
ARCH_C_FINAL_CHECKPOINT = ARCH_C_ROOT / "model_C_final.pt"
ARCH_C_METADATA_PATH = ARCH_C_ROOT / "model_C_metadata.json"

if not ARCH_C_FINAL_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"Checkpoint not found: {ARCH_C_FINAL_CHECKPOINT}"
    )

with open(ARCH_C_METADATA_PATH, "r", encoding="utf-8") as f:
    ARCH_C_METADATA = json.load(f)

print("=" * 65)
print("S3: ARCHITECTURE C CHECKPOINT VERIFICATION")
print("=" * 65)

ARCH_C_ARTIFACT = torch.load(
    ARCH_C_FINAL_CHECKPOINT,
    map_location="cpu",
    weights_only=True,
)

if not isinstance(ARCH_C_ARTIFACT, dict):
    raise TypeError("The model artifact must be a dictionary.")

required_keys = {
    "artifact_type",
    "architecture_name",
    "architecture_version",
    "model_type",
    "model_config",
    "parameter_count",
    "state_dict",
}

missing_keys = required_keys - set(ARCH_C_ARTIFACT.keys())

if missing_keys:
    raise KeyError(
        f"Missing artifact keys: {sorted(missing_keys)}"
    )

ARCH_C_STATE_DICT = ARCH_C_ARTIFACT["state_dict"]
ARCH_C_MODEL_CONFIG = ARCH_C_ARTIFACT["model_config"]

if not isinstance(ARCH_C_STATE_DICT, dict):
    raise TypeError("The model state dictionary is invalid.")

# --------------------------------------------------
# 2. Validate the saved weights
# --------------------------------------------------
non_tensor_entries = [
    key
    for key, value in ARCH_C_STATE_DICT.items()
    if not torch.is_tensor(value)
]

if non_tensor_entries:
    raise TypeError(
        "Non-tensor entries found in state_dict: "
        f"{non_tensor_entries[:10]}"
    )

print("Architecture:", ARCH_C_ARTIFACT["architecture_name"])
print("Version:", ARCH_C_ARTIFACT["architecture_version"])
print("Model type:", ARCH_C_ARTIFACT["model_type"])

if ARCH_C_ARTIFACT["model_type"] != "stacked_lstm_language_model":
    raise ValueError("Unexpected model architecture.")

if ARCH_C_MODEL_CONFIG != ARCH_C_METADATA["model_config"]:
    raise ValueError("Artifact and metadata configurations differ.")

if ARCH_C_MODEL_CONFIG["vocab_size"] != TOKENIZER_VOCAB_SIZE:
    raise ValueError("Tokenizer vocabulary does not match the model.")

print("\nModel configuration:")
print(json.dumps(ARCH_C_MODEL_CONFIG, indent=2))

# --------------------------------------------------
# 3. Check the expected LSTM parameter shapes
# --------------------------------------------------
expected_shapes = {
    "token_embedding.weight": (50000, 1280),
    "lstm.weight_ih_l0": (5120, 1280),
    "lstm.weight_hh_l0": (5120, 1280),
    "lstm.weight_ih_l2": (5120, 1280),
    "final_norm.weight": (1280,),
    "output_bias": (50000,),
}

print("\nKey tensor shapes:")

for name, expected_shape in expected_shapes.items():
    if name not in ARCH_C_STATE_DICT:
        raise KeyError(f"Expected tensor is missing: {name}")

    actual_shape = tuple(ARCH_C_STATE_DICT[name].shape)
    print(f"{name}: {actual_shape}")

    if actual_shape != expected_shape:
        raise ValueError(
            f"Shape mismatch for {name}: "
            f"expected {expected_shape}, got {actual_shape}"
        )

# --------------------------------------------------
# 4. Verify parameter count against saved metadata
# --------------------------------------------------
actual_parameter_count = sum(
    tensor.numel()
    for tensor in ARCH_C_STATE_DICT.values()
)

artifact_parameter_count = int(
    ARCH_C_ARTIFACT["parameter_count"]
)

metadata_parameter_count = int(
    ARCH_C_METADATA["parameter_count"]
)

print("\nState dictionary entries:", len(ARCH_C_STATE_DICT))
print("Counted parameter elements:", f"{actual_parameter_count:,}")
print("Artifact parameter count:", f"{artifact_parameter_count:,}")
print("Metadata parameter count:", f"{metadata_parameter_count:,}")

if not (
    actual_parameter_count
    == artifact_parameter_count
    == metadata_parameter_count
):
    raise ValueError("The model parameter counts do not match.")

print("\nPASS: Architecture C checkpoint structure verified.")
print("No training has started.")

# Retain the verified state dictionary for the next cell.
# It will be used to initialize the Phase 2 NER model.
gc.collect()

S3: ARCHITECTURE C CHECKPOINT VERIFICATION
Architecture: Architecture_C_Deep_LSTM_Language_Model
Version: C_v1
Model type: stacked_lstm_language_model

Model configuration:
{
  "vocab_size": 50000,
  "context_length": 1024,
  "embedding_dim": 1280,
  "hidden_size": 1280,
  "num_layers": 3,
  "dropout": 0.1,
  "tie_embeddings": true,
  "use_bias": true
}

Key tensor shapes:
token_embedding.weight: (50000, 1280)
lstm.weight_ih_l0: (5120, 1280)
lstm.weight_hh_l0: (5120, 1280)
lstm.weight_ih_l2: (5120, 1280)
final_norm.weight: (1280,)
output_bias: (50000,)

State dictionary entries: 16
Counted parameter elements: 103,404,880
Artifact parameter count: 103,404,880
Metadata parameter count: 103,404,880

PASS: Architecture C checkpoint structure verified.
No training has started.


31

In [8]:
# Cell 8: Reconstruct the Phase 1 Architecture C LSTM, strictly load its
# pretrained weights, and verify the contextual hidden-state output.

import gc
import torch
import torch.nn as nn

# --------------------------------------------------
# 1. Define the Phase 1 LSTM backbone
# --------------------------------------------------
class ArchitectureCLSTMBackbone(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.vocab_size = int(config["vocab_size"])
        self.context_length = int(config["context_length"])
        self.embedding_dim = int(config["embedding_dim"])
        self.hidden_size = int(config["hidden_size"])
        self.num_layers = int(config["num_layers"])

        self.token_embedding = nn.Embedding(
            num_embeddings=self.vocab_size,
            embedding_dim=self.embedding_dim,
        )

        self.lstm = nn.LSTM(
            input_size=self.embedding_dim,
            hidden_size=self.hidden_size,
            num_layers=self.num_layers,
            dropout=float(config["dropout"]),
            bias=bool(config["use_bias"]),
            batch_first=True,
        )

        self.final_norm = nn.LayerNorm(self.hidden_size)

        # Retain the Phase 1 language-model output bias so that the
        # entire saved state dictionary can be loaded strictly.
        self.output_bias = nn.Parameter(
            torch.zeros(self.vocab_size)
        )

    def forward(self, input_ids):
        embeddings = self.token_embedding(input_ids)
        recurrent_output, _ = self.lstm(embeddings)
        contextual_output = self.final_norm(recurrent_output)
        return contextual_output


# --------------------------------------------------
# 2. Instantiate the backbone from the saved configuration
# --------------------------------------------------
ARCH_C_BACKBONE = ArchitectureCLSTMBackbone(
    ARCH_C_MODEL_CONFIG
)

# Strict loading catches missing keys, unexpected keys, and shape mismatches.
load_result = ARCH_C_BACKBONE.load_state_dict(
    ARCH_C_STATE_DICT,
    strict=True,
)

if load_result.missing_keys or load_result.unexpected_keys:
    raise RuntimeError(
        f"Checkpoint loading mismatch: {load_result}"
    )

print("Pretrained Architecture C weights loaded strictly.")

# --------------------------------------------------
# 3. Move the verified model to the execution device
# --------------------------------------------------
ARCH_C_BACKBONE = ARCH_C_BACKBONE.to(DEVICE)
ARCH_C_BACKBONE.eval()

# --------------------------------------------------
# 4. Run a small forward-pass smoke test
# --------------------------------------------------
test_token_count = 32

if test_token_count > ARCH_C_BACKBONE.context_length:
    raise ValueError("Smoke-test sequence exceeds the model context length.")

test_input_ids = torch.zeros(
    (1, test_token_count),
    dtype=torch.long,
    device=DEVICE,
)

with torch.inference_mode():
    test_hidden = ARCH_C_BACKBONE(test_input_ids)

if test_hidden.shape != (
    1,
    test_token_count,
    ARCH_C_MODEL_CONFIG["hidden_size"],
):
    raise RuntimeError(
        f"Unexpected output shape: {tuple(test_hidden.shape)}"
    )

if not torch.isfinite(test_hidden).all().item():
    raise FloatingPointError(
        "The pretrained LSTM produced non-finite hidden representations."
    )

print("Device:", DEVICE)
print("Smoke-test input shape:", tuple(test_input_ids.shape))
print("Contextual output shape:", tuple(test_hidden.shape))
print("Output finite:", torch.isfinite(test_hidden).all().item())
print("Pretrained parameters:", f"{sum(p.numel() for p in ARCH_C_BACKBONE.parameters()):,}")
print("\nCell 8 completed. No NER training has started.")

# Release temporary test tensors.
del test_input_ids, test_hidden
gc.collect()

Pretrained Architecture C weights loaded strictly.
Device: cuda
Smoke-test input shape: (1, 32)
Contextual output shape: (1, 32, 1280)
Output finite: True
Pretrained parameters: 103,404,880

Cell 8 completed. No NER training has started.


0

In [9]:
# Cell 9: Verify complete and non-overlapping tokenizer character coverage,
# inspect token-length distribution, and check that long entities fit windows.

from collections import Counter
import numpy as np

# --------------------------------------------------
# 1. Define the sequence-window configuration
# --------------------------------------------------
MAX_TOKENS = 512
WINDOW_STRIDE = 384

if MAX_TOKENS > ARCH_C_MODEL_CONFIG["context_length"]:
    raise ValueError("MAX_TOKENS exceeds the Phase 1 context length.")

if not (0 < WINDOW_STRIDE <= MAX_TOKENS):
    raise ValueError("Invalid window stride.")

print("=" * 65)
print("S3: CHARACTER COVERAGE AND WINDOW AUDIT")
print("=" * 65)
print("Maximum tokens per window:", MAX_TOKENS)
print("Window stride:", WINDOW_STRIDE)

# --------------------------------------------------
# 2. Audit all text documents, including unlabeled test documents
# --------------------------------------------------
ALL_AVAILABLE_TEXTS = {**ALL_TEXTS, **fold3_text}

coverage_summary = Counter()
token_lengths_all = []
token_offset_lengths = []
max_uncovered_examples = []

for doc_id, text_value in ALL_AVAILABLE_TEXTS.items():
    encoding = tokenizer.encode(text_value, add_special_tokens=False)

    if len(encoding.ids) != len(encoding.offsets):
        raise RuntimeError(f"Token/offset count mismatch in document {doc_id}.")

    coverage = np.zeros(len(text_value), dtype=np.int16)

    for start, end in encoding.offsets:
        if not (0 <= start <= end <= len(text_value)):
            raise ValueError(
                f"Invalid offset {(start, end)} in document {doc_id}."
            )

        if start == end:
            coverage_summary["empty_offsets"] += 1
            continue

        coverage[start:end] += 1
        token_offset_lengths.append(end - start)

    coverage_summary["documents"] += 1
    coverage_summary["characters"] += len(text_value)
    coverage_summary["fully_covered_documents"] += int(
        np.all(coverage == 1)
    )
    coverage_summary["documents_with_gaps"] += int(
        np.any(coverage == 0)
    )
    coverage_summary["documents_with_overlaps"] += int(
        np.any(coverage > 1)
    )
    coverage_summary["uncovered_characters"] += int(
        np.sum(coverage == 0)
    )
    coverage_summary["overcovered_characters"] += int(
        np.sum(coverage > 1)
    )

    token_lengths_all.append(len(encoding.ids))

    if np.any(coverage != 1) and len(max_uncovered_examples) < 5:
        positions = np.where(coverage != 1)[0][:10]
        max_uncovered_examples.append({
            "doc_id": doc_id,
            "text_length": len(text_value),
            "problem_positions": [
                {
                    "position": int(p),
                    "coverage_count": int(coverage[p]),
                    "character": repr(text_value[p]),
                }
                for p in positions
            ],
        })

# --------------------------------------------------
# 3. Print coverage results
# --------------------------------------------------
print("\nDocuments audited:", coverage_summary["documents"])
print("Total characters:", f"{coverage_summary['characters']:,}")
print("Documents fully covered exactly once:",
      coverage_summary["fully_covered_documents"])
print("Documents containing uncovered characters:",
      coverage_summary["documents_with_gaps"])
print("Documents containing overlapping offsets:",
      coverage_summary["documents_with_overlaps"])
print("Uncovered characters:",
      coverage_summary["uncovered_characters"])
print("Characters covered more than once:",
      coverage_summary["overcovered_characters"])
print("Empty tokenizer offsets:", coverage_summary["empty_offsets"])

print("\nToken count statistics:")
print("Minimum:", int(np.min(token_lengths_all)))
print("Median:", int(np.median(token_lengths_all)))
print("95th percentile:", int(np.percentile(token_lengths_all, 95)))
print("Maximum:", int(np.max(token_lengths_all)))

print("\nToken offset character-span statistics:")
print("Median:", float(np.median(token_offset_lengths)))
print("95th percentile:", float(np.percentile(token_offset_lengths, 95)))
print("Maximum:", int(np.max(token_offset_lengths)))

if max_uncovered_examples:
    print("\nExamples with incomplete or overlapping coverage:")
    for example in max_uncovered_examples:
        print(example)

# --------------------------------------------------
# 4. Check whether any gold entity exceeds a practical
# character-span length or the selected token window
# --------------------------------------------------
gold_span_lengths = []
gold_token_lengths = []
entities_not_covered_by_any_window = 0

for doc_id in ALL_DOC_IDS:
    text_value = ALL_TEXTS[doc_id]
    encoding = tokenizer.encode(text_value, add_special_tokens=False)

    offsets = [
        (s, e) for s, e in encoding.offsets
        if s < e
    ]

    for annotation in ALL_GOLD[doc_id]:
        start, end, label = annotation[:3]
        gold_span_lengths.append(end - start)

        # Find tokens containing the first and last entity characters.
        start_tokens = [
            i for i, (s, e) in enumerate(offsets)
            if s <= start < e
        ]
        end_tokens = [
            i for i, (s, e) in enumerate(offsets)
            if s <= end - 1 < e
        ]

        if not start_tokens or not end_tokens:
            raise ValueError(
                f"Could not map entity boundary to a token: "
                f"{doc_id}, {(start, end)}, {label}"
            )

        first_token = start_tokens[0]
        last_token = end_tokens[-1]

        if last_token < first_token:
            raise ValueError(
                f"Reversed token mapping for entity in {doc_id}."
            )

        token_span_length = last_token - first_token + 1
        gold_token_lengths.append(token_span_length)

        # Determine whether at least one full window contains the entity.
        possible_window_starts = range(
            0,
            len(offsets),
            WINDOW_STRIDE,
        )

        contained = any(
            window_start <= first_token
            and last_token < min(
                window_start + MAX_TOKENS,
                len(offsets),
            )
            for window_start in possible_window_starts
        )

        if not contained:
            entities_not_covered_by_any_window += 1

print("\nGold entity character-span length:")
print("Median:", float(np.median(gold_span_lengths)))
print("95th percentile:", float(np.percentile(gold_span_lengths, 95)))
print("Maximum:", int(np.max(gold_span_lengths)))

print("\nGold entity token-span length:")
print("Median:", float(np.median(gold_token_lengths)))
print("95th percentile:", float(np.percentile(gold_token_lengths, 95)))
print("Maximum:", int(np.max(gold_token_lengths)))

print(
    "Entities not fully contained in any proposed window:",
    entities_not_covered_by_any_window,
)

print("\nCell 9 completed. No model training has started.")

S3: CHARACTER COVERAGE AND WINDOW AUDIT
Maximum tokens per window: 512
Window stride: 384

Documents audited: 105
Total characters: 3,386,460
Documents fully covered exactly once: 97
Documents containing uncovered characters: 0
Documents containing overlapping offsets: 8
Uncovered characters: 0
Characters covered more than once: 1146
Empty tokenizer offsets: 0

Token count statistics:
Minimum: 0
Median: 5742
95th percentile: 13622
Maximum: 23559

Token offset character-span statistics:
Median: 4.0
95th percentile: 11.0
Maximum: 128

Examples with incomplete or overlapping coverage:
{'doc_id': '37849282', 'text_length': 23141, 'problem_positions': [{'position': 12459, 'coverage_count': 2, 'character': "'ç'"}]}
{'doc_id': '1348961', 'text_length': 34154, 'problem_positions': [{'position': 21830, 'coverage_count': 2, 'character': "'ø'"}]}
{'doc_id': '47053425', 'text_length': 48800, 'problem_positions': [{'position': 45567, 'coverage_count': 2, 'character': "'प'"}, {'position': 45568, 'co

In [10]:
# Cell 10: Efficiently verify gold entity coverage in LSTM windows,
# report long spans, and identify empty test documents.

from collections import Counter
import numpy as np

# --------------------------------------------------
# 1. Validate window configuration
# --------------------------------------------------
WINDOW_SIZE = MAX_TOKENS
WINDOW_STEP = WINDOW_STRIDE

if not (0 < WINDOW_STEP <= WINDOW_SIZE):
    raise ValueError("Invalid window size or stride.")

print("=" * 65)
print("S3: GOLD ENTITY WINDOW-COVERAGE AUDIT")
print("=" * 65)
print("Window size:", WINDOW_SIZE)
print("Window step:", WINDOW_STEP)

# --------------------------------------------------
# 2. Identify empty documents without assuming all texts are non-empty
# --------------------------------------------------
EMPTY_TOKEN_DOCUMENTS = []
EMPTY_TEXT_DOCUMENTS = []

for doc_id, text_value in ALL_AVAILABLE_TEXTS.items():
    if not text_value.strip():
        EMPTY_TEXT_DOCUMENTS.append(doc_id)

    if len(tokenizer.encode(text_value, add_special_tokens=False).ids) == 0:
        EMPTY_TOKEN_DOCUMENTS.append(doc_id)

print("\nZero-token documents:", len(EMPTY_TOKEN_DOCUMENTS))
print("Zero-token document IDs:", EMPTY_TOKEN_DOCUMENTS[:20])
print("Empty/whitespace-only documents:", len(EMPTY_TEXT_DOCUMENTS))
print("Empty/whitespace-only IDs:", EMPTY_TEXT_DOCUMENTS[:20])

# --------------------------------------------------
# 3. Check each entity against overlapping token offsets
# --------------------------------------------------
coverage = Counter()
entity_widths = []
failure_examples = []

for doc_id in ALL_DOC_IDS:
    text_value = ALL_TEXTS[doc_id]
    encoding = tokenizer.encode(text_value, add_special_tokens=False)
    offsets = encoding.offsets
    token_count = len(encoding.ids)

    if token_count == 0:
        if ALL_GOLD[doc_id]:
            raise ValueError(
                f"Document {doc_id} contains entities but no tokens."
            )
        continue

    text_length = len(text_value)
    earliest_token = np.full(text_length, token_count, dtype=np.int32)
    latest_token = np.full(text_length, -1, dtype=np.int32)

    for token_idx, (start, end) in enumerate(offsets):
        if not (0 <= start <= end <= text_length):
            raise ValueError(
                f"Invalid token offset {(start, end)} in {doc_id}."
            )

        if start < end:
            earliest_token[start:end] = np.minimum(
                earliest_token[start:end], token_idx
            )
            latest_token[start:end] = np.maximum(
                latest_token[start:end], token_idx
            )

    if np.any(latest_token < 0):
        raise ValueError(
            f"Some characters are not covered by tokenizer offsets "
            f"in document {doc_id}."
        )

    for annotation in ALL_GOLD[doc_id]:
        start_char, end_char, label = annotation[:3]

        first_token = int(earliest_token[start_char])
        last_token = int(latest_token[end_char - 1])

        if last_token < first_token:
            raise ValueError(
                f"Invalid token span for {doc_id}: {annotation}"
            )

        width = last_token - first_token + 1
        entity_widths.append(width)
        coverage["entities"] += 1

        # The latest valid window start not exceeding first_token
        # provides the strongest candidate for containing the entity.
        candidate_start = (
            first_token // WINDOW_STEP
        ) * WINDOW_STEP

        contained = (
            candidate_start <= first_token
            and last_token < min(
                candidate_start + WINDOW_SIZE,
                token_count,
            )
        )

        if contained:
            coverage["contained"] += 1
        else:
            coverage["not_contained"] += 1

            if len(failure_examples) < 10:
                failure_examples.append({
                    "doc_id": doc_id,
                    "label": label,
                    "entity": text_value[start_char:end_char],
                    "character_span": (start_char, end_char),
                    "token_span": (first_token, last_token),
                    "token_width": width,
                    "document_tokens": token_count,
                })

# --------------------------------------------------
# 4. Report coverage and entity widths
# --------------------------------------------------
total = coverage["entities"]
contained = coverage["contained"]
not_contained = coverage["not_contained"]

if total == 0:
    raise ValueError("No gold entities were audited.")

print("\n" + "=" * 65)
print("ENTITY WINDOW-COVERAGE RESULTS")
print("=" * 65)
print("Gold entities checked:", total)
print(
    "Contained in at least one window:",
    f"{contained:,} ({100 * contained / total:.2f}%)"
)
print(
    "Not fully contained in any window:",
    f"{not_contained:,} ({100 * not_contained / total:.2f}%)"
)

print("\nGold entity token widths:")
print("Minimum:", int(np.min(entity_widths)))
print("Median:", float(np.median(entity_widths)))
print("90th percentile:", float(np.percentile(entity_widths, 90)))
print("95th percentile:", float(np.percentile(entity_widths, 95)))
print("Maximum:", int(np.max(entity_widths)))

if failure_examples:
    print("\nEntities not contained in a window:")
    for item in failure_examples:
        print(item)

print("\nCell 10 completed. No model training has started.")

S3: GOLD ENTITY WINDOW-COVERAGE AUDIT
Window size: 512
Window step: 384

Zero-token documents: 1
Zero-token document IDs: ['1305957']
Empty/whitespace-only documents: 1
Empty/whitespace-only IDs: ['1305957']

ENTITY WINDOW-COVERAGE RESULTS
Gold entities checked: 8393
Contained in at least one window: 8,393 (100.00%)
Not fully contained in any window: 0 (0.00%)

Gold entity token widths:
Minimum: 1
Median: 4.0
90th percentile: 10.0
95th percentile: 14.0
Maximum: 47

Cell 10 completed. No model training has started.


In [11]:
# Cell 11: Build LSTM NER windows with character-level start/end targets,
# map each character to a token representation, and mask partial edge entities.

from collections import Counter
import numpy as np

# --------------------------------------------------
# 1. Define label mappings and a character vocabulary
# --------------------------------------------------
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}
NUM_LABELS = len(LABELS)

PAD_CHAR_ID = 0
UNK_CHAR_ID = 1

# Learn the character vocabulary only from training documents.
training_character_counts = Counter(
    character
    for doc_id in TRAIN_DOC_IDS
    for character in ALL_TEXTS[doc_id]
)

CHAR_TO_ID = {
    character: index
    for index, character in enumerate(
        sorted(training_character_counts),
        start=2,
    )
}

CHAR_VOCAB_SIZE = len(CHAR_TO_ID) + 2

if CHAR_VOCAB_SIZE <= 2:
    raise ValueError("The training character vocabulary is empty.")

print("=" * 65)
print("S3: CHARACTER-LEVEL NER PREPROCESSING")
print("=" * 65)
print("Entity labels:", NUM_LABELS)
print("Training character vocabulary size:", CHAR_VOCAB_SIZE)
print("Maximum tokens per window:", MAX_TOKENS)
print("Window stride:", WINDOW_STRIDE)

# --------------------------------------------------
# 2. Create deterministic overlapping token-window starts
# --------------------------------------------------
def get_window_starts(token_count, window_size, stride):
    if token_count <= 0:
        return []

    if token_count <= window_size:
        return [0]

    starts = list(range(0, token_count - window_size + 1, stride))
    final_start = token_count - window_size

    if starts[-1] != final_start:
        starts.append(final_start)

    return starts


# --------------------------------------------------
# 3. Build one preprocessed document window
# --------------------------------------------------
def build_ner_window(doc_id, text_value, annotations, token_ids, offsets,
                     token_start, token_end):
    """
    Convert a token window into character-level training arrays.

    End targets are placed on the last entity character:
    source gold [start, end) -> start target at start,
    end target at end - 1.
    """
    local_offsets = offsets[token_start:token_end]
    local_ids = token_ids[token_start:token_end]

    non_empty_offsets = [
        (start, end)
        for start, end in local_offsets
        if start < end
    ]

    if not non_empty_offsets:
        return None

    char_start = min(start for start, _ in non_empty_offsets)
    char_end = max(end for _, end in non_empty_offsets)

    if not (0 <= char_start < char_end <= len(text_value)):
        raise ValueError(
            f"Invalid character bounds for document {doc_id}: "
            f"{char_start}, {char_end}"
        )

    window_text = text_value[char_start:char_end]
    char_length = len(window_text)

    # Map each local character to a token position in this window.
    # For overlapping tokenizer offsets, the later token overwrites
    # the earlier token for the overlapping characters.
    char_to_token = np.full(char_length, -1, dtype=np.int16)

    for local_token_index, (start, end) in enumerate(local_offsets):
        if start < end:
            left = start - char_start
            right = end - char_start
            char_to_token[left:right] = local_token_index

    if np.any(char_to_token < 0):
        bad_positions = np.flatnonzero(char_to_token < 0)[:10]
        raise ValueError(
            f"Character positions could not be mapped to tokens for "
            f"document {doc_id}, window start {token_start}. "
            f"Example positions: {bad_positions.tolist()}"
        )

    char_ids = np.fromiter(
        (CHAR_TO_ID.get(character, UNK_CHAR_ID) for character in window_text),
        dtype=np.int32,
        count=char_length,
    )

    start_targets = np.zeros(
        (char_length, NUM_LABELS),
        dtype=np.uint8,
    )
    end_targets = np.zeros(
        (char_length, NUM_LABELS),
        dtype=np.uint8,
    )

    # 1 means the character contributes to training loss.
    # Partial gold entities at window edges are masked.
    loss_mask = np.ones(char_length, dtype=np.uint8)

    contained_entities = 0
    partial_entities = 0

    for annotation in annotations:
        start, end, label = annotation[:3]

        if label not in LABEL_TO_ID:
            raise ValueError(f"Unknown entity label: {label}")

        overlaps_window = start < char_end and end > char_start
        contained_window = start >= char_start and end <= char_end

        if contained_window:
            start_index = start - char_start
            end_index = end - char_start - 1
            label_index = LABEL_TO_ID[label]

            start_targets[start_index, label_index] = 1
            end_targets[end_index, label_index] = 1
            contained_entities += 1

        elif overlaps_window:
            # Do not label a truncated entity fragment as background.
            mask_start = max(start, char_start) - char_start
            mask_end = min(end, char_end) - char_start

            if mask_start < mask_end:
                loss_mask[mask_start:mask_end] = 0

            partial_entities += 1

    return {
        "doc_id": doc_id,
        "token_start": token_start,
        "token_end": token_end,
        "char_start": char_start,
        "char_end": char_end,
        "input_ids": np.asarray(local_ids, dtype=np.int32),
        "char_ids": char_ids,
        "char_to_token": char_to_token,
        "start_targets": start_targets,
        "end_targets": end_targets,
        "loss_mask": loss_mask,
        "contained_entities": contained_entities,
        "partial_entities": partial_entities,
    }


# --------------------------------------------------
# 4. Tokenize and preprocess a document collection
# --------------------------------------------------
def build_window_collection(document_ids, split_name):
    windows = []
    skipped_empty_documents = []
    document_count = 0

    for doc_id in document_ids:
        text_value = ALL_TEXTS[doc_id]
        annotations = ALL_GOLD[doc_id]

        encoding = tokenizer.encode(
            text_value,
            add_special_tokens=False,
        )

        token_ids = encoding.ids
        offsets = encoding.offsets

        if len(token_ids) != len(offsets):
            raise ValueError(
                f"Token/offset mismatch in document {doc_id}."
            )

        if len(token_ids) == 0:
            if annotations:
                raise ValueError(
                    f"Document {doc_id} has entities but no tokens."
                )

            skipped_empty_documents.append(doc_id)
            continue

        document_count += 1

        starts = get_window_starts(
            len(token_ids),
            MAX_TOKENS,
            WINDOW_STRIDE,
        )

        for token_start in starts:
            token_end = min(token_start + MAX_TOKENS, len(token_ids))

            window = build_ner_window(
                doc_id=doc_id,
                text_value=text_value,
                annotations=annotations,
                token_ids=token_ids,
                offsets=offsets,
                token_start=token_start,
                token_end=token_end,
            )

            if window is not None:
                windows.append(window)

    print(f"\n{split_name} preprocessing:")
    print("Documents processed:", document_count)
    print("Windows created:", len(windows))
    print("Empty documents skipped:", skipped_empty_documents)

    return windows


TRAIN_WINDOWS = build_window_collection(
    TRAIN_DOC_IDS,
    "Training",
)

VAL_WINDOWS = build_window_collection(
    VAL_DOC_IDS,
    "Validation",
)

if not TRAIN_WINDOWS or not VAL_WINDOWS:
    raise RuntimeError("Training or validation preprocessing produced no windows.")

# --------------------------------------------------
# 5. Verify that original gold targets are represented
# --------------------------------------------------
def validate_window_targets(windows, split_name):
    positive_starts = 0
    positive_ends = 0
    fully_contained_count = 0
    masked_character_count = 0

    for window in windows:
        positive_starts += int(window["start_targets"].sum())
        positive_ends += int(window["end_targets"].sum())
        fully_contained_count += window["contained_entities"]
        masked_character_count += int(
            (window["loss_mask"] == 0).sum()
        )

        char_length = window["char_end"] - window["char_start"]

        if not (
            len(window["char_ids"])
            == len(window["char_to_token"])
            == len(window["loss_mask"])
            == char_length
        ):
            raise RuntimeError(
                f"Character array length mismatch in {split_name}."
            )

        if window["start_targets"].shape != (char_length, NUM_LABELS):
            raise RuntimeError(
                f"Start target shape mismatch in {split_name}."
            )

        if window["end_targets"].shape != (char_length, NUM_LABELS):
            raise RuntimeError(
                f"End target shape mismatch in {split_name}."
            )

    print(f"\n{split_name} target checks:")
    print("Positive start labels across windows:", positive_starts)
    print("Positive end labels across windows:", positive_ends)
    print("Contained entity instances across windows:", fully_contained_count)
    print("Masked character positions:", masked_character_count)

    if positive_starts != positive_ends:
        raise RuntimeError(
            f"Start/end positive counts differ in {split_name}."
        )


validate_window_targets(TRAIN_WINDOWS, "Training")
validate_window_targets(VAL_WINDOWS, "Validation")

# --------------------------------------------------
# 6. Verify document-level separation
# --------------------------------------------------
train_window_doc_ids = {window["doc_id"] for window in TRAIN_WINDOWS}
val_window_doc_ids = {window["doc_id"] for window in VAL_WINDOWS}

if train_window_doc_ids & val_window_doc_ids:
    raise RuntimeError("Train/validation document leakage detected.")

print("\nTraining documents represented:", len(train_window_doc_ids))
print("Validation documents represented:", len(val_window_doc_ids))
print("Character vocabulary size:", CHAR_VOCAB_SIZE)
print("\nCell 11 completed. No model training has started.")

S3: CHARACTER-LEVEL NER PREPROCESSING
Entity labels: 12
Training character vocabulary size: 159
Maximum tokens per window: 512
Window stride: 384

Training preprocessing:
Documents processed: 56
Windows created: 903
Empty documents skipped: []

Validation preprocessing:
Documents processed: 14
Windows created: 293
Empty documents skipped: []

Training target checks:
Positive start labels across windows: 8548
Positive end labels across windows: 8548
Contained entity instances across windows: 8548
Masked character positions: 1690

Validation target checks:
Positive start labels across windows: 2487
Positive end labels across windows: 2487
Contained entity instances across windows: 2487
Masked character positions: 684

Training documents represented: 56
Validation documents represented: 14
Character vocabulary size: 159

Cell 11 completed. No model training has started.


In [12]:
# Cell 12: Define the Architecture C character-level NER model with a
# frozen pretrained LSTM, identity-initialized low-rank multiplicative
# adapter, character embeddings, convolutional context, and span-boundary heads.

import torch
import torch.nn as nn
import torch.nn.functional as F

# --------------------------------------------------
# 1. Verify the prerequisites from preceding cells
# --------------------------------------------------
required_variables = [
    "ARCH_C_BACKBONE",
    "ARCH_C_MODEL_CONFIG",
    "CHAR_VOCAB_SIZE",
    "PAD_CHAR_ID",
    "UNK_CHAR_ID",
    "NUM_LABELS",
    "DEVICE",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        f"Run the preceding setup cells first. Missing: {missing_variables}"
    )

if not isinstance(ARCH_C_BACKBONE, nn.Module):
    raise TypeError("ARCH_C_BACKBONE must be a PyTorch module.")

if NUM_LABELS != 12:
    raise ValueError(f"Expected 12 entity labels; received {NUM_LABELS}.")

# --------------------------------------------------
# 2. Freeze the Phase 1 LSTM parameters
# --------------------------------------------------
for parameter in ARCH_C_BACKBONE.parameters():
    parameter.requires_grad_(False)

ARCH_C_BACKBONE.eval()

LORMA_RANK = 8
LORMA_ALPHA = 16.0
CHAR_EMBED_DIM = 32
NER_HIDDEN_DIM = 256
NER_DROPOUT = 0.15

# --------------------------------------------------
# 3. Define an identity-initialized low-rank multiplicative adapter
# --------------------------------------------------
class LoRMAPlusHiddenAdapter(nn.Module):
    """
    Apply the identity-preserving low-rank feature transformation

        h_new = h + (alpha / rank) * B(A(h))

    where A maps hidden_size -> rank and B maps rank -> hidden_size.

    Initializing B to zero means that the adapter initially preserves
    the pretrained hidden representations exactly.
    """

    def __init__(self, hidden_size, rank=8, alpha=16.0):
        super().__init__()

        if not (0 < rank < hidden_size):
            raise ValueError("Adapter rank must be positive and < hidden size.")

        self.down = nn.Linear(hidden_size, rank, bias=False)
        self.up = nn.Linear(rank, hidden_size, bias=False)
        self.scale = float(alpha) / float(rank)

        nn.init.kaiming_uniform_(self.down.weight, a=5**0.5)
        nn.init.zeros_(self.up.weight)

    def forward(self, hidden_states):
        low_rank_update = self.up(self.down(hidden_states))
        return hidden_states + self.scale * low_rank_update


# --------------------------------------------------
# 4. Define the character-level NER model
# --------------------------------------------------
class ArchitectureCS3NER(nn.Module):
    def __init__(
        self,
        backbone,
        char_vocab_size,
        num_labels,
        char_embed_dim=32,
        ner_hidden_dim=256,
        dropout=0.15,
        lorma_rank=8,
        lorma_alpha=16.0,
    ):
        super().__init__()

        self.backbone = backbone
        self.num_labels = num_labels
        self.backbone_hidden_size = int(
            ARCH_C_MODEL_CONFIG["hidden_size"]
        )

        # Keep the entire Phase 1 backbone frozen.
        for parameter in self.backbone.parameters():
            parameter.requires_grad_(False)

        # Adapt the pretrained contextual representation.
        self.lorma = LoRMAPlusHiddenAdapter(
            hidden_size=self.backbone_hidden_size,
            rank=lorma_rank,
            alpha=lorma_alpha,
        )

        # Character identity features are useful when several characters
        # receive the same contextual token representation.
        self.char_embedding = nn.Embedding(
            num_embeddings=char_vocab_size,
            embedding_dim=char_embed_dim,
            padding_idx=PAD_CHAR_ID,
        )

        combined_dim = self.backbone_hidden_size + char_embed_dim

        self.input_projection = nn.Linear(
            combined_dim,
            ner_hidden_dim,
        )

        self.dropout = nn.Dropout(dropout)

        # Local character context helps the model locate boundaries
        # within a tokenizer token.
        self.char_conv = nn.Conv1d(
            in_channels=ner_hidden_dim,
            out_channels=ner_hidden_dim,
            kernel_size=3,
            padding=1,
        )

        self.feature_norm = nn.LayerNorm(ner_hidden_dim)

        # Each character independently receives 12 possible start
        # scores and 12 possible end scores.
        self.start_head = nn.Linear(ner_hidden_dim, num_labels)
        self.end_head = nn.Linear(ner_hidden_dim, num_labels)

        # Keep the frozen LSTM in evaluation mode even when this model
        # is switched to training mode.
        self.backbone.eval()

    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()
        return self

    def forward(self, input_ids, char_ids, char_to_token):
        """
        input_ids:    [batch, token_length]
        char_ids:     [batch, character_length]
        char_to_token:[batch, character_length], -1 for padding

        Returns:
            start_logits: [batch, character_length, num_labels]
            end_logits:   [batch, character_length, num_labels]
        """

        if input_ids.ndim != 2:
            raise ValueError("input_ids must be a rank-2 tensor.")

        if char_ids.ndim != 2 or char_to_token.ndim != 2:
            raise ValueError("Character inputs must be rank-2 tensors.")

        if char_ids.shape != char_to_token.shape:
            raise ValueError("char_ids and char_to_token shapes differ.")

        if input_ids.shape[0] != char_ids.shape[0]:
            raise ValueError("Token and character batch sizes differ.")

        # No gradients are stored for the 103.4M-parameter backbone.
        with torch.no_grad():
            token_hidden = self.backbone(input_ids)

        # The preprocessing cell maps each character to a token position.
        # Padded character positions use -1 and are safely clamped to 0.
        token_indices = char_to_token.clamp(
            min=0,
            max=input_ids.shape[1] - 1,
        ).long()

        expanded_indices = token_indices.unsqueeze(-1).expand(
            -1,
            -1,
            token_hidden.shape[-1],
        )

        char_context = token_hidden.gather(
            dim=1,
            index=expanded_indices,
        )

        # The padding ID is reserved and never assigned to real characters.
        char_valid = char_ids.ne(PAD_CHAR_ID).unsqueeze(-1)

        # Apply the low-rank multiplicative adaptation.
        char_context = self.lorma(char_context)

        char_features = self.char_embedding(char_ids)

        combined = torch.cat(
            [char_context, char_features],
            dim=-1,
        )

        features = F.gelu(self.input_projection(combined))

        # Prevent batch padding from contributing nonzero features
        # to adjacent valid characters.
        features = features.masked_fill(~char_valid, 0.0)

        conv_features = self.char_conv(
            features.transpose(1, 2)
        ).transpose(1, 2)

        features = self.feature_norm(
            features + self.dropout(F.gelu(conv_features))
        )

        features = features.masked_fill(~char_valid, 0.0)
        features = self.dropout(features)

        start_logits = self.start_head(features)
        end_logits = self.end_head(features)

        return start_logits, end_logits


# --------------------------------------------------
# 5. Instantiate the model and count trainable parameters
# --------------------------------------------------
S3_MODEL = ArchitectureCS3NER(
    backbone=ARCH_C_BACKBONE,
    char_vocab_size=CHAR_VOCAB_SIZE,
    num_labels=NUM_LABELS,
    char_embed_dim=CHAR_EMBED_DIM,
    ner_hidden_dim=NER_HIDDEN_DIM,
    dropout=NER_DROPOUT,
    lorma_rank=LORMA_RANK,
    lorma_alpha=LORMA_ALPHA,
).to(DEVICE)

all_parameter_count = sum(
    parameter.numel()
    for parameter in S3_MODEL.parameters()
)

trainable_parameter_count = sum(
    parameter.numel()
    for parameter in S3_MODEL.parameters()
    if parameter.requires_grad
)

frozen_parameter_count = all_parameter_count - trainable_parameter_count

print("=" * 65)
print("S3: MODEL INITIALIZATION")
print("=" * 65)
print("Device:", DEVICE)
print("Labels:", NUM_LABELS)
print("LoRMA rank:", LORMA_RANK)
print("LoRMA alpha:", LORMA_ALPHA)
print("Character embedding dimension:", CHAR_EMBED_DIM)
print("NER hidden dimension:", NER_HIDDEN_DIM)
print("Total parameters:", f"{all_parameter_count:,}")
print("Trainable parameters:", f"{trainable_parameter_count:,}")
print("Frozen parameters:", f"{frozen_parameter_count:,}")

if not all(
    not parameter.requires_grad
    for parameter in S3_MODEL.backbone.parameters()
):
    raise RuntimeError("Some pretrained backbone parameters are trainable.")

if trainable_parameter_count <= 0:
    raise RuntimeError("The NER head has no trainable parameters.")

# --------------------------------------------------
# 6. Smoke-test the model using one real training window
# --------------------------------------------------
sample = TRAIN_WINDOWS[0]

sample_input_ids = torch.as_tensor(
    sample["input_ids"].astype("int64"),
    dtype=torch.long,
    device=DEVICE,
).unsqueeze(0)

sample_char_ids = torch.as_tensor(
    sample["char_ids"].astype("int64"),
    dtype=torch.long,
    device=DEVICE,
).unsqueeze(0)

sample_char_to_token = torch.as_tensor(
    sample["char_to_token"].astype("int64"),
    dtype=torch.long,
    device=DEVICE,
).unsqueeze(0)

S3_MODEL.eval()

with torch.inference_mode():
    sample_start_logits, sample_end_logits = S3_MODEL(
        input_ids=sample_input_ids,
        char_ids=sample_char_ids,
        char_to_token=sample_char_to_token,
    )

expected_shape = (
    1,
    len(sample["char_ids"]),
    NUM_LABELS,
)

if tuple(sample_start_logits.shape) != expected_shape:
    raise RuntimeError(
        f"Unexpected start-logit shape: "
        f"{tuple(sample_start_logits.shape)} vs {expected_shape}"
    )

if tuple(sample_end_logits.shape) != expected_shape:
    raise RuntimeError(
        f"Unexpected end-logit shape: "
        f"{tuple(sample_end_logits.shape)} vs {expected_shape}"
    )

if not (
    torch.isfinite(sample_start_logits).all().item()
    and torch.isfinite(sample_end_logits).all().item()
):
    raise FloatingPointError("The NER model produced non-finite logits.")

print("\nSmoke test passed.")
print("Token input shape:", tuple(sample_input_ids.shape))
print("Character input shape:", tuple(sample_char_ids.shape))
print("Start-logit shape:", tuple(sample_start_logits.shape))
print("End-logit shape:", tuple(sample_end_logits.shape))
print("All logits finite: True")
print("Cell 12 completed. No NER training has started.")

S3: MODEL INITIALIZATION
Device: cuda
Labels: 12
LoRMA rank: 8
LoRMA alpha: 16.0
Character embedding dimension: 32
NER hidden dimension: 256
Total parameters: 103,970,120
Trainable parameters: 565,240
Frozen parameters: 103,404,880

Smoke test passed.
Token input shape: (1, 512)
Character input shape: (1, 2566)
Start-logit shape: (1, 2566, 12)
End-logit shape: (1, 2566, 12)
All logits finite: True
Cell 12 completed. No NER training has started.


In [13]:
# Cell 13: Create reproducible training and validation DataLoaders with
# dynamic padding for token IDs, character IDs, boundary targets, and masks.

import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# --------------------------------------------------
# 1. Reproducible batching configuration
# --------------------------------------------------
BATCH_SIZE = 2
NUM_WORKERS = 0

if BATCH_SIZE < 1:
    raise ValueError("BATCH_SIZE must be positive.")

print("=" * 65)
print("S3: DATALOADER PREPARATION")
print("=" * 65)
print("Batch size:", BATCH_SIZE)
print("DataLoader workers:", NUM_WORKERS)
print("Device:", DEVICE)

# --------------------------------------------------
# 2. Dataset wrapper
# --------------------------------------------------
class S3NERWindowDataset(Dataset):
    def __init__(self, windows, split_name):
        self.windows = windows
        self.split_name = split_name

        if not self.windows:
            raise ValueError(f"{split_name} dataset contains no windows.")

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, index):
        window = self.windows[index]

        return {
            "input_ids": torch.as_tensor(
                window["input_ids"].astype(np.int64, copy=False),
                dtype=torch.long,
            ),
            "char_ids": torch.as_tensor(
                window["char_ids"].astype(np.int64, copy=False),
                dtype=torch.long,
            ),
            "char_to_token": torch.as_tensor(
                window["char_to_token"].astype(np.int64, copy=False),
                dtype=torch.long,
            ),
            "start_targets": torch.as_tensor(
                window["start_targets"],
                dtype=torch.float32,
            ),
            "end_targets": torch.as_tensor(
                window["end_targets"],
                dtype=torch.float32,
            ),
            "loss_mask": torch.as_tensor(
                window["loss_mask"],
                dtype=torch.float32,
            ),
            "doc_id": window["doc_id"],
        }


# --------------------------------------------------
# 3. Dynamic-padding collate function
# --------------------------------------------------
def collate_s3_windows(batch):
    if not batch:
        raise ValueError("Cannot collate an empty batch.")

    batch_size = len(batch)
    max_token_len = max(item["input_ids"].numel() for item in batch)
    max_char_len = max(item["char_ids"].numel() for item in batch)

    num_labels = batch[0]["start_targets"].shape[-1]

    input_ids = torch.zeros(
        (batch_size, max_token_len),
        dtype=torch.long,
    )

    char_ids = torch.full(
        (batch_size, max_char_len),
        fill_value=PAD_CHAR_ID,
        dtype=torch.long,
    )

    # -1 marks characters that do not map to a real token.
    char_to_token = torch.full(
        (batch_size, max_char_len),
        fill_value=-1,
        dtype=torch.long,
    )

    start_targets = torch.zeros(
        (batch_size, max_char_len, num_labels),
        dtype=torch.float32,
    )

    end_targets = torch.zeros_like(start_targets)

    # Zero means the character must not contribute to the loss.
    loss_mask = torch.zeros(
        (batch_size, max_char_len),
        dtype=torch.float32,
    )

    doc_ids = []

    for i, item in enumerate(batch):
        n_tokens = item["input_ids"].numel()
        n_chars = item["char_ids"].numel()

        input_ids[i, :n_tokens] = item["input_ids"]
        char_ids[i, :n_chars] = item["char_ids"]
        char_to_token[i, :n_chars] = item["char_to_token"]

        start_targets[i, :n_chars] = item["start_targets"]
        end_targets[i, :n_chars] = item["end_targets"]
        loss_mask[i, :n_chars] = item["loss_mask"]

        doc_ids.append(item["doc_id"])

    # Ignore padded positions even if their underlying target arrays are zero.
    loss_mask = loss_mask * char_ids.ne(PAD_CHAR_ID).float()

    return {
        "input_ids": input_ids,
        "char_ids": char_ids,
        "char_to_token": char_to_token,
        "start_targets": start_targets,
        "end_targets": end_targets,
        "loss_mask": loss_mask,
        "doc_ids": doc_ids,
    }


# --------------------------------------------------
# 4. Build the DataLoaders
# --------------------------------------------------
TRAIN_DATASET = S3NERWindowDataset(TRAIN_WINDOWS, "Training")
VAL_DATASET = S3NERWindowDataset(VAL_WINDOWS, "Validation")

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

TRAIN_LOADER = DataLoader(
    TRAIN_DATASET,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    collate_fn=collate_s3_windows,
    generator=loader_generator,
    drop_last=False,
)

VAL_LOADER = DataLoader(
    VAL_DATASET,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    collate_fn=collate_s3_windows,
    drop_last=False,
)

# --------------------------------------------------
# 5. Inspect one batch from each split
# --------------------------------------------------
def inspect_s3_batch(loader, split_name):
    batch = next(iter(loader))

    print(f"\n{split_name} batch:")
    for key in [
        "input_ids",
        "char_ids",
        "char_to_token",
        "start_targets",
        "end_targets",
        "loss_mask",
    ]:
        print(f"  {key}: {tuple(batch[key].shape)}")

    print("  Documents:", batch["doc_ids"])
    print("  Valid characters:", int(batch["loss_mask"].sum().item()))
    print(
        "  Positive starts:",
        int(batch["start_targets"].sum().item()),
    )
    print(
        "  Positive ends:",
        int(batch["end_targets"].sum().item()),
    )

    # Verify that every valid character points to a real input token.
    for i in range(batch["input_ids"].shape[0]):
        valid_positions = batch["char_ids"][i].ne(PAD_CHAR_ID)
        indices = batch["char_to_token"][i][valid_positions]

        if indices.numel() > 0:
            if indices.min().item() < 0:
                raise ValueError(
                    f"{split_name}: valid character maps to a negative token."
                )

            if indices.max().item() >= batch["input_ids"].shape[1]:
                raise ValueError(
                    f"{split_name}: character maps beyond the padded token sequence."
                )

    # Verify target values and mask.
    if not torch.all((batch["start_targets"] == 0) |
                     (batch["start_targets"] == 1)):
        raise ValueError("Start targets must be binary.")

    if not torch.all((batch["end_targets"] == 0) |
                     (batch["end_targets"] == 1)):
        raise ValueError("End targets must be binary.")

    if not torch.all((batch["loss_mask"] == 0) |
                     (batch["loss_mask"] == 1)):
        raise ValueError("Loss mask must be binary.")

    return batch


TRAIN_SAMPLE_BATCH = inspect_s3_batch(TRAIN_LOADER, "Training")
VAL_SAMPLE_BATCH = inspect_s3_batch(VAL_LOADER, "Validation")

# --------------------------------------------------
# 6. Run a forward-pass test on a dynamically padded batch
# --------------------------------------------------
S3_MODEL.eval()

device_batch = {
    key: value.to(DEVICE, non_blocking=True)
    for key, value in TRAIN_SAMPLE_BATCH.items()
    if torch.is_tensor(value)
}

with torch.inference_mode():
    test_start_logits, test_end_logits = S3_MODEL(
        input_ids=device_batch["input_ids"],
        char_ids=device_batch["char_ids"],
        char_to_token=device_batch["char_to_token"],
    )

expected_shape = (
    device_batch["char_ids"].shape[0],
    device_batch["char_ids"].shape[1],
    NUM_LABELS,
)

if tuple(test_start_logits.shape) != expected_shape:
    raise RuntimeError("Start-logit batch shape mismatch.")

if tuple(test_end_logits.shape) != expected_shape:
    raise RuntimeError("End-logit batch shape mismatch.")

if not (
    torch.isfinite(test_start_logits).all().item()
    and torch.isfinite(test_end_logits).all().item()
):
    raise FloatingPointError("Batch forward pass produced non-finite logits.")

print("\nTraining windows:", len(TRAIN_DATASET))
print("Training batches per epoch:", len(TRAIN_LOADER))
print("Validation windows:", len(VAL_DATASET))
print("Validation batches:", len(VAL_LOADER))
print("Batch forward test passed.")
print("No training has started.")
print("\nCell 13 completed.")

S3: DATALOADER PREPARATION
Batch size: 2
DataLoader workers: 0
Device: cuda

Training batch:
  input_ids: (2, 512)
  char_ids: (2, 2362)
  char_to_token: (2, 2362)
  start_targets: (2, 2362, 12)
  end_targets: (2, 2362, 12)
  loss_mask: (2, 2362)
  Documents: ['1646640', '90251163']
  Valid characters: 4704
  Positive starts: 19
  Positive ends: 19

Validation batch:
  input_ids: (2, 512)
  char_ids: (2, 2634)
  char_to_token: (2, 2634)
  start_targets: (2, 2634, 12)
  end_targets: (2, 2634, 12)
  loss_mask: (2, 2634)
  Documents: ['1011232', '1011232']
  Valid characters: 5064
  Positive starts: 25
  Positive ends: 25

Training windows: 903
Training batches per epoch: 452
Validation windows: 293
Validation batches: 147
Batch forward test passed.
No training has started.

Cell 13 completed.


In [14]:
# Cell 14: Cache frozen Architecture C token representations once so
# subsequent training epochs only optimize the adapter and NER prediction head.

import gc
import numpy as np
import torch

# --------------------------------------------------
# 1. Configure and validate feature caching
# --------------------------------------------------
CACHE_BATCH_SIZE = 4
CACHE_DTYPE = torch.float16

if not torch.cuda.is_available():
    CACHE_DTYPE = torch.float32

if not TRAIN_WINDOWS or not VAL_WINDOWS:
    raise RuntimeError("Training and validation windows must exist.")

S3_MODEL.eval()
ARCH_C_BACKBONE.eval()

print("=" * 65)
print("S3: PRECOMPUTING FROZEN LSTM REPRESENTATIONS")
print("=" * 65)
print("Device:", DEVICE)
print("Initial cache batch size:", CACHE_BATCH_SIZE)
print("Cached representation dtype:", CACHE_DTYPE)

# --------------------------------------------------
# 2. Extract representations for one collection
# --------------------------------------------------
def cache_backbone_representations(windows, split_name, initial_batch_size=4):
    batch_size = initial_batch_size

    while True:
        cached_features = []
        failed_due_to_oom = False

        try:
            for batch_start in range(0, len(windows), batch_size):
                batch_windows = windows[
                    batch_start:batch_start + batch_size
                ]

                token_arrays = [
                    window["input_ids"]
                    for window in batch_windows
                ]

                lengths = {len(array) for array in token_arrays}
                if len(lengths) != 1:
                    raise ValueError(
                        f"{split_name} windows have inconsistent token lengths: "
                        f"{sorted(lengths)}"
                    )

                input_ids = torch.as_tensor(
                    np.stack(token_arrays),
                    dtype=torch.long,
                    device=DEVICE,
                )

                # The backbone is frozen, so only the forward values
                # are needed; no autograd graph is retained.
                with torch.inference_mode():
                    if DEVICE.type == "cuda":
                        with torch.autocast(
                            device_type="cuda",
                            dtype=torch.float16,
                        ):
                            token_hidden = ARCH_C_BACKBONE(input_ids)
                    else:
                        token_hidden = ARCH_C_BACKBONE(input_ids)

                if not torch.isfinite(token_hidden).all().item():
                    raise FloatingPointError(
                        f"{split_name} produced non-finite cached features."
                    )

                # Keep compact CPU copies, not GPU-resident feature tensors.
                token_hidden_cpu = token_hidden.to(
                    device="cpu",
                    dtype=CACHE_DTYPE,
                ).contiguous()

                cached_features.extend(
                    token_hidden_cpu[i].clone()
                    for i in range(token_hidden_cpu.shape[0])
                )

                del input_ids, token_hidden, token_hidden_cpu

            # Only attach features after the complete split succeeds.
            if len(cached_features) != len(windows):
                raise RuntimeError(
                    f"{split_name}: cached {len(cached_features)} features "
                    f"for {len(windows)} windows."
                )

            for window, features in zip(windows, cached_features):
                window["cached_token_hidden"] = features

            break

        except torch.cuda.OutOfMemoryError:
            failed_due_to_oom = True

            del cached_features
            gc.collect()
            torch.cuda.empty_cache()

            if batch_size <= 1:
                raise RuntimeError(
                    f"GPU out-of-memory even with batch size 1 while "
                    f"caching {split_name} features."
                )

            print(
                f"{split_name}: CUDA out-of-memory at batch size {batch_size}; "
                "retrying with a smaller batch."
            )

        if failed_due_to_oom:
            batch_size = max(1, batch_size // 2)

    total_bytes = sum(
        window["cached_token_hidden"].numel()
        * window["cached_token_hidden"].element_size()
        for window in windows
    )

    print(f"\n{split_name}:")
    print("Windows cached:", len(windows))
    print("Successful cache batch size:", batch_size)
    print("Feature shape per window:",
          tuple(windows[0]["cached_token_hidden"].shape))
    print("Cached feature storage:",
          f"{total_bytes / (1024 ** 3):.2f} GiB")


# --------------------------------------------------
# 3. Cache training and validation features
# --------------------------------------------------
cache_backbone_representations(
    TRAIN_WINDOWS,
    "Training",
    initial_batch_size=CACHE_BATCH_SIZE,
)

cache_backbone_representations(
    VAL_WINDOWS,
    "Validation",
    initial_batch_size=CACHE_BATCH_SIZE,
)

# --------------------------------------------------
# 4. Verify the cached features
# --------------------------------------------------
for split_name, windows in [
    ("Training", TRAIN_WINDOWS),
    ("Validation", VAL_WINDOWS),
]:
    for window in [windows[0], windows[len(windows) // 2], windows[-1]]:
        features = window.get("cached_token_hidden")

        if features is None:
            raise RuntimeError(f"{split_name}: missing cached features.")

        expected_shape = (
            len(window["input_ids"]),
            ARCH_C_MODEL_CONFIG["hidden_size"],
        )

        if tuple(features.shape) != expected_shape:
            raise RuntimeError(
                f"{split_name}: unexpected feature shape "
                f"{tuple(features.shape)}; expected {expected_shape}."
            )

        if not torch.isfinite(features).all().item():
            raise FloatingPointError(
                f"{split_name}: non-finite cached features detected."
            )

print("\nPASS: frozen LSTM representations are cached and verified.")
print("No NER training has started.")
print("Cell 14 completed.")

S3: PRECOMPUTING FROZEN LSTM REPRESENTATIONS
Device: cuda
Initial cache batch size: 4
Cached representation dtype: torch.float16

Training:
Windows cached: 903
Successful cache batch size: 4
Feature shape per window: (512, 1280)
Cached feature storage: 1.10 GiB

Validation:
Windows cached: 293
Successful cache batch size: 4
Feature shape per window: (512, 1280)
Cached feature storage: 0.36 GiB

PASS: frozen LSTM representations are cached and verified.
No NER training has started.
Cell 14 completed.


In [15]:
# Cell 15: Create DataLoaders that include cached LSTM features, then
# verify a differentiable forward pass through the LoRMA adapter and NER head.

import gc
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# --------------------------------------------------
# 1. Create a dataset that also returns cached features
# --------------------------------------------------
class S3CachedWindowDataset(Dataset):
    def __init__(self, windows, split_name):
        self.windows = windows
        self.base_dataset = S3NERWindowDataset(windows, split_name)
        self.split_name = split_name

        for index, window in enumerate(self.windows):
            features = window.get("cached_token_hidden")

            if features is None:
                raise ValueError(
                    f"{split_name} window {index} has no cached features."
                )

            expected_shape = (
                len(window["input_ids"]),
                ARCH_C_MODEL_CONFIG["hidden_size"],
            )

            if tuple(features.shape) != expected_shape:
                raise ValueError(
                    f"{split_name} window {index}: cached feature shape "
                    f"{tuple(features.shape)} != {expected_shape}."
                )

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, index):
        item = self.base_dataset[index]
        item["cached_token_hidden"] = self.windows[index][
            "cached_token_hidden"
        ]
        return item


# --------------------------------------------------
# 2. Collate cached features with the existing targets
# --------------------------------------------------
def collate_s3_cached_windows(batch):
    if not batch:
        raise ValueError("Cannot collate an empty cached-feature batch.")

    # Reuse the already validated padding logic for the standard fields.
    result = collate_s3_windows(batch)

    batch_size = len(batch)
    max_token_length = result["input_ids"].shape[1]
    hidden_size = ARCH_C_MODEL_CONFIG["hidden_size"]

    cached_features = torch.zeros(
        (batch_size, max_token_length, hidden_size),
        dtype=torch.float16,
    )

    for i, item in enumerate(batch):
        features = item["cached_token_hidden"]

        if not torch.is_tensor(features):
            raise TypeError("Cached features must be PyTorch tensors.")

        if features.ndim != 2 or features.shape[1] != hidden_size:
            raise ValueError(
                f"Unexpected cached-feature shape: {tuple(features.shape)}"
            )

        if features.shape[0] != item["input_ids"].numel():
            raise ValueError(
                "Cached feature length does not match token sequence length."
            )

        token_length = features.shape[0]
        cached_features[i, :token_length] = features.to(torch.float16)

    result["cached_token_hidden"] = cached_features
    return result


# --------------------------------------------------
# 3. Construct the cached-feature datasets and loaders
# --------------------------------------------------
TRAIN_CACHED_DATASET = S3CachedWindowDataset(
    TRAIN_WINDOWS,
    "Training",
)

VAL_CACHED_DATASET = S3CachedWindowDataset(
    VAL_WINDOWS,
    "Validation",
)

cached_loader_generator = torch.Generator()
cached_loader_generator.manual_seed(SEED)

TRAIN_CACHED_LOADER = DataLoader(
    TRAIN_CACHED_DATASET,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    collate_fn=collate_s3_cached_windows,
    generator=cached_loader_generator,
    drop_last=False,
)

VAL_CACHED_LOADER = DataLoader(
    VAL_CACHED_DATASET,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    collate_fn=collate_s3_cached_windows,
    drop_last=False,
)


# --------------------------------------------------
# 4. Forward pass using cached features
# --------------------------------------------------
def s3_forward_cached(model, cached_token_hidden, char_ids, char_to_token):
    """
    Use precomputed token representations.

    cached_token_hidden: [batch, token_length, hidden_size]
    char_ids:            [batch, char_length]
    char_to_token:       [batch, char_length]
    """

    if cached_token_hidden.ndim != 3:
        raise ValueError("Cached token features must have rank 3.")

    if char_ids.shape != char_to_token.shape:
        raise ValueError("Character IDs and token mapping shapes differ.")

    if cached_token_hidden.shape[0] != char_ids.shape[0]:
        raise ValueError("Token and character batch sizes differ.")

    if cached_token_hidden.shape[2] != model.backbone_hidden_size:
        raise ValueError("Cached features have the wrong hidden dimension.")

    # Convert FP16 cache to FP32 to match the trainable model parameters.
    token_hidden = cached_token_hidden.to(
        device=char_ids.device,
        dtype=model.input_projection.weight.dtype,
    )

    char_ids = char_ids.long()
    char_to_token = char_to_token.long()

    token_indices = char_to_token.clamp(
        min=0,
        max=token_hidden.shape[1] - 1,
    )

    expanded_indices = token_indices.unsqueeze(-1).expand(
        -1,
        -1,
        token_hidden.shape[-1],
    )

    char_context = token_hidden.gather(
        dim=1,
        index=expanded_indices,
    )

    char_valid = char_ids.ne(PAD_CHAR_ID).unsqueeze(-1)

    # Apply the trainable identity-initialized low-rank adapter.
    char_context = model.lorma(char_context)

    char_features = model.char_embedding(char_ids)

    combined = torch.cat(
        [char_context, char_features],
        dim=-1,
    )

    features = torch.nn.functional.gelu(
        model.input_projection(combined)
    )

    features = features.masked_fill(~char_valid, 0.0)

    conv_features = model.char_conv(
        features.transpose(1, 2)
    ).transpose(1, 2)

    features = model.feature_norm(
        features + model.dropout(
            torch.nn.functional.gelu(conv_features)
        )
    )

    features = features.masked_fill(~char_valid, 0.0)
    features = model.dropout(features)

    start_logits = model.start_head(features)
    end_logits = model.end_head(features)

    return start_logits, end_logits


# --------------------------------------------------
# 5. Verify cached batch construction and forward pass
# --------------------------------------------------
batch = next(iter(TRAIN_CACHED_LOADER))

print("=" * 65)
print("S3: CACHED-FEATURE DATALOADER CHECK")
print("=" * 65)

print("Cached features:", tuple(batch["cached_token_hidden"].shape))
print("Character IDs:", tuple(batch["char_ids"].shape))
print("Start targets:", tuple(batch["start_targets"].shape))
print("End targets:", tuple(batch["end_targets"].shape))

device_batch = {
    key: value.to(DEVICE, non_blocking=True)
    for key, value in batch.items()
    if torch.is_tensor(value)
}

S3_MODEL.train()
S3_MODEL.zero_grad(set_to_none=True)

start_logits, end_logits = s3_forward_cached(
    S3_MODEL,
    cached_token_hidden=device_batch["cached_token_hidden"],
    char_ids=device_batch["char_ids"],
    char_to_token=device_batch["char_to_token"],
)

expected_shape = (
    device_batch["char_ids"].shape[0],
    device_batch["char_ids"].shape[1],
    NUM_LABELS,
)

if tuple(start_logits.shape) != expected_shape:
    raise RuntimeError("Cached forward pass has an incorrect start shape.")

if tuple(end_logits.shape) != expected_shape:
    raise RuntimeError("Cached forward pass has an incorrect end shape.")

# Verify that gradients can flow through the adapter and NER head.
smoke_loss = start_logits.float().mean() + end_logits.float().mean()
smoke_loss.backward()

for name, module in [
    ("LoRMA adapter", S3_MODEL.lorma),
    ("Start head", S3_MODEL.start_head),
    ("End head", S3_MODEL.end_head),
]:
    has_gradient = any(
        parameter.grad is not None
        and torch.isfinite(parameter.grad).all().item()
        for parameter in module.parameters()
        if parameter.requires_grad
    )

    if not has_gradient:
        raise RuntimeError(f"{name} did not receive finite gradients.")

if any(
    parameter.grad is not None
    for parameter in S3_MODEL.backbone.parameters()
):
    raise RuntimeError("Frozen backbone unexpectedly received gradients.")

S3_MODEL.zero_grad(set_to_none=True)

print("\nCached forward pass passed.")
print("Start logits:", tuple(start_logits.shape))
print("End logits:", tuple(end_logits.shape))
print("Adapter and prediction-head gradients: verified")
print("Frozen backbone gradients: absent")
print("Training has not started; only a smoke-test backward pass ran.")
print("\nCell 15 completed.")

del batch, device_batch, start_logits, end_logits, smoke_loss
gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

S3: CACHED-FEATURE DATALOADER CHECK
Cached features: (2, 512, 1280)
Character IDs: (2, 2362)
Start targets: (2, 2362, 12)
End targets: (2, 2362, 12)

Cached forward pass passed.
Start logits: (2, 2362, 12)
End logits: (2, 2362, 12)
Adapter and prediction-head gradients: verified
Frozen backbone gradients: absent
Training has not started; only a smoke-test backward pass ran.

Cell 15 completed.


In [16]:
# Cell 16: Configure class-balanced boundary losses, AdamW, gradient clipping,
# a validation-driven learning-rate scheduler, and a no-update gradient test.

import gc
import numpy as np
import torch
import torch.nn as nn

# --------------------------------------------------
# 1. Calculate per-label positive weights from training targets only
# --------------------------------------------------
def calculate_boundary_positive_weights(windows):
    positive_counts = np.zeros(NUM_LABELS, dtype=np.float64)
    valid_character_count = 0.0

    for window in windows:
        mask = window["loss_mask"].astype(np.float64)

        valid_character_count += mask.sum()

        positive_counts += (
            window["start_targets"].astype(np.float64)
            * mask[:, None]
        ).sum(axis=0)

    if valid_character_count <= 0:
        raise ValueError("No valid character positions are available.")

    negative_counts = valid_character_count - positive_counts

    if np.any(positive_counts <= 0):
        missing = [
            LABELS[i]
            for i in np.flatnonzero(positive_counts <= 0)
        ]
        raise ValueError(
            "Some labels have no positive training starts: "
            + ", ".join(missing)
        )

    # Square-root weighting is less aggressive than raw neg/pos ratios.
    positive_weights = np.sqrt(
        negative_counts / positive_counts
    )

    positive_weights = np.clip(
        positive_weights,
        a_min=1.0,
        a_max=30.0,
    )

    return positive_counts, negative_counts, positive_weights


S3_POSITIVE_COUNTS, S3_NEGATIVE_COUNTS, S3_POSITIVE_WEIGHTS = (
    calculate_boundary_positive_weights(TRAIN_WINDOWS)
)

S3_POS_WEIGHT_TENSOR = torch.tensor(
    S3_POSITIVE_WEIGHTS,
    dtype=torch.float32,
    device=DEVICE,
)

print("=" * 65)
print("S3: CLASS-BALANCED BOUNDARY LOSS")
print("=" * 65)

for i, label in enumerate(LABELS):
    print(
        f"{label:12s} "
        f"positive={int(S3_POSITIVE_COUNTS[i]):6d} "
        f"negative={int(S3_NEGATIVE_COUNTS[i]):9d} "
        f"pos_weight={S3_POSITIVE_WEIGHTS[i]:.2f}"
    )

# --------------------------------------------------
# 2. Define masked, class-balanced binary cross-entropy
# --------------------------------------------------
def s3_boundary_loss(
    start_logits,
    end_logits,
    start_targets,
    end_targets,
    loss_mask,
):
    """
    Compute class-balanced BCE for start/end boundary predictions.

    Shapes:
      logits/targets: [batch, characters, labels]
      loss_mask:      [batch, characters]

    Padding and partial-entity character positions do not contribute.
    """

    expected_shape = start_targets.shape

    if start_logits.shape != expected_shape:
        raise ValueError("Start logits and targets have different shapes.")

    if end_logits.shape != end_targets.shape:
        raise ValueError("End logits and targets have different shapes.")

    if start_logits.shape != end_logits.shape:
        raise ValueError("Start and end logit shapes differ.")

    if loss_mask.shape != start_logits.shape[:2]:
        raise ValueError("Loss mask has an incorrect shape.")

    mask = loss_mask.float().unsqueeze(-1)

    start_element_loss = nn.functional.binary_cross_entropy_with_logits(
        start_logits.float(),
        start_targets.float(),
        pos_weight=S3_POS_WEIGHT_TENSOR,
        reduction="none",
    )

    end_element_loss = nn.functional.binary_cross_entropy_with_logits(
        end_logits.float(),
        end_targets.float(),
        pos_weight=S3_POS_WEIGHT_TENSOR,
        reduction="none",
    )

    denominator = (
        loss_mask.float().sum()
        * start_logits.shape[-1]
    ).clamp_min(1.0)

    start_loss = (start_element_loss * mask).sum() / denominator
    end_loss = (end_element_loss * mask).sum() / denominator

    total_loss = 0.5 * (start_loss + end_loss)

    return total_loss, start_loss.detach(), end_loss.detach()


# --------------------------------------------------
# 3. Create optimizer and learning-rate scheduler
# --------------------------------------------------
S3_LEARNING_RATE = 3e-4
S3_WEIGHT_DECAY = 0.01
S3_MAX_GRAD_NORM = 1.0

trainable_parameters = [
    parameter
    for parameter in S3_MODEL.parameters()
    if parameter.requires_grad
]

if not trainable_parameters:
    raise RuntimeError("No trainable model parameters were found.")

S3_OPTIMIZER = torch.optim.AdamW(
    trainable_parameters,
    lr=S3_LEARNING_RATE,
    weight_decay=S3_WEIGHT_DECAY,
)

# Validation Strict Span Macro-F1 will be the monitored quantity.
S3_SCHEDULER = torch.optim.lr_scheduler.ReduceLROnPlateau(
    S3_OPTIMIZER,
    mode="max",
    factor=0.5,
    patience=1,
    threshold=1e-4,
    min_lr=1e-5,
)

print("\nOptimizer:", type(S3_OPTIMIZER).__name__)
print("Initial learning rate:", S3_LEARNING_RATE)
print("Weight decay:", S3_WEIGHT_DECAY)
print("Maximum gradient norm:", S3_MAX_GRAD_NORM)
print("Trainable tensors:", len(trainable_parameters))

# --------------------------------------------------
# 4. Verify loss and gradients without updating weights
# --------------------------------------------------
sample_batch = next(iter(TRAIN_CACHED_LOADER))

device_batch = {
    key: value.to(DEVICE, non_blocking=True)
    for key, value in sample_batch.items()
    if torch.is_tensor(value)
}

S3_MODEL.train()
S3_OPTIMIZER.zero_grad(set_to_none=True)

start_logits, end_logits = s3_forward_cached(
    S3_MODEL,
    cached_token_hidden=device_batch["cached_token_hidden"],
    char_ids=device_batch["char_ids"],
    char_to_token=device_batch["char_to_token"],
)

smoke_loss, smoke_start_loss, smoke_end_loss = s3_boundary_loss(
    start_logits=start_logits,
    end_logits=end_logits,
    start_targets=device_batch["start_targets"],
    end_targets=device_batch["end_targets"],
    loss_mask=device_batch["loss_mask"],
)

if not torch.isfinite(smoke_loss).item():
    raise FloatingPointError("Boundary loss is not finite.")

smoke_loss.backward()

gradient_count = 0

for name, parameter in S3_MODEL.named_parameters():
    if parameter.requires_grad and parameter.grad is not None:
        if not torch.isfinite(parameter.grad).all().item():
            raise FloatingPointError(
                f"Non-finite gradient detected in {name}."
            )
        gradient_count += 1

if gradient_count == 0:
    raise RuntimeError("No trainable parameters received gradients.")

gradient_norm = torch.nn.utils.clip_grad_norm_(
    trainable_parameters,
    max_norm=S3_MAX_GRAD_NORM,
)

if not torch.isfinite(gradient_norm).item():
    raise FloatingPointError("Gradient norm is not finite.")

print("\nLoss/gradient test:")
print("Total loss:", float(smoke_loss.detach().cpu()))
print("Start loss:", float(smoke_start_loss.cpu()))
print("End loss:", float(smoke_end_loss.cpu()))
print("Trainable tensors with gradients:", gradient_count)
print("Pre-clipping gradient norm:", float(gradient_norm))

# Discard the smoke-test gradients. The optimizer has not been stepped.
S3_OPTIMIZER.zero_grad(set_to_none=True)
S3_MODEL.zero_grad(set_to_none=True)

del sample_batch, device_batch
del start_logits, end_logits, smoke_loss
del smoke_start_loss, smoke_end_loss, gradient_norm
gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print("\nPASS: loss, gradients, and clipping verified.")
print("No optimizer update or NER training has occurred.")
print("Cell 16 completed.")

S3: CLASS-BALANCED BOUNDARY LOSS
APP          positive=   386 negative=  2252646 pos_weight=30.00
RESP         positive=   393 negative=  2252639 pos_weight=30.00
A.COUNSEL    positive=   190 negative=  2252842 pos_weight=30.00
R.COUNSEL    positive=   160 negative=  2252872 pos_weight=30.00
JUDGE        positive=   230 negative=  2252802 pos_weight=30.00
WIT          positive=   213 negative=  2252819 pos_weight=30.00
AUTH         positive=  1036 negative=  2251996 pos_weight=30.00
COURT        positive=  1018 negative=  2252014 pos_weight=30.00
STAT         positive=  1268 negative=  2251764 pos_weight=30.00
PREC         positive=  1230 negative=  2251802 pos_weight=30.00
DATE         positive=  1771 negative=  2251261 pos_weight=30.00
CASENO       positive=   653 negative=  2252379 pos_weight=30.00

Optimizer: AdamW
Initial learning rate: 0.0003
Weight decay: 0.01
Maximum gradient norm: 1.0
Trainable tensors: 13

Loss/gradient test:
Total loss: 0.743130624294281
Start loss: 0.736950

In [17]:
# Cell 17: Define the strict character-span decoder, document-level macro-F1
# evaluator, and oracle test that checks whether gold targets decode exactly.

from collections import defaultdict
import numpy as np

# --------------------------------------------------
# 1. Define the maximum allowed span length
# --------------------------------------------------
# Use a generous character limit for legal entities.
# The oracle test below verifies whether this limit preserves validation gold.
MAX_SPAN_CHARS = 512

if MAX_SPAN_CHARS < 1:
    raise ValueError("MAX_SPAN_CHARS must be positive.")

print("=" * 65)
print("S3: STRICT SPAN DECODER")
print("=" * 65)
print("Maximum decoded span length:", MAX_SPAN_CHARS)

# --------------------------------------------------
# 2. Decode start/end scores into local character spans
# --------------------------------------------------
def decode_boundary_spans(
    start_scores,
    end_scores,
    start_threshold=0.5,
    end_threshold=0.5,
    max_span_chars=MAX_SPAN_CHARS,
):
    """
    Decode spans from arrays of shape [characters, num_labels].

    A predicted end position refers to the final entity character.
    The returned span uses end-exclusive offsets.

    Greedy selection is performed independently per entity label.
    Boundaries are used at most once, and same-label spans cannot overlap.
    """

    start_scores = np.asarray(start_scores)
    end_scores = np.asarray(end_scores)

    if start_scores.ndim != 2 or end_scores.ndim != 2:
        raise ValueError("Start/end scores must be 2-D arrays.")

    if start_scores.shape != end_scores.shape:
        raise ValueError("Start/end score shapes do not match.")

    num_chars, num_labels = start_scores.shape

    if num_labels != NUM_LABELS:
        raise ValueError(
            f"Expected {NUM_LABELS} labels, got {num_labels}."
        )

    if not (
        np.isfinite(start_scores).all()
        and np.isfinite(end_scores).all()
    ):
        raise FloatingPointError("Non-finite boundary scores detected.")

    decoded = []

    for label_id in range(num_labels):
        starts = np.flatnonzero(
            start_scores[:, label_id] >= start_threshold
        )
        ends = np.flatnonzero(
            end_scores[:, label_id] >= end_threshold
        )

        if len(starts) == 0 or len(ends) == 0:
            continue

        # Construct candidate pairs that satisfy span ordering and length.
        candidates = []

        for start in starts:
            valid_ends = ends[
                (ends >= start)
                & ((ends - start + 1) <= max_span_chars)
            ]

            for end_char_index in valid_ends:
                start_score = float(start_scores[start, label_id])
                end_score = float(end_scores[end_char_index, label_id])

                # Geometric mean balances confidence at both boundaries.
                candidate_score = np.sqrt(
                    max(0.0, start_score) * max(0.0, end_score)
                )

                candidates.append((
                    -candidate_score,
                    int(end_char_index - start + 1),
                    int(start),
                    int(end_char_index),
                ))

        # Prefer high confidence, then shorter spans, then earlier positions.
        candidates.sort()

        used_starts = set()
        used_ends = set()
        selected_intervals = []

        for neg_score, span_length, start, end_char_index in candidates:
            end_exclusive = end_char_index + 1

            if start in used_starts or end_char_index in used_ends:
                continue

            # Avoid overlapping spans of the same class.
            overlaps = any(
                start < existing_end and existing_start < end_exclusive
                for existing_start, existing_end in selected_intervals
            )

            if overlaps:
                continue

            decoded.append((
                start,
                end_exclusive,
                ID_TO_LABEL[label_id],
            ))

            used_starts.add(start)
            used_ends.add(end_char_index)
            selected_intervals.append((start, end_exclusive))

    return sorted(decoded, key=lambda item: (item[0], item[1], item[2]))


# --------------------------------------------------
# 3. Define document-level strict span metrics
# --------------------------------------------------
def calculate_strict_span_metrics(gold_spans, predicted_spans):
    """
    Each input is an iterable of:
        (document_id, start_character, end_character_exclusive, label)

    Strict match requires exact document ID, both offsets, and label.
    Macro-F1 averages over all labels in LABELS.
    """

    gold_set = set(gold_spans)
    pred_set = set(predicted_spans)

    per_label = {}
    f1_values = []

    for label in LABELS:
        gold_label = {span for span in gold_set if span[3] == label}
        pred_label = {span for span in pred_set if span[3] == label}

        true_positives = len(gold_label & pred_label)
        false_positives = len(pred_label - gold_label)
        false_negatives = len(gold_label - pred_label)

        precision = (
            true_positives / (true_positives + false_positives)
            if true_positives + false_positives > 0
            else 0.0
        )

        recall = (
            true_positives / (true_positives + false_negatives)
            if true_positives + false_negatives > 0
            else 0.0
        )

        f1 = (
            2.0 * precision * recall / (precision + recall)
            if precision + recall > 0
            else 0.0
        )

        per_label[label] = {
            "gold": len(gold_label),
            "predicted": len(pred_label),
            "tp": true_positives,
            "fp": false_positives,
            "fn": false_negatives,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }

        f1_values.append(f1)

    return {
        "macro_f1": float(np.mean(f1_values)),
        "gold_count": len(gold_set),
        "predicted_count": len(pred_set),
        "exact_true_positives": len(gold_set & pred_set),
        "per_label": per_label,
    }


# --------------------------------------------------
# 4. Construct the gold validation span set
# --------------------------------------------------
VAL_GOLD_SPANS = {
    (doc_id, start, end, label)
    for doc_id in VAL_DOC_IDS
    for start, end, label, *_ in ALL_GOLD[doc_id]
}

if not VAL_GOLD_SPANS:
    raise ValueError("Validation gold span set is empty.")

long_gold_spans = [
    span for span in VAL_GOLD_SPANS
    if span[2] - span[1] > MAX_SPAN_CHARS
]

if long_gold_spans:
    raise ValueError(
        f"{len(long_gold_spans)} validation gold spans exceed "
        f"MAX_SPAN_CHARS={MAX_SPAN_CHARS}. Increase the limit."
    )

print("\nValidation gold spans:", len(VAL_GOLD_SPANS))

# --------------------------------------------------
# 5. Oracle test: decode the gold boundary target arrays
# --------------------------------------------------
# If target-to-character mapping and decoding are correct, each validation
# entity should be reconstructed exactly after duplicate windows are merged.
VAL_ORACLE_SPANS = set()

for window in VAL_WINDOWS:
    decoded_local = decode_boundary_spans(
        start_scores=window["start_targets"].astype(np.float32),
        end_scores=window["end_targets"].astype(np.float32),
        start_threshold=0.5,
        end_threshold=0.5,
    )

    for local_start, local_end, label in decoded_local:
        global_start = window["char_start"] + local_start
        global_end = window["char_start"] + local_end

        VAL_ORACLE_SPANS.add((
            window["doc_id"],
            global_start,
            global_end,
            label,
        ))

# --------------------------------------------------
# 6. Report exact reconstruction results
# --------------------------------------------------
oracle_metrics = calculate_strict_span_metrics(
    VAL_GOLD_SPANS,
    VAL_ORACLE_SPANS,
)

print("\n" + "=" * 65)
print("S3: ORACLE DECODER VALIDATION")
print("=" * 65)
print("Gold spans:", len(VAL_GOLD_SPANS))
print("Decoded oracle spans:", len(VAL_ORACLE_SPANS))
print("Exact true positives:", oracle_metrics["exact_true_positives"])
print("False positives:", len(VAL_ORACLE_SPANS - VAL_GOLD_SPANS))
print("False negatives:", len(VAL_GOLD_SPANS - VAL_ORACLE_SPANS))
print("Oracle strict span macro-F1:", f"{oracle_metrics['macro_f1']:.6f}")

if VAL_ORACLE_SPANS != VAL_GOLD_SPANS:
    missing = list(VAL_GOLD_SPANS - VAL_ORACLE_SPANS)[:10]
    extra = list(VAL_ORACLE_SPANS - VAL_GOLD_SPANS)[:10]

    print("\nExamples of missed gold spans:")
    for span in missing:
        print(span)

    print("\nExamples of extra decoded spans:")
    for span in extra:
        print(span)

    raise RuntimeError(
        "Oracle decoding did not reproduce validation gold exactly. "
        "Do not train until this issue is resolved."
    )

if abs(oracle_metrics["macro_f1"] - 1.0) > 1e-12:
    raise RuntimeError("Oracle strict macro-F1 should equal 1.0.")

print("\nPASS: oracle decoding reproduces all validation gold spans exactly.")
print("Cell 17 completed. No NER training has started.")

S3: STRICT SPAN DECODER
Maximum decoded span length: 512

Validation gold spans: 1861

S3: ORACLE DECODER VALIDATION
Gold spans: 1861
Decoded oracle spans: 1861
Exact true positives: 1861
False positives: 0
False negatives: 0
Oracle strict span macro-F1: 1.000000

PASS: oracle decoding reproduces all validation gold spans exactly.
Cell 17 completed. No NER training has started.


In [18]:
# Cell 18: Define an efficient candidate-limited character-span decoder,
# validation threshold calibration helpers, and a gold-target oracle test.

import numpy as np

# --------------------------------------------------
# 1. Configure bounded candidate decoding
# --------------------------------------------------
S3_MAX_BOUNDARY_CANDIDATES = 512
S3_MAX_END_OPTIONS_PER_START = 16
S3_THRESHOLD_GRID = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]

if S3_MAX_BOUNDARY_CANDIDATES < 1:
    raise ValueError("Boundary candidate limit must be positive.")

if S3_MAX_END_OPTIONS_PER_START < 1:
    raise ValueError("End candidate limit must be positive.")

print("=" * 65)
print("S3: EFFICIENT STRICT-SPAN DECODER")
print("=" * 65)
print("Max boundary candidates per label:", S3_MAX_BOUNDARY_CANDIDATES)
print("Max end options per start:", S3_MAX_END_OPTIONS_PER_START)
print("Threshold grid:", S3_THRESHOLD_GRID)


# --------------------------------------------------
# 2. Decode one entity label from character probabilities
# --------------------------------------------------
def decode_one_label_fast(
    start_scores,
    end_scores,
    label_id,
    threshold,
    max_span_chars=MAX_SPAN_CHARS,
    max_boundary_candidates=S3_MAX_BOUNDARY_CANDIDATES,
    max_end_options=S3_MAX_END_OPTIONS_PER_START,
):
    """
    Decode character probabilities for one entity label.

    start_scores/end_scores: [characters, labels]
    Return: [(start, end_exclusive, label), ...]

    The candidate limits bound worst-case work when probabilities are
    diffuse. Every returned span uses end-exclusive character offsets.
    """

    start_scores = np.asarray(start_scores)
    end_scores = np.asarray(end_scores)

    if start_scores.ndim != 2 or end_scores.ndim != 2:
        raise ValueError("Boundary scores must be two-dimensional.")

    if start_scores.shape != end_scores.shape:
        raise ValueError("Start/end score shapes differ.")

    if not (0 <= label_id < NUM_LABELS):
        raise ValueError(f"Invalid label ID: {label_id}")

    if not np.isfinite(start_scores[:, label_id]).all():
        raise FloatingPointError("Non-finite start probabilities.")

    if not np.isfinite(end_scores[:, label_id]).all():
        raise FloatingPointError("Non-finite end probabilities.")

    n_chars = start_scores.shape[0]
    label = ID_TO_LABEL[label_id]

    start_scores_for_label = start_scores[:, label_id]
    end_scores_for_label = end_scores[:, label_id]

    start_indices = np.flatnonzero(
        start_scores_for_label >= threshold
    )
    end_indices = np.flatnonzero(
        end_scores_for_label >= threshold
    )

    # Keep the most confident candidate boundaries if predictions are dense.
    if len(start_indices) > max_boundary_candidates:
        selected = np.argpartition(
            start_scores_for_label[start_indices],
            -max_boundary_candidates,
        )[-max_boundary_candidates:]
        start_indices = start_indices[selected]

    if len(end_indices) > max_boundary_candidates:
        selected = np.argpartition(
            end_scores_for_label[end_indices],
            -max_boundary_candidates,
        )[-max_boundary_candidates:]
        end_indices = end_indices[selected]

    start_indices.sort()
    end_indices.sort()

    if len(start_indices) == 0 or len(end_indices) == 0:
        return []

    candidates = []

    for start in start_indices:
        low = np.searchsorted(end_indices, start, side="left")
        highest_legal_end = min(n_chars - 1, start + max_span_chars - 1)
        high = np.searchsorted(
            end_indices,
            highest_legal_end,
            side="right",
        )

        valid_ends = end_indices[low:high]

        if len(valid_ends) == 0:
            continue

        # Prefer high-confidence ends; on a score tie, consider shorter spans first.
        if len(valid_ends) > max_end_options:
            end_confidences = end_scores_for_label[valid_ends]

            selected = np.argpartition(
                end_confidences,
                -max_end_options,
            )[-max_end_options:]

            valid_ends = valid_ends[selected]

        valid_ends = sorted(
            (int(end) for end in valid_ends),
            key=lambda end: (
                -float(end_scores_for_label[end]),
                end,
            ),
        )

        for end_index in valid_ends:
            start_probability = float(start_scores_for_label[start])
            end_probability = float(end_scores_for_label[end_index])

            confidence = np.sqrt(
                max(0.0, start_probability)
                * max(0.0, end_probability)
            )

            candidates.append((
                -confidence,
                end_index - int(start) + 1,
                int(start),
                int(end_index),
            ))

    # Prefer high confidence, then shorter spans and earlier positions.
    candidates.sort()

    selected_spans = []
    used_starts = set()
    used_ends = set()

    for negative_confidence, length, start, end_index in candidates:
        end_exclusive = end_index + 1

        if start in used_starts or end_index in used_ends:
            continue

        # Prevent duplicate or overlapping spans for the same label.
        if any(
            start < existing_end and existing_start < end_exclusive
            for existing_start, existing_end in selected_spans
        ):
            continue

        selected_spans.append((start, end_exclusive))
        used_starts.add(start)
        used_ends.add(end_index)

    return [
        (start, end, label)
        for start, end in sorted(selected_spans)
    ]


# --------------------------------------------------
# 3. Decode all labels for one document
# --------------------------------------------------
def decode_document_fast(
    start_scores,
    end_scores,
    label_thresholds,
):
    if len(label_thresholds) != NUM_LABELS:
        raise ValueError(
            f"Expected {NUM_LABELS} thresholds, got {len(label_thresholds)}."
        )

    decoded = []

    for label_id in range(NUM_LABELS):
        decoded.extend(
            decode_one_label_fast(
                start_scores=start_scores,
                end_scores=end_scores,
                label_id=label_id,
                threshold=float(label_thresholds[label_id]),
            )
        )

    return sorted(decoded, key=lambda span: (span[0], span[1], span[2]))


# --------------------------------------------------
# 4. Tune thresholds for one label using validation spans
# --------------------------------------------------
def tune_one_label_threshold(
    document_probabilities,
    gold_spans,
    label_id,
    threshold_grid=S3_THRESHOLD_GRID,
):
    """
    Tune a shared start/end probability threshold for one label.
    Uses validation gold spans only; never uses fold3 test annotations.
    """

    label = ID_TO_LABEL[label_id]

    gold_label_spans = {
        span for span in gold_spans
        if span[3] == label
    }

    best_result = None

    for threshold in threshold_grid:
        predicted_label_spans = set()

        for doc_id, scores in document_probabilities.items():
            decoded = decode_one_label_fast(
                scores["start"],
                scores["end"],
                label_id=label_id,
                threshold=threshold,
            )

            predicted_label_spans.update(
                (doc_id, start, end, predicted_label)
                for start, end, predicted_label in decoded
            )

        tp = len(predicted_label_spans & gold_label_spans)
        fp = len(predicted_label_spans - gold_label_spans)
        fn = len(gold_label_spans - predicted_label_spans)

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        # On ties, prefer better precision, fewer predictions, then higher threshold.
        ranking = (f1, precision, -len(predicted_label_spans), threshold)

        if best_result is None or ranking > best_result["ranking"]:
            best_result = {
                "threshold": float(threshold),
                "f1": float(f1),
                "precision": float(precision),
                "recall": float(recall),
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "ranking": ranking,
            }

    return best_result


# --------------------------------------------------
# 5. Aggregate document-level strict span metrics
# --------------------------------------------------
def evaluate_document_probabilities(
    document_probabilities,
    gold_spans,
    label_thresholds,
):
    predicted_spans = set()

    for doc_id, scores in document_probabilities.items():
        decoded = decode_document_fast(
            scores["start"],
            scores["end"],
            label_thresholds,
        )

        predicted_spans.update(
            (doc_id, start, end, label)
            for start, end, label in decoded
        )

    metrics = calculate_strict_span_metrics(
        gold_spans,
        predicted_spans,
    )

    return metrics, predicted_spans


# --------------------------------------------------
# 6. Regression test the optimized decoder on gold targets
# --------------------------------------------------
VAL_ORACLE_SPANS_FAST = set()

for window in VAL_WINDOWS:
    local_decoded = decode_document_fast(
        window["start_targets"].astype(np.float32),
        window["end_targets"].astype(np.float32),
        label_thresholds=[0.5] * NUM_LABELS,
    )

    for start, end, label in local_decoded:
        VAL_ORACLE_SPANS_FAST.add((
            window["doc_id"],
            window["char_start"] + start,
            window["char_start"] + end,
            label,
        ))

oracle_true_positives = len(VAL_ORACLE_SPANS_FAST & VAL_GOLD_SPANS)
oracle_false_positives = len(VAL_ORACLE_SPANS_FAST - VAL_GOLD_SPANS)
oracle_false_negatives = len(VAL_GOLD_SPANS - VAL_ORACLE_SPANS_FAST)

print("\nOracle regression test:")
print("Gold spans:", len(VAL_GOLD_SPANS))
print("Decoded spans:", len(VAL_ORACLE_SPANS_FAST))
print("True positives:", oracle_true_positives)
print("False positives:", oracle_false_positives)
print("False negatives:", oracle_false_negatives)

if VAL_ORACLE_SPANS_FAST != VAL_GOLD_SPANS:
    print("Examples missed:", list(VAL_GOLD_SPANS - VAL_ORACLE_SPANS_FAST)[:10])
    print("Examples extra:", list(VAL_ORACLE_SPANS_FAST - VAL_GOLD_SPANS)[:10])
    raise RuntimeError(
        "Optimized decoder failed the oracle regression test. "
        "Training is blocked until decoding is corrected."
    )

print("\nPASS: optimized decoder preserves exact gold-span reconstruction.")
print("Threshold-calibration helpers are ready for validation.")
print("Cell 18 completed. No NER training has started.")

S3: EFFICIENT STRICT-SPAN DECODER
Max boundary candidates per label: 512
Max end options per start: 16
Threshold grid: [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

Oracle regression test:
Gold spans: 1861
Decoded spans: 1861
True positives: 1861
False positives: 0
False negatives: 0

PASS: optimized decoder preserves exact gold-span reconstruction.
Threshold-calibration helpers are ready for validation.
Cell 18 completed. No NER training has started.


In [19]:
# Cell 19: Implement efficient non-overlapping span selection, aggregate
# boundary probabilities across overlapping windows, and test validation inference.

import gc
import numpy as np
import torch

# --------------------------------------------------
# 1. Use an interval occupancy array for efficient decoding
# --------------------------------------------------
def decode_one_label_fast(
    start_scores,
    end_scores,
    label_id,
    threshold,
    max_span_chars=MAX_SPAN_CHARS,
    max_boundary_candidates=S3_MAX_BOUNDARY_CANDIDATES,
    max_end_options=S3_MAX_END_OPTIONS_PER_START,
):
    """Decode one label into end-exclusive spans without same-label overlap."""

    start_scores = np.asarray(start_scores)
    end_scores = np.asarray(end_scores)

    if start_scores.ndim != 2 or end_scores.shape != start_scores.shape:
        raise ValueError("Invalid start/end score shapes.")

    if not (0 <= label_id < NUM_LABELS):
        raise ValueError(f"Invalid label ID: {label_id}")

    starts_all = start_scores[:, label_id]
    ends_all = end_scores[:, label_id]

    if not np.isfinite(starts_all).all() or not np.isfinite(ends_all).all():
        raise FloatingPointError("Non-finite boundary scores.")

    starts = np.flatnonzero(starts_all >= threshold)
    ends = np.flatnonzero(ends_all >= threshold)

    def retain_top(indices, scores, limit):
        if len(indices) > limit:
            selected = np.argpartition(scores[indices], -limit)[-limit:]
            indices = indices[selected]
        return np.sort(indices)

    starts = retain_top(starts, starts_all, max_boundary_candidates)
    ends = retain_top(ends, ends_all, max_boundary_candidates)

    if not len(starts) or not len(ends):
        return []

    candidates = []

    for start in starts:
        left = np.searchsorted(ends, start, side="left")
        right = np.searchsorted(
            ends,
            min(len(starts_all) - 1, start + max_span_chars - 1),
            side="right",
        )

        valid_ends = ends[left:right]

        if not len(valid_ends):
            continue

        valid_ends = retain_top(valid_ends, ends_all, max_end_options)

        for end_idx in valid_ends:
            confidence = np.sqrt(
                max(0.0, float(starts_all[start]))
                * max(0.0, float(ends_all[end_idx]))
            )

            candidates.append((
                -confidence,
                int(end_idx - start + 1),
                int(start),
                int(end_idx),
            ))

    candidates.sort()

    # A Boolean occupancy vector avoids repeatedly scanning a list
    # of previously selected intervals.
    occupied = np.zeros(len(starts_all), dtype=np.bool_)
    chosen = []
    used_starts = set()
    used_ends = set()

    for neg_confidence, width, start, end_idx in candidates:
        if start in used_starts or end_idx in used_ends:
            continue

        end_exclusive = end_idx + 1

        if occupied[start:end_exclusive].any():
            continue

        chosen.append((start, end_exclusive, ID_TO_LABEL[label_id]))
        occupied[start:end_exclusive] = True
        used_starts.add(start)
        used_ends.add(end_idx)

    return sorted(chosen)


# --------------------------------------------------
# 2. Aggregate overlapping-window predictions at document level
# --------------------------------------------------
def predict_document_probabilities(windows, loader, model):
    """
    Returns:
        {
            doc_id: {
                "start": float32 array [document_characters, labels],
                "end":   float32 array [document_characters, labels]
            }
        }

    The loader must have shuffle=False and match the supplied window order.
    """

    model.eval()

    sums = {}
    counts = {}
    window_cursor = 0

    with torch.inference_mode():
        for batch in loader:
            tensor_batch = {
                key: value.to(DEVICE, non_blocking=True)
                for key, value in batch.items()
                if torch.is_tensor(value)
            }

            start_logits, end_logits = s3_forward_cached(
                model,
                cached_token_hidden=tensor_batch["cached_token_hidden"],
                char_ids=tensor_batch["char_ids"],
                char_to_token=tensor_batch["char_to_token"],
            )

            start_probs = torch.sigmoid(start_logits.float()).cpu().numpy()
            end_probs = torch.sigmoid(end_logits.float()).cpu().numpy()

            batch_size = len(batch["doc_ids"])

            if start_probs.shape != (
                batch_size,
                batch["char_ids"].shape[1],
                NUM_LABELS,
            ):
                raise RuntimeError("Unexpected validation output shape.")

            for i in range(batch_size):
                if window_cursor >= len(windows):
                    raise RuntimeError("Loader returned more windows than expected.")

                window = windows[window_cursor]
                window_cursor += 1

                if batch["doc_ids"][i] != window["doc_id"]:
                    raise RuntimeError(
                        "Loader/window order mismatch. Use shuffle=False."
                    )

                doc_id = window["doc_id"]
                char_start = window["char_start"]
                char_end = window["char_end"]
                char_length = len(window["char_ids"])
                document_length = len(ALL_TEXTS[doc_id])

                if char_end - char_start != char_length:
                    raise RuntimeError(
                        f"Character-bound inconsistency for document {doc_id}."
                    )

                if doc_id not in sums:
                    sums[doc_id] = {
                        "start": np.zeros(
                            (document_length, NUM_LABELS),
                            dtype=np.float32,
                        ),
                        "end": np.zeros(
                            (document_length, NUM_LABELS),
                            dtype=np.float32,
                        ),
                    }
                    counts[doc_id] = np.zeros(
                        document_length,
                        dtype=np.int32,
                    )

                sums[doc_id]["start"][char_start:char_end] += (
                    start_probs[i, :char_length]
                )
                sums[doc_id]["end"][char_start:char_end] += (
                    end_probs[i, :char_length]
                )
                counts[doc_id][char_start:char_end] += 1

            del tensor_batch, start_logits, end_logits
            del start_probs, end_probs

    if window_cursor != len(windows):
        raise RuntimeError(
            f"Processed {window_cursor} of {len(windows)} windows."
        )

    document_probabilities = {}

    for doc_id, probability_sums in sums.items():
        char_counts = counts[doc_id]

        if np.any(char_counts == 0):
            missing_positions = np.flatnonzero(char_counts == 0)[:10]
            raise RuntimeError(
                f"Some document characters lack predictions for {doc_id}: "
                f"{missing_positions.tolist()}"
            )

        document_probabilities[doc_id] = {
            "start": (
                probability_sums["start"] / char_counts[:, None]
            ),
            "end": (
                probability_sums["end"] / char_counts[:, None]
            ),
        }

    return document_probabilities


# --------------------------------------------------
# 3. Run the end-to-end validation inference smoke test
# --------------------------------------------------
print("=" * 65)
print("S3: VALIDATION INFERENCE SMOKE TEST")
print("=" * 65)

VAL_PROBABILITIES_INITIAL = predict_document_probabilities(
    windows=VAL_WINDOWS,
    loader=VAL_CACHED_LOADER,
    model=S3_MODEL,
)

if set(VAL_PROBABILITIES_INITIAL) != set(VAL_DOC_IDS):
    raise RuntimeError(
        "Predicted validation document IDs do not match validation IDs."
    )

for doc_id, probabilities in VAL_PROBABILITIES_INITIAL.items():
    expected_length = len(ALL_TEXTS[doc_id])

    for boundary_name in ("start", "end"):
        scores = probabilities[boundary_name]

        if scores.shape != (expected_length, NUM_LABELS):
            raise RuntimeError(
                f"Wrong {boundary_name} probability shape for {doc_id}: "
                f"{scores.shape}"
            )

        if not np.isfinite(scores).all():
            raise FloatingPointError(
                f"Non-finite {boundary_name} probabilities in {doc_id}."
            )

        if scores.min() < 0 or scores.max() > 1:
            raise ValueError("Probability values must be between 0 and 1.")

print("Validation documents predicted:", len(VAL_PROBABILITIES_INITIAL))
print("Expected validation documents:", len(VAL_DOC_IDS))
print("All character positions received aggregated probabilities.")

# Report probabilities without using these scores to select a checkpoint.
start_all = np.concatenate([
    result["start"].reshape(-1)
    for result in VAL_PROBABILITIES_INITIAL.values()
])
end_all = np.concatenate([
    result["end"].reshape(-1)
    for result in VAL_PROBABILITIES_INITIAL.values()
])

print("\nInitial-model probability diagnostics:")
print("Start probability percentiles:", np.percentile(start_all, [50, 90, 95, 99]).round(4))
print("End probability percentiles:", np.percentile(end_all, [50, 90, 95, 99]).round(4))
print("Start probabilities >= 0.5:", f"{100 * np.mean(start_all >= 0.5):.2f}%")
print("End probabilities >= 0.5:", f"{100 * np.mean(end_all >= 0.5):.2f}%")

print("\nPASS: validation inference and character-level aggregation verified.")
print("No NER training has started.")
print("Cell 19 completed.")

gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

S3: VALIDATION INFERENCE SMOKE TEST
Validation documents predicted: 14
Expected validation documents: 14
All character positions received aggregated probabilities.

Initial-model probability diagnostics:
Start probability percentiles: [0.492  0.6642 0.7076 0.7791]
End probability percentiles: [0.5036 0.677  0.7189 0.7861]
Start probabilities >= 0.5: 47.70%
End probabilities >= 0.5: 50.99%

PASS: validation inference and character-level aggregation verified.
No NER training has started.
Cell 19 completed.


In [20]:
# Cell 20: Initialize start/end prediction biases from training-set boundary
# prevalence and prepare a lower threshold grid for strict-span calibration.

import gc
import numpy as np
import torch

# --------------------------------------------------
# 1. Estimate the weighted-BCE optimal starting probability per label
# --------------------------------------------------
if len(S3_POSITIVE_COUNTS) != NUM_LABELS:
    raise ValueError("Positive-count vector has the wrong number of labels.")

positive_counts = np.asarray(S3_POSITIVE_COUNTS, dtype=np.float64)
negative_counts = np.asarray(S3_NEGATIVE_COUNTS, dtype=np.float64)
positive_weights = np.asarray(S3_POSITIVE_WEIGHTS, dtype=np.float64)

if np.any(positive_counts <= 0):
    raise ValueError("At least one label has zero positive training boundaries.")

if np.any(negative_counts <= 0):
    raise ValueError("At least one label has zero negative positions.")

# Raw positive frequency in valid training characters.
raw_prevalence = positive_counts / (positive_counts + negative_counts)

# For weighted binary cross-entropy, the optimum probability is:
# q = w*p / (1 - p + w*p)
weighted_prevalence = (
    positive_weights * raw_prevalence
    / (1.0 - raw_prevalence + positive_weights * raw_prevalence)
)

# Clamp solely for numerical safety before converting probability to logit.
weighted_prevalence = np.clip(weighted_prevalence, 1e-6, 1.0 - 1e-6)
initial_bias = np.log(
    weighted_prevalence / (1.0 - weighted_prevalence)
)

# --------------------------------------------------
# 2. Initialize both boundary heads consistently
# --------------------------------------------------
with torch.no_grad():
    start_bias_tensor = torch.as_tensor(
        initial_bias,
        dtype=S3_MODEL.start_head.bias.dtype,
        device=S3_MODEL.start_head.bias.device,
    )
    end_bias_tensor = torch.as_tensor(
        initial_bias,
        dtype=S3_MODEL.end_head.bias.dtype,
        device=S3_MODEL.end_head.bias.device,
    )

    S3_MODEL.start_head.bias.copy_(start_bias_tensor)
    S3_MODEL.end_head.bias.copy_(end_bias_tensor)

# Confirm bias values are finite.
if not (
    torch.isfinite(S3_MODEL.start_head.bias).all().item()
    and torch.isfinite(S3_MODEL.end_head.bias).all().item()
):
    raise FloatingPointError("Non-finite classifier bias after initialization.")

# --------------------------------------------------
# 3. Define a threshold grid appropriate for rare boundaries
# --------------------------------------------------
S3_THRESHOLD_GRID = [
    0.001, 0.002, 0.003, 0.005,
    0.008, 0.012, 0.020, 0.030,
    0.050, 0.080, 0.120, 0.200,
]

print("=" * 65)
print("S3: BOUNDARY-BIAS INITIALIZATION")
print("=" * 65)

for i, label in enumerate(LABELS):
    print(
        f"{label:12s} "
        f"raw_prevalence={raw_prevalence[i]:.6f} "
        f"weighted_target={weighted_prevalence[i]:.6f} "
        f"initial_bias={initial_bias[i]:.3f}"
    )

print("\nThreshold grid for validation tuning:", S3_THRESHOLD_GRID)

# --------------------------------------------------
# 4. Refresh optimizer gradients without updating parameters
# --------------------------------------------------
# The optimizer has not stepped yet, so no optimizer state needs resetting.
S3_OPTIMIZER.zero_grad(set_to_none=True)
S3_MODEL.zero_grad(set_to_none=True)

# --------------------------------------------------
# 5. Recheck initial validation probabilities after bias initialization
# --------------------------------------------------
VAL_PROBABILITIES_INITIAL = predict_document_probabilities(
    windows=VAL_WINDOWS,
    loader=VAL_CACHED_LOADER,
    model=S3_MODEL,
)

start_values = np.concatenate([
    result["start"].reshape(-1)
    for result in VAL_PROBABILITIES_INITIAL.values()
])

end_values = np.concatenate([
    result["end"].reshape(-1)
    for result in VAL_PROBABILITIES_INITIAL.values()
])

print("\nInitial validation probability diagnostics:")
print(
    "Start percentiles:",
    np.percentile(start_values, [50, 90, 95, 99]).round(5)
)
print(
    "End percentiles:",
    np.percentile(end_values, [50, 90, 95, 99]).round(5)
)

print("\nThreshold candidate rates:")
for threshold in [0.005, 0.01, 0.02, 0.05, 0.10, 0.20, 0.50]:
    start_rate = 100.0 * np.mean(start_values >= threshold)
    end_rate = 100.0 * np.mean(end_values >= threshold)

    print(
        f"threshold={threshold:>5.3f} | "
        f"start={start_rate:>6.2f}% | "
        f"end={end_rate:>6.2f}%"
    )

print("\nCell 20 completed. No NER training has started.")

del start_values, end_values
gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

S3: BOUNDARY-BIAS INITIALIZATION
APP          raw_prevalence=0.000171 weighted_target=0.005114 initial_bias=-5.271
RESP         raw_prevalence=0.000174 weighted_target=0.005207 initial_bias=-5.253
A.COUNSEL    raw_prevalence=0.000084 weighted_target=0.002524 initial_bias=-5.979
R.COUNSEL    raw_prevalence=0.000071 weighted_target=0.002126 initial_bias=-6.151
JUDGE        raw_prevalence=0.000102 weighted_target=0.003053 initial_bias=-5.788
WIT          raw_prevalence=0.000095 weighted_target=0.002828 initial_bias=-5.865
AUTH         raw_prevalence=0.000460 weighted_target=0.013613 initial_bias=-4.283
COURT        raw_prevalence=0.000452 weighted_target=0.013380 initial_bias=-4.301
STAT         raw_prevalence=0.000563 weighted_target=0.016613 initial_bias=-4.081
PREC         raw_prevalence=0.000546 weighted_target=0.016123 initial_bias=-4.111
DATE         raw_prevalence=0.000786 weighted_target=0.023056 initial_bias=-3.747
CASENO       raw_prevalence=0.000290 weighted_target=0.008622 ini

In [21]:
# Cell 21: Train the Architecture C S3 NER adapter and prediction heads,
# tune per-label thresholds on validation data, and checkpoint the best
# model using document-level strict span macro-F1.

import gc
import json
import time
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm

# --------------------------------------------------
# 1. Validate all training prerequisites
# --------------------------------------------------
required_objects = [
    "S3_MODEL",
    "S3_OPTIMIZER",
    "S3_SCHEDULER",
    "TRAIN_CACHED_LOADER",
    "VAL_CACHED_LOADER",
    "TRAIN_WINDOWS",
    "VAL_WINDOWS",
    "VAL_GOLD_SPANS",
    "S3_POS_WEIGHT_TENSOR",
    "S3_MAX_GRAD_NORM",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        f"Run the previous cells first. Missing: {missing_objects}"
    )

if any(p.requires_grad for p in S3_MODEL.backbone.parameters()):
    raise RuntimeError("The pretrained LSTM must remain frozen.")

# --------------------------------------------------
# 2. Set a practical training budget
# --------------------------------------------------
S3_MAX_EPOCHS = 4
S3_EARLY_STOPPING_PATIENCE = 2

# Threshold tuning uses validation annotations only.
S3_VALIDATION_THRESHOLD_GRID = [
    0.002, 0.005, 0.010, 0.020,
    0.030, 0.050, 0.080, 0.120,
    0.200, 0.300, 0.400, 0.500,
]

S3_OUTPUT_DIR = Path("/kaggle/working/S3_Architecture_C")
S3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

S3_BEST_CHECKPOINT_PATH = (
    S3_OUTPUT_DIR / "S3_best_trainable_checkpoint.pt"
)
S3_HISTORY_PATH = S3_OUTPUT_DIR / "S3_training_history.json"

# Save only trainable parameters; the Phase 1 LSTM stays in its original
# attached dataset and will be loaded again from the verified artifact.
S3_TRAINABLE_PARAMETER_NAMES = {
    name
    for name, parameter in S3_MODEL.named_parameters()
    if parameter.requires_grad
}

if not S3_TRAINABLE_PARAMETER_NAMES:
    raise RuntimeError("No trainable parameters were found.")

S3_HISTORY = []
S3_BEST_MACRO_F1 = -1.0
S3_BEST_EPOCH = None
S3_EPOCHS_WITHOUT_IMPROVEMENT = 0
S3_BEST_THRESHOLDS = None

print("=" * 70)
print("S3: ARCHITECTURE C LSTM + LOW-RANK ADAPTATION")
print("=" * 70)
print("Device:", DEVICE)
print("Training windows:", len(TRAIN_WINDOWS))
print("Validation windows:", len(VAL_WINDOWS))
print("Max epochs:", S3_MAX_EPOCHS)
print("Trainable parameters:", len(S3_TRAINABLE_PARAMETER_NAMES))
print("Validation threshold candidates:",
      S3_VALIDATION_THRESHOLD_GRID)

# --------------------------------------------------
# 3. Define one training epoch
# --------------------------------------------------
def train_s3_one_epoch():
    S3_MODEL.train()

    running_loss = 0.0
    running_start_loss = 0.0
    running_end_loss = 0.0
    examples_seen = 0
    steps_completed = 0

    progress = tqdm(
        TRAIN_CACHED_LOADER,
        desc="S3 training",
        leave=False,
    )

    for batch in progress:
        device_batch = {
            key: value.to(DEVICE, non_blocking=True)
            for key, value in batch.items()
            if torch.is_tensor(value)
        }

        S3_OPTIMIZER.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda"),
        ):
            start_logits, end_logits = s3_forward_cached(
                S3_MODEL,
                cached_token_hidden=device_batch[
                    "cached_token_hidden"
                ],
                char_ids=device_batch["char_ids"],
                char_to_token=device_batch["char_to_token"],
            )

            loss, start_loss, end_loss = s3_boundary_loss(
                start_logits=start_logits,
                end_logits=end_logits,
                start_targets=device_batch["start_targets"],
                end_targets=device_batch["end_targets"],
                loss_mask=device_batch["loss_mask"],
            )

        if not torch.isfinite(loss).item():
            raise FloatingPointError(
                f"Non-finite training loss at step {steps_completed}."
            )

        # All trainable parameters remain FP32; autocast handles compute
        # precision, while GradScaler protects small gradients.
        if DEVICE.type == "cuda":
            if "S3_SCALER" not in globals():
                S3_SCALER = torch.amp.GradScaler("cuda")

            S3_SCALER.scale(loss).backward()
            S3_SCALER.unscale_(S3_OPTIMIZER)
        else:
            loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(
            [
                p for p in S3_MODEL.parameters()
                if p.requires_grad
            ],
            max_norm=S3_MAX_GRAD_NORM,
        )

        if not torch.isfinite(grad_norm).item():
            raise FloatingPointError(
                f"Non-finite gradient norm at step {steps_completed}."
            )

        if DEVICE.type == "cuda":
            S3_SCALER.step(S3_OPTIMIZER)
            S3_SCALER.update()
        else:
            S3_OPTIMIZER.step()

        running_loss += float(loss.detach().cpu())
        running_start_loss += float(start_loss.cpu())
        running_end_loss += float(end_loss.cpu())

        examples_seen += int(device_batch["input_ids"].shape[0])
        steps_completed += 1

        progress.set_postfix({
            "loss": f"{running_loss / steps_completed:.4f}",
            "lr": f"{S3_OPTIMIZER.param_groups[0]['lr']:.2e}",
        })

        del device_batch, start_logits, end_logits
        del loss, start_loss, end_loss, grad_norm

    if steps_completed == 0:
        raise RuntimeError("No training steps were completed.")

    return {
        "train_loss": running_loss / steps_completed,
        "train_start_loss": running_start_loss / steps_completed,
        "train_end_loss": running_end_loss / steps_completed,
        "steps": steps_completed,
        "windows_seen": examples_seen,
    }


# --------------------------------------------------
# 4. Tune a single boundary threshold for each label
# --------------------------------------------------
def tune_s3_validation_thresholds(document_probabilities):
    threshold_results = {}
    thresholds = []

    for label_id, label in enumerate(LABELS):
        result = tune_one_label_threshold(
            document_probabilities=document_probabilities,
            gold_spans=VAL_GOLD_SPANS,
            label_id=label_id,
            threshold_grid=S3_VALIDATION_THRESHOLD_GRID,
        )

        threshold_results[label] = result
        thresholds.append(result["threshold"])

    return thresholds, threshold_results


# --------------------------------------------------
# 5. Train, validate, and save the best checkpoint
# --------------------------------------------------
for epoch in range(1, S3_MAX_EPOCHS + 1):
    epoch_start = time.time()

    print("\n" + "=" * 70)
    print(f"S3 EPOCH {epoch}/{S3_MAX_EPOCHS}")
    print("=" * 70)

    training_metrics = train_s3_one_epoch()

    # Compute probabilities for each original validation document.
    VAL_PROBABILITIES = predict_document_probabilities(
        windows=VAL_WINDOWS,
        loader=VAL_CACHED_LOADER,
        model=S3_MODEL,
    )

    if set(VAL_PROBABILITIES) != set(VAL_DOC_IDS):
        raise RuntimeError(
            "Validation prediction IDs do not match validation documents."
        )

    # Calibrate one threshold per label using validation gold only.
    tuned_thresholds, threshold_results = tune_s3_validation_thresholds(
        VAL_PROBABILITIES
    )

    val_metrics, val_predicted_spans = evaluate_document_probabilities(
        document_probabilities=VAL_PROBABILITIES,
        gold_spans=VAL_GOLD_SPANS,
        label_thresholds=tuned_thresholds,
    )

    val_macro_f1 = float(val_metrics["macro_f1"])

    if not np.isfinite(val_macro_f1):
        raise FloatingPointError("Validation macro-F1 is not finite.")

    # A larger macro-F1 is better. The first epoch always establishes
    # a baseline checkpoint.
    improved = val_macro_f1 > S3_BEST_MACRO_F1 + 1e-8

    current_lr = float(S3_OPTIMIZER.param_groups[0]["lr"])
    S3_SCHEDULER.step(val_macro_f1)

    epoch_duration = time.time() - epoch_start

    epoch_record = {
        "epoch": epoch,
        **training_metrics,
        "validation_strict_span_macro_f1": val_macro_f1,
        "validation_gold_spans": int(val_metrics["gold_count"]),
        "validation_predicted_spans": int(
            val_metrics["predicted_count"]
        ),
        "validation_exact_true_positives": int(
            val_metrics["exact_true_positives"]
        ),
        "validation_per_label": val_metrics["per_label"],
        "validation_thresholds": {
            label: float(tuned_thresholds[i])
            for i, label in enumerate(LABELS)
        },
        "threshold_search_results": threshold_results,
        "learning_rate": current_lr,
        "duration_seconds": epoch_duration,
    }

    S3_HISTORY.append(epoch_record)

    print("\nTraining results:")
    print(f"  Training loss:       {training_metrics['train_loss']:.5f}")
    print(f"  Start-boundary loss: {training_metrics['train_start_loss']:.5f}")
    print(f"  End-boundary loss:   {training_metrics['train_end_loss']:.5f}")
    print(f"  Steps completed:     {training_metrics['steps']}")
    print(f"  Epoch duration:      {epoch_duration:.1f} seconds")

    print("\nValidation results:")
    print(f"  Strict Span Macro-F1: {val_macro_f1:.6f}")
    print(f"  Gold spans:           {val_metrics['gold_count']}")
    print(f"  Predicted spans:      {val_metrics['predicted_count']}")
    print(f"  Exact true positives: {val_metrics['exact_true_positives']}")

    print("\nPer-label validation metrics:")
    print(
        f"{'Label':12s} {'P':>8s} {'R':>8s} {'F1':>8s} "
        f"{'TP':>6s} {'FP':>6s} {'FN':>6s} {'Thr':>7s}"
    )

    for label_id, label in enumerate(LABELS):
        result = val_metrics["per_label"][label]
        threshold = tuned_thresholds[label_id]

        print(
            f"{label:12s} "
            f"{result['precision']:8.4f} "
            f"{result['recall']:8.4f} "
            f"{result['f1']:8.4f} "
            f"{result['tp']:6d} "
            f"{result['fp']:6d} "
            f"{result['fn']:6d} "
            f"{threshold:7.3f}"
        )

    # Update the best validation checkpoint.
    if improved:
        S3_BEST_MACRO_F1 = val_macro_f1
        S3_BEST_EPOCH = epoch
        S3_BEST_THRESHOLDS = list(tuned_thresholds)
        S3_EPOCHS_WITHOUT_IMPROVEMENT = 0

        full_state = S3_MODEL.state_dict()
        trainable_state = {
            name: full_state[name].detach().cpu().clone()
            for name in S3_TRAINABLE_PARAMETER_NAMES
        }

        checkpoint_payload = {
            "experiment_id": "CS781_Phase2_S3_Architecture_C_LSTM",
            "architecture_name": "Architecture_C_Deep_LSTM_Language_Model",
            "phase1_checkpoint": str(ARCH_C_FINAL_CHECKPOINT),
            "phase1_tokenizer": str(PHASE1_TOKENIZER_PATH),
            "epoch": epoch,
            "validation_strict_span_macro_f1": val_macro_f1,
            "validation_metrics": val_metrics,
            "thresholds": {
                label: float(tuned_thresholds[i])
                for i, label in enumerate(LABELS)
            },
            "labels": LABELS,
            "label_to_id": LABEL_TO_ID,
            "char_to_id": CHAR_TO_ID,
            "char_vocab_size": CHAR_VOCAB_SIZE,
            "model_config": {
                "lorma_rank": LORMA_RANK,
                "lorma_alpha": LORMA_ALPHA,
                "char_embed_dim": CHAR_EMBED_DIM,
                "ner_hidden_dim": NER_HIDDEN_DIM,
                "dropout": NER_DROPOUT,
                "max_span_chars": MAX_SPAN_CHARS,
                "max_tokens": MAX_TOKENS,
                "window_stride": WINDOW_STRIDE,
            },
            "trainable_state_dict": trainable_state,
        }

        torch.save(
            checkpoint_payload,
            S3_BEST_CHECKPOINT_PATH,
        )

        print("\nNEW BEST CHECKPOINT SAVED:")
        print(" ", S3_BEST_CHECKPOINT_PATH)
        print(f"  Best validation macro-F1: {S3_BEST_MACRO_F1:.6f}")

    else:
        S3_EPOCHS_WITHOUT_IMPROVEMENT += 1
        print(
            "\nNo validation improvement. "
            f"Patience: {S3_EPOCHS_WITHOUT_IMPROVEMENT}/"
            f"{S3_EARLY_STOPPING_PATIENCE}"
        )

    # Persist the training history after every epoch for recovery.
    with S3_HISTORY_PATH.open("w", encoding="utf-8") as f:
        json.dump(S3_HISTORY, f, ensure_ascii=False, indent=2)

    print("Training history saved:", S3_HISTORY_PATH)

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    gc.collect()

    if S3_EPOCHS_WITHOUT_IMPROVEMENT >= S3_EARLY_STOPPING_PATIENCE:
        print("\nEarly stopping triggered.")
        break


# --------------------------------------------------
# 6. Restore the best saved trainable weights
# --------------------------------------------------
if not S3_BEST_CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        "No best checkpoint was saved. Review the training output."
    )

S3_BEST_CHECKPOINT = torch.load(
    S3_BEST_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

best_trainable_state = S3_BEST_CHECKPOINT["trainable_state_dict"]

load_result = S3_MODEL.load_state_dict(
    best_trainable_state,
    strict=False,
)

expected_missing = (
    set(S3_MODEL.state_dict().keys())
    - S3_TRAINABLE_PARAMETER_NAMES
)

if set(load_result.missing_keys) != expected_missing:
    raise RuntimeError(
        "Unexpected missing model keys while restoring the best checkpoint."
    )

if load_result.unexpected_keys:
    raise RuntimeError(
        f"Unexpected checkpoint keys: {load_result.unexpected_keys}"
    )

S3_MODEL.eval()

S3_BEST_THRESHOLDS = [
    float(S3_BEST_CHECKPOINT["thresholds"][label])
    for label in LABELS
]

print("\n" + "=" * 70)
print("S3 TRAINING COMPLETE")
print("=" * 70)
print("Best epoch:", S3_BEST_CHECKPOINT["epoch"])
print(
    "Best validation Strict Span Macro-F1:",
    f"{S3_BEST_CHECKPOINT['validation_strict_span_macro_f1']:.6f}",
)
print("Restored best trainable weights:", True)
print("Checkpoint:", S3_BEST_CHECKPOINT_PATH)
print("History:", S3_HISTORY_PATH)
print("Training has finished; no test-fold gold annotations were used.")

S3: ARCHITECTURE C LSTM + LOW-RANK ADAPTATION
Device: cuda
Training windows: 903
Validation windows: 293
Max epochs: 4
Trainable parameters: 13
Validation threshold candidates: [0.002, 0.005, 0.01, 0.02, 0.03, 0.05, 0.08, 0.12, 0.2, 0.3, 0.4, 0.5]

S3 EPOCH 1/4


S3 training:   0%|          | 0/452 [00:00<?, ?it/s]


Training results:
  Training loss:       0.01096
  Start-boundary loss: 0.01208
  End-boundary loss:   0.00983
  Steps completed:     452
  Epoch duration:      28.4 seconds

Validation results:
  Strict Span Macro-F1: 0.303803
  Gold spans:           1861
  Predicted spans:      3247
  Exact true positives: 797

Per-label validation metrics:
Label               P        R       F1     TP     FP     FN     Thr
APP            0.0748   0.1959   0.1083     19    235     78   0.500
RESP           0.0636   0.1618   0.0913     11    162     57   0.500
A.COUNSEL      0.3077   0.5581   0.3967     24     54     19   0.400
R.COUNSEL      0.3600   0.6102   0.4528     36     64     23   0.400
JUDGE          0.0858   0.4348   0.1434     20    213     26   0.500
WIT            0.0789   0.1818   0.1101      6     70     27   0.400
AUTH           0.0943   0.2895   0.1422     66    634    162   0.500
COURT          0.4460   0.6434   0.5268    157    195     87   0.500
STAT           0.4954   0.5311   

S3 training:   0%|          | 0/452 [00:00<?, ?it/s]


Training results:
  Training loss:       0.00401
  Start-boundary loss: 0.00409
  End-boundary loss:   0.00394
  Steps completed:     452
  Epoch duration:      21.7 seconds

Validation results:
  Strict Span Macro-F1: 0.348148
  Gold spans:           1861
  Predicted spans:      3158
  Exact true positives: 879

Per-label validation metrics:
Label               P        R       F1     TP     FP     FN     Thr
APP            0.0735   0.2062   0.1084     20    252     77   0.500
RESP           0.0791   0.2941   0.1246     20    233     48   0.500
A.COUNSEL      0.2903   0.6279   0.3971     27     66     16   0.500
R.COUNSEL      0.4062   0.6610   0.5032     39     57     20   0.500
JUDGE          0.2077   0.5870   0.3068     27    103     19   0.500
WIT            0.0659   0.1818   0.0968      6     85     27   0.500
AUTH           0.1308   0.3114   0.1842     71    472    157   0.400
COURT          0.4951   0.6189   0.5501    151    154     93   0.500
STAT           0.5724   0.5574   

S3 training:   0%|          | 0/452 [00:00<?, ?it/s]


Training results:
  Training loss:       0.00291
  Start-boundary loss: 0.00296
  End-boundary loss:   0.00287
  Steps completed:     452
  Epoch duration:      20.0 seconds

Validation results:
  Strict Span Macro-F1: 0.347184
  Gold spans:           1861
  Predicted spans:      3123
  Exact true positives: 874

Per-label validation metrics:
Label               P        R       F1     TP     FP     FN     Thr
APP            0.0996   0.2474   0.1420     24    217     73   0.500
RESP           0.1029   0.2059   0.1373     14    122     54   0.500
A.COUNSEL      0.2778   0.6977   0.3974     30     78     13   0.500
R.COUNSEL      0.4095   0.7288   0.5244     43     62     16   0.500
JUDGE          0.2155   0.5435   0.3086     25     91     21   0.500
WIT            0.0938   0.2727   0.1395      9     87     24   0.500
AUTH           0.1201   0.2675   0.1658     61    447    167   0.400
COURT          0.5222   0.6270   0.5698    153    140     91   0.500
STAT           0.6229   0.6066   

S3 training:   0%|          | 0/452 [00:00<?, ?it/s]


Training results:
  Training loss:       0.00224
  Start-boundary loss: 0.00220
  End-boundary loss:   0.00227
  Steps completed:     452
  Epoch duration:      13.7 seconds

Validation results:
  Strict Span Macro-F1: 0.372295
  Gold spans:           1861
  Predicted spans:      2400
  Exact true positives: 827

Per-label validation metrics:
Label               P        R       F1     TP     FP     FN     Thr
APP            0.1398   0.1340   0.1368     13     80     84   0.300
RESP           0.2692   0.2059   0.2333     14     38     54   0.400
A.COUNSEL      0.2857   0.6512   0.3972     28     70     15   0.300
R.COUNSEL      0.4583   0.5593   0.5038     33     39     26   0.500
JUDGE          0.4146   0.3696   0.3908     17     24     29   0.400
WIT            0.0526   0.1212   0.0734      4     72     29   0.200
AUTH           0.1109   0.3070   0.1630     70    561    158   0.200
COURT          0.5516   0.6352   0.5905    155    126     89   0.500
STAT           0.6379   0.6066   

In [22]:
# Cell 22: Preserve the best Architecture C checkpoint as a fallback,
# correctly identify the current adapter method, and verify the saved artifact.

import json
from pathlib import Path
import torch

# --------------------------------------------------
# 1. Verify the current best checkpoint
# --------------------------------------------------
source_path = Path(S3_BEST_CHECKPOINT_PATH)

if not source_path.is_file():
    raise FileNotFoundError(
        f"Best S3 checkpoint not found: {source_path}"
    )

source_checkpoint = torch.load(
    source_path,
    map_location="cpu",
    weights_only=False,
)

if "trainable_state_dict" not in source_checkpoint:
    raise KeyError("The checkpoint has no trainable_state_dict.")

if "thresholds" not in source_checkpoint:
    raise KeyError("The checkpoint has no validation-selected thresholds.")

# --------------------------------------------------
# 2. Correctly describe this fallback experiment
# --------------------------------------------------
fallback_checkpoint = dict(source_checkpoint)

fallback_checkpoint["experiment_id"] = (
    "CS781_Phase2_S3_Architecture_C_AdditiveAdapter_Fallback"
)

fallback_checkpoint["adaptation_method"] = (
    "Additive low-rank hidden-state adapter; "
    "not the paper's multiplicative LoRMA method"
)

fallback_checkpoint["validation_metric"] = (
    "Document-level strict span macro-F1"
)

# --------------------------------------------------
# 3. Save an independent fallback file
# --------------------------------------------------
S3_FALLBACK_PATH = (
    Path("/kaggle/working/S3_Architecture_C")
    / "S3_additive_adapter_fallback.pt"
)

S3_FALLBACK_PATH.parent.mkdir(parents=True, exist_ok=True)

torch.save(fallback_checkpoint, S3_FALLBACK_PATH)

# Save an accompanying human-readable summary.
S3_FALLBACK_SUMMARY_PATH = (
    S3_FALLBACK_PATH.parent / "S3_additive_adapter_fallback_summary.json"
)

summary = {
    "experiment_id": fallback_checkpoint["experiment_id"],
    "architecture_name": fallback_checkpoint["architecture_name"],
    "adaptation_method": fallback_checkpoint["adaptation_method"],
    "best_epoch": fallback_checkpoint["epoch"],
    "validation_strict_span_macro_f1": (
        fallback_checkpoint["validation_strict_span_macro_f1"]
    ),
    "phase1_checkpoint": fallback_checkpoint["phase1_checkpoint"],
    "phase1_tokenizer": fallback_checkpoint["phase1_tokenizer"],
    "thresholds": fallback_checkpoint["thresholds"],
    "labels": fallback_checkpoint["labels"],
    "model_config": fallback_checkpoint["model_config"],
}

with S3_FALLBACK_SUMMARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

# --------------------------------------------------
# 4. Reload and verify the preserved artifact
# --------------------------------------------------
verified_fallback = torch.load(
    S3_FALLBACK_PATH,
    map_location="cpu",
    weights_only=False,
)

if (
    verified_fallback["validation_strict_span_macro_f1"]
    != fallback_checkpoint["validation_strict_span_macro_f1"]
):
    raise RuntimeError("Fallback metric verification failed.")

if set(verified_fallback["trainable_state_dict"]) != (
    S3_TRAINABLE_PARAMETER_NAMES
):
    raise RuntimeError("Fallback trainable parameter names do not match.")

print("=" * 65)
print("S3 FALLBACK PRESERVED")
print("=" * 65)
print("Best epoch:", verified_fallback["epoch"])
print(
    "Validation Strict Span Macro-F1:",
    f"{verified_fallback['validation_strict_span_macro_f1']:.6f}",
)
print("Method:", verified_fallback["adaptation_method"])
print("Checkpoint:", S3_FALLBACK_PATH)
print("Summary:", S3_FALLBACK_SUMMARY_PATH)
print("Trainable parameter entries:", len(verified_fallback["trainable_state_dict"]))
print("Verification: PASS")

S3 FALLBACK PRESERVED
Best epoch: 4
Validation Strict Span Macro-F1: 0.372295
Method: Additive low-rank hidden-state adapter; not the paper's multiplicative LoRMA method
Checkpoint: /kaggle/working/S3_Architecture_C/S3_additive_adapter_fallback.pt
Summary: /kaggle/working/S3_Architecture_C/S3_additive_adapter_fallback_summary.json
Trainable parameter entries: 13
Verification: PASS


In [23]:
# Cell 23: Implement true LoRMA+ multiplicative adaptation on the
# square recurrent gate matrices of the Phase 1 Architecture C LSTM.
# Keep the previous additive-adapter S3 checkpoint untouched.

import copy
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import parametrize

# --------------------------------------------------
# 1. Verify prerequisites and preserve the fallback
# --------------------------------------------------
if not ARCH_C_STATE_DICT:
    raise RuntimeError("Verified Phase 1 state dictionary is unavailable.")

if not S3_FALLBACK_PATH.is_file():
    raise FileNotFoundError(
        "The additive-adapter fallback checkpoint must exist before "
        "building the true LoRMA+ model."
    )

print("=" * 70)
print("S3: TRUE LoRMA+ MULTIPLICATIVE LSTM ADAPTATION")
print("=" * 70)

# --------------------------------------------------
# 2. Define the LoRMA+ parametrization for four LSTM gates
# --------------------------------------------------
class LoRMAPlusLSTMGateParametrization(nn.Module):
    """
    For each square recurrent gate matrix W0, form:

        W_eff = (I + (alpha / rank) * B @ A) @ W0

    Equivalently:
        W_eff = W0 + (alpha / rank) * B @ (A @ W0)

    PyTorch packs the four LSTM recurrent gates into a tensor shaped
    [4 * hidden_size, hidden_size]. This module applies the formula
    separately to its four square hidden_size x hidden_size blocks.
    """

    def __init__(self, hidden_size, rank=8, alpha=16.0):
        super().__init__()

        if not (0 < rank < hidden_size):
            raise ValueError("rank must be positive and smaller than hidden_size.")

        self.hidden_size = hidden_size
        self.rank = rank
        self.scale = float(alpha) / float(rank)

        # Four independent low-rank transformations, one per gate.
        self.A = nn.Parameter(
            torch.empty(4, rank, hidden_size)
        )
        self.B = nn.Parameter(
            torch.zeros(4, hidden_size, rank)
        )

        for gate_index in range(4):
            nn.init.kaiming_uniform_(
                self.A[gate_index],
                a=5 ** 0.5,
            )

    def forward(self, original_weight):
        expected_shape = (4 * self.hidden_size, self.hidden_size)

        if tuple(original_weight.shape) != expected_shape:
            raise ValueError(
                f"Expected recurrent weight shape {expected_shape}, "
                f"got {tuple(original_weight.shape)}."
            )

        gate_weights = original_weight.chunk(4, dim=0)
        adapted_gate_weights = []

        for gate_index, gate_weight in enumerate(gate_weights):
            low_rank_projection = (
                self.A[gate_index] @ gate_weight
            )

            multiplicative_update = (
                self.B[gate_index] @ low_rank_projection
            )

            adapted_gate_weights.append(
                gate_weight + self.scale * multiplicative_update
            )

        return torch.cat(adapted_gate_weights, dim=0)


# --------------------------------------------------
# 3. Reconstruct a clean Phase 1 backbone
# --------------------------------------------------
S3_LORMA_RANK = 8
S3_LORMA_ALPHA = 16.0

S3_LORMA_BACKBONE = ArchitectureCLSTMBackbone(
    ARCH_C_MODEL_CONFIG
)

base_load_result = S3_LORMA_BACKBONE.load_state_dict(
    ARCH_C_STATE_DICT,
    strict=True,
)

if base_load_result.missing_keys or base_load_result.unexpected_keys:
    raise RuntimeError(
        f"Phase 1 weight loading mismatch: {base_load_result}"
    )

# Freeze Phase 1 parameters BEFORE registering trainable parametrizations.
for parameter in S3_LORMA_BACKBONE.parameters():
    parameter.requires_grad_(False)

# --------------------------------------------------
# 4. Apply LoRMA+ to each layer's recurrent gate weights
# --------------------------------------------------
for layer_index in range(
    int(ARCH_C_MODEL_CONFIG["num_layers"])
):
    weight_name = f"weight_hh_l{layer_index}"

    parametrize.register_parametrization(
        S3_LORMA_BACKBONE.lstm,
        weight_name,
        LoRMAPlusLSTMGateParametrization(
            hidden_size=int(ARCH_C_MODEL_CONFIG["hidden_size"]),
            rank=S3_LORMA_RANK,
            alpha=S3_LORMA_ALPHA,
        ),
    )

# --------------------------------------------------
# 5. Define the character-level NER model using this backbone
# --------------------------------------------------
class ArchitectureCTrueLoRMAPlusNER(nn.Module):
    def __init__(self, backbone, source_ner_model):
        super().__init__()

        self.backbone = backbone
        self.backbone_hidden_size = int(
            ARCH_C_MODEL_CONFIG["hidden_size"]
        )

        # Transfer the trained NER head from the saved fallback.
        # Do not copy its additive hidden-state adapter.
        self.char_embedding = copy.deepcopy(
            source_ner_model.char_embedding
        )
        self.input_projection = copy.deepcopy(
            source_ner_model.input_projection
        )
        self.dropout = copy.deepcopy(source_ner_model.dropout)
        self.char_conv = copy.deepcopy(source_ner_model.char_conv)
        self.feature_norm = copy.deepcopy(source_ner_model.feature_norm)
        self.start_head = copy.deepcopy(source_ner_model.start_head)
        self.end_head = copy.deepcopy(source_ner_model.end_head)

        # All original Phase 1 weights remain frozen, but the LSTM must
        # remain differentiable with respect to its LoRMA parametrizations.
        for parameter in self.backbone.parameters():
            if "parametrizations." not in getattr(parameter, "_source_name", ""):
                pass

    def train(self, mode=True):
        super().train(mode)
        self.backbone.train(mode)
        return self

    def forward(self, input_ids, char_ids, char_to_token):
        if input_ids.ndim != 2:
            raise ValueError("input_ids must have shape [batch, tokens].")

        if char_ids.shape != char_to_token.shape:
            raise ValueError("Character IDs and token mapping shapes differ.")

        # Unlike the fallback experiment, this call runs the recurrent
        # backbone live so that gradients can reach its LoRMA+ adapters.
        token_hidden = self.backbone(input_ids)

        token_indices = char_to_token.clamp(
            min=0,
            max=input_ids.shape[1] - 1,
        ).long()

        expanded_indices = token_indices.unsqueeze(-1).expand(
            -1,
            -1,
            token_hidden.shape[-1],
        )

        char_context = token_hidden.gather(
            dim=1,
            index=expanded_indices,
        )

        char_valid = char_ids.ne(PAD_CHAR_ID).unsqueeze(-1)

        char_features = self.char_embedding(char_ids)

        combined = torch.cat(
            [char_context, char_features],
            dim=-1,
        )

        features = F.gelu(self.input_projection(combined))
        features = features.masked_fill(~char_valid, 0.0)

        conv_features = self.char_conv(
            features.transpose(1, 2)
        ).transpose(1, 2)

        features = self.feature_norm(
            features + self.dropout(F.gelu(conv_features))
        )

        features = features.masked_fill(~char_valid, 0.0)
        features = self.dropout(features)

        return self.start_head(features), self.end_head(features)


# --------------------------------------------------
# 6. Construct the independent LoRMA+ model
# --------------------------------------------------
S3_LORMA_BACKBONE = S3_LORMA_BACKBONE.to(DEVICE)

S3_LORMA_MODEL = ArchitectureCTrueLoRMAPlusNER(
    backbone=S3_LORMA_BACKBONE,
    source_ner_model=S3_MODEL,
).to(DEVICE)

# Freeze original backbone parameters explicitly.
# Parameters belonging to registered LoRMA parametrization modules
# remain trainable.
for name, parameter in S3_LORMA_MODEL.backbone.named_parameters():
    if ".parametrizations." not in name:
        parameter.requires_grad_(False)

# --------------------------------------------------
# 7. Verify identity initialization and parameter counts
# --------------------------------------------------
S3_LORMA_MODEL.eval()

print("\nIdentity-initialization checks:")

for layer_index in range(
    int(ARCH_C_MODEL_CONFIG["num_layers"])
):
    weight_name = f"weight_hh_l{layer_index}"

    effective_weight = getattr(
        S3_LORMA_MODEL.backbone.lstm,
        weight_name,
    ).detach()

    original_weight = ARCH_C_STATE_DICT[
        f"lstm.{weight_name}"
    ].to(DEVICE)

    maximum_difference = (
        effective_weight.float() - original_weight.float()
    ).abs().max().item()

    print(
        f"Layer {layer_index}: maximum effective-weight difference "
        f"= {maximum_difference:.8f}"
    )

    if not torch.allclose(
        effective_weight,
        original_weight,
        rtol=0.0,
        atol=1e-6,
    ):
        raise RuntimeError(
            f"LoRMA+ initialization changed recurrent weights in layer "
            f"{layer_index}."
        )

trainable_parameters = [
    parameter
    for parameter in S3_LORMA_MODEL.parameters()
    if parameter.requires_grad
]

trainable_parameter_count = sum(
    parameter.numel()
    for parameter in trainable_parameters
)

frozen_parameter_count = sum(
    parameter.numel()
    for parameter in S3_LORMA_MODEL.parameters()
    if not parameter.requires_grad
)

print("\nTrainable parameters:", f"{trainable_parameter_count:,}")
print("Frozen parameters:", f"{frozen_parameter_count:,}")
print("Total parameters:", f"{trainable_parameter_count + frozen_parameter_count:,}")

if trainable_parameter_count == 0:
    raise RuntimeError("The LoRMA+ model has no trainable parameters.")

# --------------------------------------------------
# 8. Run one real-window forward/backward smoke test
# --------------------------------------------------
sample = TRAIN_WINDOWS[0]

test_input_ids = torch.as_tensor(
    sample["input_ids"].astype("int64"),
    dtype=torch.long,
    device=DEVICE,
).unsqueeze(0)

test_char_ids = torch.as_tensor(
    sample["char_ids"].astype("int64"),
    dtype=torch.long,
    device=DEVICE,
).unsqueeze(0)

test_char_to_token = torch.as_tensor(
    sample["char_to_token"].astype("int64"),
    dtype=torch.long,
    device=DEVICE,
).unsqueeze(0)

S3_LORMA_MODEL.eval()

with torch.inference_mode():
    test_start_logits, test_end_logits = S3_LORMA_MODEL(
        test_input_ids,
        test_char_ids,
        test_char_to_token,
    )

expected_shape = (
    1,
    len(sample["char_ids"]),
    NUM_LABELS,
)

if tuple(test_start_logits.shape) != expected_shape:
    raise RuntimeError(
        f"Wrong start-logit shape: {tuple(test_start_logits.shape)}"
    )

if tuple(test_end_logits.shape) != expected_shape:
    raise RuntimeError(
        f"Wrong end-logit shape: {tuple(test_end_logits.shape)}"
    )

if not (
    torch.isfinite(test_start_logits).all().item()
    and torch.isfinite(test_end_logits).all().item()
):
    raise FloatingPointError("Non-finite logits from the LoRMA+ model.")

del test_start_logits, test_end_logits
S3_LORMA_MODEL.train()
S3_LORMA_MODEL.zero_grad(set_to_none=True)

test_start_logits, test_end_logits = S3_LORMA_MODEL(
    test_input_ids,
    test_char_ids,
    test_char_to_token,
)

smoke_loss = (
    test_start_logits.float().mean()
    + test_end_logits.float().mean()
)

smoke_loss.backward()

adapter_gradients_found = False
for module_name, module in S3_LORMA_MODEL.backbone.named_modules():
    if isinstance(module, LoRMAPlusLSTMGateParametrization):
        for parameter in module.parameters():
            if parameter.grad is not None:
                if not torch.isfinite(parameter.grad).all().item():
                    raise FloatingPointError(
                        f"Non-finite LoRMA+ gradient in {module_name}."
                    )
                if parameter.grad.abs().sum().item() > 0:
                    adapter_gradients_found = True

if not adapter_gradients_found:
    raise RuntimeError(
        "LoRMA+ parameters did not receive nonzero gradients."
    )

S3_LORMA_MODEL.zero_grad(set_to_none=True)

print("\nForward-pass shape:", expected_shape)
print("Finite logits: True")
print("Nonzero LoRMA+ adapter gradients: verified")
print("PASS: actual multiplicative weight adaptation is active.")
print("No optimizer update or NER training has occurred.")
print("\nCell 23 completed.")

del test_input_ids, test_char_ids, test_char_to_token
del test_start_logits, test_end_logits, smoke_loss
gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

S3: TRUE LoRMA+ MULTIPLICATIVE LSTM ADAPTATION

Identity-initialization checks:
Layer 0: maximum effective-weight difference = 0.00000000
Layer 1: maximum effective-weight difference = 0.00000000
Layer 2: maximum effective-weight difference = 0.00000000

Trainable parameters: 790,520
Frozen parameters: 103,404,880
Total parameters: 104,195,400

Forward-pass shape: (1, 2566, 12)
Finite logits: True
Nonzero LoRMA+ adapter gradients: verified
PASS: actual multiplicative weight adaptation is active.
No optimizer update or NER training has occurred.

Cell 23 completed.


In [24]:
# Cell 24: Configure the genuine multiplicative-adaptation optimizer and
# validate one mixed-precision training batch without updating model weights.

import gc
import torch

# --------------------------------------------------
# 1. Validate the model and training prerequisites
# --------------------------------------------------
if not isinstance(S3_LORMA_MODEL, torch.nn.Module):
    raise TypeError("S3_LORMA_MODEL has not been initialized.")

if not TRAIN_WINDOWS or not VAL_WINDOWS:
    raise RuntimeError("Training and validation windows are unavailable.")

if any(
    parameter.requires_grad
    for name, parameter in S3_LORMA_MODEL.backbone.named_parameters()
    if ".parametrizations." not in name
):
    raise RuntimeError(
        "Some original Phase 1 backbone parameters are unexpectedly trainable."
    )

if not any(
    parameter.requires_grad
    for name, parameter in S3_LORMA_MODEL.backbone.named_parameters()
    if ".parametrizations." in name
):
    raise RuntimeError("No trainable recurrent adapters were found.")

print("=" * 70)
print("S3: LoRMA+ OPTIMIZER AND TRAINING-BATCH DRY RUN")
print("=" * 70)

# --------------------------------------------------
# 2. Create separate optimizer groups
# --------------------------------------------------
lorma_parameters = [
    parameter
    for name, parameter in S3_LORMA_MODEL.named_parameters()
    if parameter.requires_grad
    and ".parametrizations." in name
]

ner_head_parameters = [
    parameter
    for name, parameter in S3_LORMA_MODEL.named_parameters()
    if parameter.requires_grad
    and ".parametrizations." not in name
]

if not lorma_parameters:
    raise RuntimeError("The LoRMA+ parameter group is empty.")

if not ner_head_parameters:
    raise RuntimeError("The NER-head parameter group is empty.")

S3_LORMA_ADAPTER_LR = 2e-4
S3_LORMA_HEAD_LR = 1e-4
S3_LORMA_WEIGHT_DECAY = 0.01
S3_LORMA_MAX_GRAD_NORM = 1.0

S3_LORMA_OPTIMIZER = torch.optim.AdamW(
    [
        {
            "params": lorma_parameters,
            "lr": S3_LORMA_ADAPTER_LR,
            "name": "multiplicative_adapters",
        },
        {
            "params": ner_head_parameters,
            "lr": S3_LORMA_HEAD_LR,
            "name": "ner_head",
        },
    ],
    weight_decay=S3_LORMA_WEIGHT_DECAY,
)

S3_LORMA_SCALER = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda"),
)

print("Trainable LoRMA+ parameters:", f"{sum(p.numel() for p in lorma_parameters):,}")
print("Trainable NER-head parameters:", f"{sum(p.numel() for p in ner_head_parameters):,}")
print("LoRMA+ learning rate:", S3_LORMA_ADAPTER_LR)
print("NER-head learning rate:", S3_LORMA_HEAD_LR)
print("Weight decay:", S3_LORMA_WEIGHT_DECAY)
print("Gradient clipping limit:", S3_LORMA_MAX_GRAD_NORM)
print("Mixed precision enabled:", DEVICE.type == "cuda")

# --------------------------------------------------
# 3. Fetch one real training batch
# --------------------------------------------------
dry_batch = next(iter(TRAIN_LOADER))

dry_device_batch = {
    key: value.to(DEVICE, non_blocking=True)
    for key, value in dry_batch.items()
    if torch.is_tensor(value)
}

S3_LORMA_MODEL.train()
S3_LORMA_OPTIMIZER.zero_grad(set_to_none=True)

# --------------------------------------------------
# 4. Run forward/backward with mixed precision
# --------------------------------------------------
with torch.autocast(
    device_type=DEVICE.type,
    dtype=torch.float16,
    enabled=(DEVICE.type == "cuda"),
):
    dry_start_logits, dry_end_logits = S3_LORMA_MODEL(
        input_ids=dry_device_batch["input_ids"],
        char_ids=dry_device_batch["char_ids"],
        char_to_token=dry_device_batch["char_to_token"],
    )

    dry_loss, dry_start_loss, dry_end_loss = s3_boundary_loss(
        start_logits=dry_start_logits,
        end_logits=dry_end_logits,
        start_targets=dry_device_batch["start_targets"],
        end_targets=dry_device_batch["end_targets"],
        loss_mask=dry_device_batch["loss_mask"],
    )

if not torch.isfinite(dry_loss).item():
    raise FloatingPointError("The LoRMA+ dry-run loss is non-finite.")

if DEVICE.type == "cuda":
    S3_LORMA_SCALER.scale(dry_loss).backward()
    S3_LORMA_SCALER.unscale_(S3_LORMA_OPTIMIZER)
else:
    dry_loss.backward()

dry_grad_norm = torch.nn.utils.clip_grad_norm_(
    [
        parameter
        for parameter in S3_LORMA_MODEL.parameters()
        if parameter.requires_grad
    ],
    max_norm=S3_LORMA_MAX_GRAD_NORM,
)

if not torch.isfinite(dry_grad_norm).item():
    raise FloatingPointError("The LoRMA+ dry-run gradient norm is non-finite.")

# --------------------------------------------------
# 5. Check gradients on each recurrent adapter and head
# --------------------------------------------------
adapter_gradient_sums = []

for name, module in S3_LORMA_MODEL.backbone.named_modules():
    if isinstance(module, LoRMAPlusLSTMGateParametrization):
        for parameter_name, parameter in module.named_parameters():
            if parameter.grad is None:
                raise RuntimeError(
                    f"Missing gradient for {name}.{parameter_name}."
                )

            if not torch.isfinite(parameter.grad).all().item():
                raise FloatingPointError(
                    f"Non-finite gradient for {name}.{parameter_name}."
                )

            adapter_gradient_sums.append(
                float(parameter.grad.abs().sum().detach().cpu())
            )

if not adapter_gradient_sums or not any(
    value > 0 for value in adapter_gradient_sums
):
    raise RuntimeError("No nonzero recurrent adapter gradients were detected.")

if any(
    parameter.grad is not None
    for name, parameter in S3_LORMA_MODEL.backbone.named_parameters()
    if ".parametrizations." not in name
):
    raise RuntimeError("Original pretrained weights unexpectedly received gradients.")

# Confirm gradients also reach the NER start/end heads.
for head_name in ["start_head", "end_head"]:
    head = getattr(S3_LORMA_MODEL, head_name)

    if not any(
        parameter.grad is not None
        and torch.isfinite(parameter.grad).all().item()
        for parameter in head.parameters()
        if parameter.requires_grad
    ):
        raise RuntimeError(f"No finite gradients detected for {head_name}.")

print("\nDry-run results:")
print("Input tokens:", tuple(dry_device_batch["input_ids"].shape))
print("Character input:", tuple(dry_device_batch["char_ids"].shape))
print("Start logits:", tuple(dry_start_logits.shape))
print("End logits:", tuple(dry_end_logits.shape))
print("Total loss:", float(dry_loss.detach().float().cpu()))
print("Start loss:", float(dry_start_loss.cpu()))
print("End loss:", float(dry_end_loss.cpu()))
print("Gradient norm before clipping:", float(dry_grad_norm))
print("Recurrent adapters with finite gradients:", len(adapter_gradient_sums))
print("Nonzero recurrent adapter gradients:", sum(v > 0 for v in adapter_gradient_sums))
print("Frozen Phase 1 weights have no gradients: True")

# --------------------------------------------------
# 6. Discard smoke-test gradients without updating weights
# --------------------------------------------------
S3_LORMA_OPTIMIZER.zero_grad(set_to_none=True)
S3_LORMA_MODEL.zero_grad(set_to_none=True)

del dry_batch, dry_device_batch
del dry_start_logits, dry_end_logits
del dry_loss, dry_start_loss, dry_end_loss, dry_grad_norm

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print("\nPASS: optimizer and mixed-precision training path verified.")
print("No optimizer step was performed.")
print("Cell 24 completed.")

S3: LoRMA+ OPTIMIZER AND TRAINING-BATCH DRY RUN
Trainable LoRMA+ parameters: 245,760
Trainable NER-head parameters: 544,760
LoRMA+ learning rate: 0.0002
NER-head learning rate: 0.0001
Weight decay: 0.01
Gradient clipping limit: 1.0
Mixed precision enabled: True

Dry-run results:
Input tokens: (2, 512)
Character input: (2, 2574)
Start logits: (2, 2574, 12)
End logits: (2, 2574, 12)
Total loss: 0.0012966536451131105
Start loss: 0.0012573555577546358
End loss: 0.0013359517324715853
Gradient norm before clipping: 0.012874840758740902
Recurrent adapters with finite gradients: 6
Nonzero recurrent adapter gradients: 3
Frozen Phase 1 weights have no gradients: True

PASS: optimizer and mixed-precision training path verified.
No optimizer step was performed.
Cell 24 completed.


In [25]:
# Cell 25: Implement document-level validation inference for the actual
# LoRMA+ LSTM model and verify its predictions on one validation batch.

import gc
import numpy as np
import torch

# --------------------------------------------------
# 1. Define validation inference for the live LoRMA+ model
# --------------------------------------------------
def predict_document_probabilities_lorma(windows, loader, model):
    """
    Run the live LoRMA+ LSTM on all windows, then average start/end
    probabilities over overlapping windows to recover document scores.
    """

    if len(windows) == 0:
        raise ValueError("Cannot infer from an empty window list.")

    model.eval()

    probability_sums = {}
    overlap_counts = {}
    window_cursor = 0

    with torch.inference_mode():
        for batch in loader:
            device_batch = {
                key: value.to(DEVICE, non_blocking=True)
                for key, value in batch.items()
                if torch.is_tensor(value)
            }

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda"),
            ):
                start_logits, end_logits = model(
                    input_ids=device_batch["input_ids"],
                    char_ids=device_batch["char_ids"],
                    char_to_token=device_batch["char_to_token"],
                )

            start_probabilities = torch.sigmoid(
                start_logits.float()
            ).cpu().numpy()

            end_probabilities = torch.sigmoid(
                end_logits.float()
            ).cpu().numpy()

            batch_size = len(batch["doc_ids"])

            for i in range(batch_size):
                if window_cursor >= len(windows):
                    raise RuntimeError(
                        "Validation loader returned extra windows."
                    )

                window = windows[window_cursor]
                window_cursor += 1

                if batch["doc_ids"][i] != window["doc_id"]:
                    raise RuntimeError(
                        "Validation loader and window ordering differ. "
                        "Ensure the loader uses shuffle=False."
                    )

                doc_id = window["doc_id"]
                char_start = window["char_start"]
                char_end = window["char_end"]
                window_char_length = len(window["char_ids"])
                document_length = len(ALL_TEXTS[doc_id])

                if char_end - char_start != window_char_length:
                    raise RuntimeError(
                        f"Invalid character bounds in window for {doc_id}."
                    )

                if doc_id not in probability_sums:
                    probability_sums[doc_id] = {
                        "start": np.zeros(
                            (document_length, NUM_LABELS),
                            dtype=np.float32,
                        ),
                        "end": np.zeros(
                            (document_length, NUM_LABELS),
                            dtype=np.float32,
                        ),
                    }
                    overlap_counts[doc_id] = np.zeros(
                        document_length,
                        dtype=np.int32,
                    )

                probability_sums[doc_id]["start"][
                    char_start:char_end
                ] += start_probabilities[i, :window_char_length]

                probability_sums[doc_id]["end"][
                    char_start:char_end
                ] += end_probabilities[i, :window_char_length]

                overlap_counts[doc_id][char_start:char_end] += 1

            del device_batch, start_logits, end_logits
            del start_probabilities, end_probabilities

    if window_cursor != len(windows):
        raise RuntimeError(
            f"Processed {window_cursor} of {len(windows)} windows; "
            f"expected {len(windows)}."
        )

    document_probabilities = {}

    for doc_id, sums in probability_sums.items():
        counts = overlap_counts[doc_id]

        if np.any(counts == 0):
            missing = np.flatnonzero(counts == 0)[:10]
            raise RuntimeError(
                f"Characters without predictions in {doc_id}: "
                f"{missing.tolist()}"
            )

        document_probabilities[doc_id] = {
            "start": sums["start"] / counts[:, None],
            "end": sums["end"] / counts[:, None],
        }

    return document_probabilities


# --------------------------------------------------
# 2. Check prerequisites
# --------------------------------------------------
if "S3_LORMA_MODEL" not in globals():
    raise RuntimeError("S3_LORMA_MODEL is not defined.")

if "VAL_LOADER" not in globals():
    raise RuntimeError("VAL_LOADER is not defined.")

print("=" * 65)
print("S3: TRUE LoRMA+ VALIDATION INFERENCE CHECK")
print("=" * 65)

# --------------------------------------------------
# 3. Smoke-test one real validation batch
# --------------------------------------------------
S3_LORMA_MODEL.eval()

validation_batch = next(iter(VAL_LOADER))

device_batch = {
    key: value.to(DEVICE, non_blocking=True)
    for key, value in validation_batch.items()
    if torch.is_tensor(value)
}

with torch.inference_mode():
    with torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.float16,
        enabled=(DEVICE.type == "cuda"),
    ):
        start_logits, end_logits = S3_LORMA_MODEL(
            input_ids=device_batch["input_ids"],
            char_ids=device_batch["char_ids"],
            char_to_token=device_batch["char_to_token"],
        )

if start_logits.shape != (
    device_batch["char_ids"].shape[0],
    device_batch["char_ids"].shape[1],
    NUM_LABELS,
):
    raise RuntimeError("LoRMA+ validation start-logit shape mismatch.")

if end_logits.shape != start_logits.shape:
    raise RuntimeError("LoRMA+ validation end-logit shape mismatch.")

if not (
    torch.isfinite(start_logits).all().item()
    and torch.isfinite(end_logits).all().item()
):
    raise FloatingPointError("Non-finite validation logits.")

print("Validation batch documents:", validation_batch["doc_ids"])
print("Token batch shape:", tuple(device_batch["input_ids"].shape))
print("Character batch shape:", tuple(device_batch["char_ids"].shape))
print("Start-logit shape:", tuple(start_logits.shape))
print("End-logit shape:", tuple(end_logits.shape))
print("Finite logits: True")

del validation_batch, device_batch, start_logits, end_logits
gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print("\nPASS: true LoRMA+ validation forward path verified.")
print("No optimizer update or training occurred.")
print("Cell 25 completed.")

S3: TRUE LoRMA+ VALIDATION INFERENCE CHECK
Validation batch documents: ['1011232', '1011232']
Token batch shape: (2, 512)
Character batch shape: (2, 2634)
Start-logit shape: (2, 2634, 12)
End-logit shape: (2, 2634, 12)
Finite logits: True

PASS: true LoRMA+ validation forward path verified.
No optimizer update or training occurred.
Cell 25 completed.


In [26]:
# Cell 26: Train the actual multiplicative LoRMA+ adapters and character
# NER head, evaluate document-level strict span macro-F1, and save the best
# validation checkpoint independently of the additive-adapter fallback.

import gc
import json
import time
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm

# --------------------------------------------------
# 1. Validate prerequisites and preserve the fallback
# --------------------------------------------------
if not S3_FALLBACK_PATH.is_file():
    raise FileNotFoundError("The previously saved S3 fallback is missing.")

if not isinstance(S3_LORMA_MODEL, torch.nn.Module):
    raise RuntimeError("The true LoRMA+ model is unavailable.")

if any(
    parameter.requires_grad
    for name, parameter in S3_LORMA_MODEL.backbone.named_parameters()
    if ".parametrizations." not in name
):
    raise RuntimeError("Original Phase 1 weights must remain frozen.")

if not TRAIN_LOADER or not VAL_LOADER:
    raise RuntimeError("Training or validation DataLoaders are unavailable.")

# --------------------------------------------------
# 2. Set the training budget and output paths
# --------------------------------------------------
S3_LORMA_MAX_EPOCHS = 3
S3_LORMA_PATIENCE = 2

S3_LORMA_VALIDATION_THRESHOLD_GRID = [
    0.002, 0.005, 0.010, 0.020,
    0.030, 0.050, 0.080, 0.120,
    0.200, 0.300, 0.400, 0.500,
]

S3_LORMA_OUTPUT_DIR = Path("/kaggle/working/S3_Architecture_C")
S3_LORMA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

S3_LORMA_BEST_PATH = (
    S3_LORMA_OUTPUT_DIR / "S3_LoRMA_multiplicative_best.pt"
)
S3_LORMA_HISTORY_PATH = (
    S3_LORMA_OUTPUT_DIR / "S3_LoRMA_multiplicative_history.json"
)

S3_LORMA_TRAINABLE_NAMES = {
    name
    for name, parameter in S3_LORMA_MODEL.named_parameters()
    if parameter.requires_grad
}

if not S3_LORMA_TRAINABLE_NAMES:
    raise RuntimeError("No trainable LoRMA+ parameters were found.")

S3_LORMA_HISTORY = []
S3_LORMA_BEST_F1 = -1.0
S3_LORMA_BEST_EPOCH = None
S3_LORMA_BEST_THRESHOLDS = None
S3_LORMA_BAD_EPOCHS = 0
# Reset GradScaler after the dry run, which intentionally did not update it.
S3_LORMA_SCALER = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda"),
)

S3_LORMA_OPTIMIZER.zero_grad(set_to_none=True)
S3_LORMA_MODEL.zero_grad(set_to_none=True)
S3_LORMA_SCHEDULER = torch.optim.lr_scheduler.ReduceLROnPlateau(
    S3_LORMA_OPTIMIZER,
    mode="max",
    factor=0.5,
    patience=1,
    threshold=1e-4,
    min_lr=1e-5,
)

print("=" * 70)
print("S3: TRUE MULTIPLICATIVE LoRMA+ TRAINING")
print("=" * 70)
print("Device:", DEVICE)
print("Training windows:", len(TRAIN_WINDOWS))
print("Validation windows:", len(VAL_WINDOWS))
print("Maximum epochs:", S3_LORMA_MAX_EPOCHS)
print("Trainable parameters:", sum(
    p.numel() for p in S3_LORMA_MODEL.parameters() if p.requires_grad
))
print("Fallback preserved:", S3_FALLBACK_PATH)
print("Best checkpoint destination:", S3_LORMA_BEST_PATH)

# --------------------------------------------------
# 3. Train one epoch with the live recurrent backbone
# --------------------------------------------------
def train_s3_lorma_epoch():
    S3_LORMA_MODEL.train()

    loss_sum = 0.0
    start_loss_sum = 0.0
    end_loss_sum = 0.0
    batches_seen = 0

    progress = tqdm(
        TRAIN_LOADER,
        desc="S3 multiplicative LoRMA+",
        leave=False,
    )

    for batch in progress:
        device_batch = {
            key: value.to(DEVICE, non_blocking=True)
            for key, value in batch.items()
            if torch.is_tensor(value)
        }

        S3_LORMA_OPTIMIZER.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda"),
        ):
            start_logits, end_logits = S3_LORMA_MODEL(
                input_ids=device_batch["input_ids"],
                char_ids=device_batch["char_ids"],
                char_to_token=device_batch["char_to_token"],
            )

            loss, start_loss, end_loss = s3_boundary_loss(
                start_logits=start_logits,
                end_logits=end_logits,
                start_targets=device_batch["start_targets"],
                end_targets=device_batch["end_targets"],
                loss_mask=device_batch["loss_mask"],
            )

        if not torch.isfinite(loss).item():
            raise FloatingPointError(
                f"Non-finite LoRMA+ loss in batch {batches_seen}."
            )

        if DEVICE.type == "cuda":
            S3_LORMA_SCALER.scale(loss).backward()
            S3_LORMA_SCALER.unscale_(S3_LORMA_OPTIMIZER)
        else:
            loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(
            [
                parameter
                for parameter in S3_LORMA_MODEL.parameters()
                if parameter.requires_grad
            ],
            max_norm=S3_LORMA_MAX_GRAD_NORM,
        )

        if not torch.isfinite(grad_norm).item():
            raise FloatingPointError(
                f"Non-finite gradient norm in batch {batches_seen}."
            )

        if DEVICE.type == "cuda":
            S3_LORMA_SCALER.step(S3_LORMA_OPTIMIZER)
            S3_LORMA_SCALER.update()
        else:
            S3_LORMA_OPTIMIZER.step()

        loss_sum += float(loss.detach().float().cpu())
        start_loss_sum += float(start_loss.cpu())
        end_loss_sum += float(end_loss.cpu())
        batches_seen += 1

        progress.set_postfix({
            "loss": f"{loss_sum / batches_seen:.5f}",
            "lr_adapter": f"{S3_LORMA_OPTIMIZER.param_groups[0]['lr']:.1e}",
        })

        del device_batch, start_logits, end_logits
        del loss, start_loss, end_loss, grad_norm

    if batches_seen == 0:
        raise RuntimeError("No training batches were processed.")

    return {
        "train_loss": loss_sum / batches_seen,
        "train_start_loss": start_loss_sum / batches_seen,
        "train_end_loss": end_loss_sum / batches_seen,
        "training_batches": batches_seen,
    }


# --------------------------------------------------
# 4. Evaluate validation data and tune thresholds
# --------------------------------------------------
def evaluate_s3_lorma_validation():
    probabilities = predict_document_probabilities_lorma(
        windows=VAL_WINDOWS,
        loader=VAL_LOADER,
        model=S3_LORMA_MODEL,
    )

    if set(probabilities) != set(VAL_DOC_IDS):
        raise RuntimeError("Validation prediction document IDs do not match.")

    thresholds = []
    threshold_results = {}

    for label_id, label in enumerate(LABELS):
        result = tune_one_label_threshold(
            document_probabilities=probabilities,
            gold_spans=VAL_GOLD_SPANS,
            label_id=label_id,
            threshold_grid=S3_LORMA_VALIDATION_THRESHOLD_GRID,
        )

        thresholds.append(float(result["threshold"]))
        threshold_results[label] = result

    metrics, predicted_spans = evaluate_document_probabilities(
        document_probabilities=probabilities,
        gold_spans=VAL_GOLD_SPANS,
        label_thresholds=thresholds,
    )

    return probabilities, thresholds, threshold_results, metrics, predicted_spans


# --------------------------------------------------
# 5. Train, evaluate, and save the best checkpoint
# --------------------------------------------------
for epoch in range(1, S3_LORMA_MAX_EPOCHS + 1):
    epoch_start = time.time()

    print("\n" + "=" * 70)
    print(f"TRUE LoRMA+ EPOCH {epoch}/{S3_LORMA_MAX_EPOCHS}")
    print("=" * 70)

    training_metrics = train_s3_lorma_epoch()

    (
        VAL_LORMA_PROBABILITIES,
        thresholds,
        threshold_results,
        validation_metrics,
        validation_predictions,
    ) = evaluate_s3_lorma_validation()

    macro_f1 = float(validation_metrics["macro_f1"])

    if not np.isfinite(macro_f1):
        raise FloatingPointError("Validation Strict Span Macro-F1 is non-finite.")

    S3_LORMA_SCHEDULER.step(macro_f1)

    improved = macro_f1 > S3_LORMA_BEST_F1 + 1e-8
    elapsed = time.time() - epoch_start

    record = {
        "epoch": epoch,
        **training_metrics,
        "validation_strict_span_macro_f1": macro_f1,
        "validation_gold_spans": validation_metrics["gold_count"],
        "validation_predicted_spans": validation_metrics["predicted_count"],
        "validation_exact_true_positives": validation_metrics[
            "exact_true_positives"
        ],
        "validation_per_label": validation_metrics["per_label"],
        "validation_thresholds": {
            label: thresholds[i]
            for i, label in enumerate(LABELS)
        },
        "threshold_search_results": threshold_results,
        "duration_seconds": elapsed,
        "learning_rates_after_scheduler": {
            group.get("name", f"group_{i}"): float(group["lr"])
            for i, group in enumerate(S3_LORMA_OPTIMIZER.param_groups)
        },
    }

    S3_LORMA_HISTORY.append(record)

    print("\nTraining:")
    print(f"  Loss: {training_metrics['train_loss']:.6f}")
    print(f"  Start loss: {training_metrics['train_start_loss']:.6f}")
    print(f"  End loss: {training_metrics['train_end_loss']:.6f}")
    print(f"  Batches: {training_metrics['training_batches']}")
    print(f"  Epoch duration: {elapsed:.1f} seconds")

    print("\nValidation:")
    print(f"  Strict Span Macro-F1: {macro_f1:.6f}")
    print(f"  Gold spans: {validation_metrics['gold_count']}")
    print(f"  Predicted spans: {validation_metrics['predicted_count']}")
    print(f"  Exact true positives: {validation_metrics['exact_true_positives']}")

    print("\nPer-label results:")
    print(
        f"{'Label':12s} {'P':>8s} {'R':>8s} {'F1':>8s} "
        f"{'TP':>6s} {'FP':>6s} {'FN':>6s} {'Thr':>7s}"
    )

    for label_id, label in enumerate(LABELS):
        result = validation_metrics["per_label"][label]
        print(
            f"{label:12s} "
            f"{result['precision']:8.4f} "
            f"{result['recall']:8.4f} "
            f"{result['f1']:8.4f} "
            f"{result['tp']:6d} "
            f"{result['fp']:6d} "
            f"{result['fn']:6d} "
            f"{thresholds[label_id]:7.3f}"
        )

    if improved:
        S3_LORMA_BEST_F1 = macro_f1
        S3_LORMA_BEST_EPOCH = epoch
        S3_LORMA_BEST_THRESHOLDS = list(thresholds)
        S3_LORMA_BAD_EPOCHS = 0

        full_state = S3_LORMA_MODEL.state_dict()

        trainable_state = {
            name: full_state[name].detach().cpu().clone()
            for name in S3_LORMA_TRAINABLE_NAMES
        }

        checkpoint = {
            "experiment_id": "CS781_Phase2_S3_Architecture_C_True_LoRMA",
            "adaptation_method": (
                "Low-rank multiplicative weight adaptation "
                "on the recurrent LSTM gate matrices"
            ),
            "architecture_name": ARCH_C_METADATA["architecture_name"],
            "phase1_checkpoint": str(ARCH_C_FINAL_CHECKPOINT),
            "phase1_tokenizer": str(PHASE1_TOKENIZER_PATH),
            "epoch": epoch,
            "validation_strict_span_macro_f1": macro_f1,
            "validation_metrics": validation_metrics,
            "thresholds": {
                label: float(thresholds[i])
                for i, label in enumerate(LABELS)
            },
            "labels": LABELS,
            "label_to_id": LABEL_TO_ID,
            "char_to_id": CHAR_TO_ID,
            "char_vocab_size": CHAR_VOCAB_SIZE,
            "model_config": {
                "lorma_rank": S3_LORMA_RANK,
                "lorma_alpha": S3_LORMA_ALPHA,
                "char_embed_dim": CHAR_EMBED_DIM,
                "ner_hidden_dim": NER_HIDDEN_DIM,
                "dropout": NER_DROPOUT,
                "max_span_chars": MAX_SPAN_CHARS,
                "max_tokens": MAX_TOKENS,
                "window_stride": WINDOW_STRIDE,
            },
            "trainable_parameter_names": sorted(
                S3_LORMA_TRAINABLE_NAMES
            ),
            "trainable_state_dict": trainable_state,
        }

        torch.save(checkpoint, S3_LORMA_BEST_PATH)

        print("\nNEW BEST TRUE LoRMA+ CHECKPOINT SAVED:")
        print(" ", S3_LORMA_BEST_PATH)
        print(f"  Best validation Macro-F1: {S3_LORMA_BEST_F1:.6f}")

    else:
        S3_LORMA_BAD_EPOCHS += 1
        print(
            f"\nNo improvement: "
            f"{S3_LORMA_BAD_EPOCHS}/{S3_LORMA_PATIENCE}"
        )

    # Persist the training history even if the next epoch is interrupted.
    with S3_LORMA_HISTORY_PATH.open("w", encoding="utf-8") as f:
        json.dump(
            S3_LORMA_HISTORY,
            f,
            ensure_ascii=False,
            indent=2,
        )

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    gc.collect()

    if S3_LORMA_BAD_EPOCHS >= S3_LORMA_PATIENCE:
        print("\nEarly stopping triggered.")
        break


# --------------------------------------------------
# 6. Restore the best trainable weights from disk
# --------------------------------------------------
if not S3_LORMA_BEST_PATH.is_file():
    raise FileNotFoundError(
        "No true LoRMA+ checkpoint was saved. Inspect the training output."
    )

best_checkpoint = torch.load(
    S3_LORMA_BEST_PATH,
    map_location="cpu",
    weights_only=False,
)

expected_trainable_names = set(
    best_checkpoint["trainable_parameter_names"]
)

if expected_trainable_names != S3_LORMA_TRAINABLE_NAMES:
    raise RuntimeError(
        "Saved trainable parameter names differ from the current model."
    )

load_result = S3_LORMA_MODEL.load_state_dict(
    best_checkpoint["trainable_state_dict"],
    strict=False,
)

expected_missing = (
    set(S3_LORMA_MODEL.state_dict().keys())
    - S3_LORMA_TRAINABLE_NAMES
)

if set(load_result.missing_keys) != expected_missing:
    raise RuntimeError(
        "Unexpected missing keys while restoring the best LoRMA+ checkpoint."
    )

if load_result.unexpected_keys:
    raise RuntimeError(
        f"Unexpected keys when restoring checkpoint: {load_result.unexpected_keys}"
    )

S3_LORMA_MODEL.eval()

S3_LORMA_BEST_THRESHOLDS = [
    float(best_checkpoint["thresholds"][label])
    for label in LABELS
]

print("\n" + "=" * 70)
print("TRUE LoRMA+ TRAINING COMPLETE")
print("=" * 70)
print("Best epoch:", best_checkpoint["epoch"])
print(
    "Best validation Strict Span Macro-F1:",
    f"{best_checkpoint['validation_strict_span_macro_f1']:.6f}",
)
print("Checkpoint:", S3_LORMA_BEST_PATH)
print("History:", S3_LORMA_HISTORY_PATH)
print("Fallback remains preserved:", S3_FALLBACK_PATH)
print("Restored best trainable weights: True")
print("Cell 26 completed.")

S3: TRUE MULTIPLICATIVE LoRMA+ TRAINING
Device: cuda
Training windows: 903
Validation windows: 293
Maximum epochs: 3
Trainable parameters: 790520
Fallback preserved: /kaggle/working/S3_Architecture_C/S3_additive_adapter_fallback.pt
Best checkpoint destination: /kaggle/working/S3_Architecture_C/S3_LoRMA_multiplicative_best.pt

TRUE LoRMA+ EPOCH 1/3


S3 multiplicative LoRMA+:   0%|          | 0/452 [00:00<?, ?it/s]


Training:
  Loss: 0.001689
  Start loss: 0.001647
  End loss: 0.001731
  Batches: 452
  Epoch duration: 130.2 seconds

Validation:
  Strict Span Macro-F1: 0.365263
  Gold spans: 1861
  Predicted spans: 2790
  Exact true positives: 863

Per-label results:
Label               P        R       F1     TP     FP     FN     Thr
APP            0.0909   0.2268   0.1298     22    220     75   0.400
RESP           0.1605   0.1912   0.1745     13     68     55   0.500
A.COUNSEL      0.2947   0.6512   0.4058     28     67     15   0.400
R.COUNSEL      0.4494   0.6780   0.5405     40     49     19   0.500
JUDGE          0.2647   0.5870   0.3649     27     75     19   0.500
WIT            0.0500   0.2424   0.0829      8    152     25   0.120
AUTH           0.1132   0.3377   0.1696     77    603    151   0.120
COURT          0.5498   0.6107   0.5786    149    122     95   0.500
STAT           0.6126   0.6066   0.6096    185    117    120   0.300
PREC           0.1862   0.2399   0.2097     65    284 

S3 multiplicative LoRMA+:   0%|          | 0/452 [00:00<?, ?it/s]


Training:
  Loss: 0.001430
  Start loss: 0.001406
  End loss: 0.001453
  Batches: 452
  Epoch duration: 129.5 seconds

Validation:
  Strict Span Macro-F1: 0.391665
  Gold spans: 1861
  Predicted spans: 2380
  Exact true positives: 862

Per-label results:
Label               P        R       F1     TP     FP     FN     Thr
APP            0.1610   0.1959   0.1767     19     99     78   0.500
RESP           0.1910   0.2500   0.2166     17     72     51   0.400
A.COUNSEL      0.3483   0.7209   0.4697     31     58     12   0.500
R.COUNSEL      0.4333   0.6610   0.5235     39     51     20   0.500
JUDGE          0.4262   0.5652   0.4860     26     35     20   0.500
WIT            0.0667   0.1212   0.0860      4     56     29   0.400
AUTH           0.1260   0.2851   0.1747     65    451    163   0.120
COURT          0.5683   0.6311   0.5981    154    117     90   0.500
STAT           0.5987   0.5869   0.5927    179    120    126   0.300
PREC           0.2084   0.2915   0.2431     79    300 

S3 multiplicative LoRMA+:   0%|          | 0/452 [00:00<?, ?it/s]


Training:
  Loss: 0.001240
  Start loss: 0.001218
  End loss: 0.001262
  Batches: 452
  Epoch duration: 129.5 seconds

Validation:
  Strict Span Macro-F1: 0.381844
  Gold spans: 1861
  Predicted spans: 2356
  Exact true positives: 859

Per-label results:
Label               P        R       F1     TP     FP     FN     Thr
APP            0.1565   0.1856   0.1698     18     97     79   0.500
RESP           0.1846   0.1765   0.1805     12     53     56   0.500
A.COUNSEL      0.2647   0.6279   0.3724     27     75     16   0.400
R.COUNSEL      0.4255   0.6780   0.5229     40     54     19   0.500
JUDGE          0.4242   0.6087   0.5000     28     38     18   0.500
WIT            0.0606   0.1212   0.0808      4     62     29   0.400
AUTH           0.1153   0.2412   0.1560     55    422    173   0.120
COURT          0.5836   0.6434   0.6121    157    112     87   0.500
STAT           0.6644   0.6361   0.6499    194     98    111   0.300
PREC           0.1945   0.2878   0.2321     78    323 

In [27]:
# Cell 27: Run Architecture C true LoRMA+ inference on Fold 3, aggregate
# character probabilities across windows, decode strict spans, and audit output.

import gc
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# --------------------------------------------------
# 1. Validate the best checkpoint and thresholds
# --------------------------------------------------
S3_LORMA_BEST_PATH = (
    Path("/kaggle/working/S3_Architecture_C")
    / "S3_LoRMA_multiplicative_best.pt"
)

if not S3_LORMA_BEST_PATH.is_file():
    raise FileNotFoundError(
        f"Best true LoRMA+ checkpoint not found: {S3_LORMA_BEST_PATH}"
    )

if S3_LORMA_BEST_THRESHOLDS is None:
    raise RuntimeError("Validation-selected thresholds are unavailable.")

if len(S3_LORMA_BEST_THRESHOLDS) != NUM_LABELS:
    raise ValueError("Saved threshold count does not match entity labels.")

if set(fold3_text) != set(TEST_DOC_IDS) if "TEST_DOC_IDS" in globals() else False:
    raise RuntimeError("Unexpected mismatch in test document IDs.")

# Restore best trained trainable parameters.
best_checkpoint = torch.load(
    S3_LORMA_BEST_PATH,
    map_location="cpu",
    weights_only=False,
)

if not np.isclose(
    best_checkpoint["validation_strict_span_macro_f1"],
    S3_LORMA_BEST_F1,
):
    raise RuntimeError("Best checkpoint metric does not match training state.")

load_result = S3_LORMA_MODEL.load_state_dict(
    best_checkpoint["trainable_state_dict"],
    strict=False,
)

expected_missing = (
    set(S3_LORMA_MODEL.state_dict().keys())
    - S3_LORMA_TRAINABLE_NAMES
)

if set(load_result.missing_keys) != expected_missing:
    raise RuntimeError("Unexpected missing keys when loading the best checkpoint.")

if load_result.unexpected_keys:
    raise RuntimeError("Unexpected keys when loading the best checkpoint.")

S3_LORMA_BEST_THRESHOLDS = [
    float(best_checkpoint["thresholds"][label])
    for label in LABELS
]

# --------------------------------------------------
# 2. Build the Fold 3 inference windows
# --------------------------------------------------
TEST_DOC_IDS = sorted(fold3_text.keys())
TEST_WINDOWS = []
S3_EMPTY_TEST_DOC_IDS = []

print("=" * 65)
print("S3: FOLD 3 INFERENCE")
print("=" * 65)
print("Test documents:", len(TEST_DOC_IDS))
print("Best validation epoch:", best_checkpoint["epoch"])
print(
    "Best validation Strict Span Macro-F1:",
    f"{best_checkpoint['validation_strict_span_macro_f1']:.6f}",
)

for doc_id in TEST_DOC_IDS:
    text_value = fold3_text[doc_id]

    encoding = tokenizer.encode(
        text_value,
        add_special_tokens=False,
    )

    if len(encoding.ids) != len(encoding.offsets):
        raise RuntimeError(
            f"Token and offset count mismatch for test document {doc_id}."
        )

    token_ids = encoding.ids
    offsets = encoding.offsets

    if not token_ids:
        S3_EMPTY_TEST_DOC_IDS.append(doc_id)
        continue

    window_starts = get_window_starts(
        len(token_ids),
        MAX_TOKENS,
        WINDOW_STRIDE,
    )

    for token_start in window_starts:
        token_end = min(token_start + MAX_TOKENS, len(token_ids))

        window = build_ner_window(
            doc_id=doc_id,
            text_value=text_value,
            annotations=[],
            token_ids=token_ids,
            offsets=offsets,
            token_start=token_start,
            token_end=token_end,
        )

        if window is None:
            continue

        TEST_WINDOWS.append(window)

if not TEST_WINDOWS:
    raise RuntimeError("No test windows were created.")

TEST_DATASET = S3NERWindowDataset(TEST_WINDOWS, "Test")

TEST_LOADER = DataLoader(
    TEST_DATASET,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    collate_fn=collate_s3_windows,
    drop_last=False,
)

print("\nTest windows created:", len(TEST_WINDOWS))
print("Empty test documents:", len(S3_EMPTY_TEST_DOC_IDS))
print("Empty test document IDs:", S3_EMPTY_TEST_DOC_IDS)

# --------------------------------------------------
# 3. Aggregate predictions across all windows
# --------------------------------------------------
def predict_lorma_document_probabilities(
    windows,
    loader,
    model,
    text_mapping,
):
    model.eval()

    probability_sums = {}
    overlap_counts = {}
    cursor = 0

    with torch.inference_mode():
        for batch in tqdm(loader, desc="Fold 3 inference"):
            device_batch = {
                key: value.to(DEVICE, non_blocking=True)
                for key, value in batch.items()
                if torch.is_tensor(value)
            }

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda"),
            ):
                start_logits, end_logits = model(
                    input_ids=device_batch["input_ids"],
                    char_ids=device_batch["char_ids"],
                    char_to_token=device_batch["char_to_token"],
                )

            start_probabilities = torch.sigmoid(
                start_logits.float()
            ).cpu().numpy()

            end_probabilities = torch.sigmoid(
                end_logits.float()
            ).cpu().numpy()

            for i, doc_id in enumerate(batch["doc_ids"]):
                if cursor >= len(windows):
                    raise RuntimeError("More test batch examples than windows.")

                window = windows[cursor]
                cursor += 1

                if doc_id != window["doc_id"]:
                    raise RuntimeError("Test loader order mismatch.")

                text_value = text_mapping[doc_id]
                doc_length = len(text_value)
                char_start = window["char_start"]
                char_end = window["char_end"]
                window_char_length = len(window["char_ids"])

                if char_end - char_start != window_char_length:
                    raise RuntimeError(
                        f"Character-span mismatch in document {doc_id}."
                    )

                if doc_id not in probability_sums:
                    probability_sums[doc_id] = {
                        "start": np.zeros(
                            (doc_length, NUM_LABELS),
                            dtype=np.float32,
                        ),
                        "end": np.zeros(
                            (doc_length, NUM_LABELS),
                            dtype=np.float32,
                        ),
                    }

                    overlap_counts[doc_id] = np.zeros(
                        doc_length,
                        dtype=np.int32,
                    )

                probability_sums[doc_id]["start"][
                    char_start:char_end
                ] += start_probabilities[i, :window_char_length]

                probability_sums[doc_id]["end"][
                    char_start:char_end
                ] += end_probabilities[i, :window_char_length]

                overlap_counts[doc_id][char_start:char_end] += 1

            del device_batch, start_logits, end_logits
            del start_probabilities, end_probabilities

    if cursor != len(windows):
        raise RuntimeError(
            f"Processed {cursor} of {len(windows)} test windows."
        )

    result = {}

    for doc_id, sums in probability_sums.items():
        counts = overlap_counts[doc_id]

        if np.any(counts == 0):
            raise RuntimeError(
                f"Characters without predictions in test document {doc_id}."
            )

        result[doc_id] = {
            "start": sums["start"] / counts[:, None],
            "end": sums["end"] / counts[:, None],
        }

    return result


S3_TEST_PROBABILITIES = predict_lorma_document_probabilities(
    windows=TEST_WINDOWS,
    loader=TEST_LOADER,
    model=S3_LORMA_MODEL,
    text_mapping=fold3_text,
)

expected_nonempty_ids = set(TEST_DOC_IDS) - set(S3_EMPTY_TEST_DOC_IDS)

if set(S3_TEST_PROBABILITIES) != expected_nonempty_ids:
    raise RuntimeError(
        "Test probability document IDs do not match the nonempty test documents."
    )

# --------------------------------------------------
# 4. Decode all test documents using validation thresholds
# --------------------------------------------------
S3_FOLD3_PREDICTIONS = {}

for doc_id in TEST_DOC_IDS:
    if doc_id in S3_EMPTY_TEST_DOC_IDS:
        S3_FOLD3_PREDICTIONS[doc_id] = []
        continue

    probabilities = S3_TEST_PROBABILITIES[doc_id]

    spans = decode_document_fast(
        start_scores=probabilities["start"],
        end_scores=probabilities["end"],
        label_thresholds=S3_LORMA_BEST_THRESHOLDS,
    )

    text_value = fold3_text[doc_id]
    verified_spans = []

    for start, end, label in spans:
        if not isinstance(start, int) or not isinstance(end, int):
            raise TypeError(f"Non-integer offsets in {doc_id}: {(start, end)}")

        if not (0 <= start < end <= len(text_value)):
            raise ValueError(
                f"Invalid span {(start, end, label)} in document {doc_id}."
            )

        if label not in LABEL_TO_ID:
            raise ValueError(f"Invalid entity label {label!r}.")

        verified_spans.append([start, end, label])

    S3_FOLD3_PREDICTIONS[doc_id] = verified_spans

# --------------------------------------------------
# 5. Report prediction counts and verify the output structure
# --------------------------------------------------
if set(S3_FOLD3_PREDICTIONS) != set(fold3_text):
    raise RuntimeError("Final prediction IDs do not match Fold 3 IDs.")

label_prediction_counts = {label: 0 for label in LABELS}
total_predictions = 0

for doc_id, spans in S3_FOLD3_PREDICTIONS.items():
    if not isinstance(spans, list):
        raise TypeError(f"Predictions for document {doc_id} must be a list.")

    total_predictions += len(spans)

    for span in spans:
        if not isinstance(span, list) or len(span) != 3:
            raise ValueError(
                f"Expected [start, end, label], got {span!r}."
            )

        label_prediction_counts[span[2]] += 1

print("\n" + "=" * 65)
print("S3: FOLD 3 PREDICTION AUDIT")
print("=" * 65)
print("Documents in test set:", len(fold3_text))
print("Documents in prediction dictionary:", len(S3_FOLD3_PREDICTIONS))
print("Total predicted spans:", total_predictions)
print("Empty documents retained:", len(S3_EMPTY_TEST_DOC_IDS))
print("\nPredictions per label:")

for label in LABELS:
    print(f"  {label:12s}: {label_prediction_counts[label]}")

print("\nFirst three documents:")
for doc_id in TEST_DOC_IDS[:3]:
    print(doc_id, ":", S3_FOLD3_PREDICTIONS[doc_id][:5])

print("\nPASS: all Fold 3 document IDs have valid prediction lists.")
print("Cell 27 completed. No test gold labels were used.")

S3: FOLD 3 INFERENCE
Test documents: 35
Best validation epoch: 2
Best validation Strict Span Macro-F1: 0.391665

Test windows created: 637
Empty test documents: 1
Empty test document IDs: ['1305957']


Fold 3 inference:   0%|          | 0/319 [00:00<?, ?it/s]


S3: FOLD 3 PREDICTION AUDIT
Documents in test set: 35
Documents in prediction dictionary: 35
Total predicted spans: 5013
Empty documents retained: 1

Predictions per label:
  APP         : 220
  RESP        : 214
  A.COUNSEL   : 189
  R.COUNSEL   : 181
  JUDGE       : 146
  WIT         : 168
  AUTH        : 1000
  COURT       : 547
  STAT        : 757
  PREC        : 687
  DATE        : 622
  CASENO      : 282

First three documents:
10104667 : [[20, 42, 'COURT'], [72, 404, 'CASENO'], [102, 137, 'APP'], [107, 137, 'RESP'], [173, 187, 'RESP']]
101482639 : [[88, 133, 'RESP'], [96, 113, 'CASENO'], [127, 133, 'APP'], [159, 172, 'RESP'], [210, 220, 'A.COUNSEL']]
110007179 : [[20, 42, 'COURT'], [72, 101, 'CASENO'], [212, 241, 'CASENO'], [298, 317, 'RESP'], [342, 353, 'JUDGE']]

PASS: all Fold 3 document IDs have valid prediction lists.
Cell 27 completed. No test gold labels were used.


In [28]:
# Cell 28: Audit validation and test span lengths, compare prediction
# distributions, and inspect the longest predictions without modifying them.

from collections import defaultdict
import numpy as np

# --------------------------------------------------
# 1. Convert predictions into audit records
# --------------------------------------------------
def build_span_audit_records(predictions, text_mapping):
    records = []

    for doc_id, spans in predictions.items():
        text_value = text_mapping[doc_id]

        for start, end, label in spans:
            if not (0 <= start < end <= len(text_value)):
                raise ValueError(
                    f"Invalid span: {doc_id}, {(start, end, label)}"
                )

            records.append({
                "doc_id": doc_id,
                "start": start,
                "end": end,
                "length": end - start,
                "label": label,
                "text": text_value[start:end],
            })

    return records


S3_TEST_SPAN_RECORDS = build_span_audit_records(
    S3_FOLD3_PREDICTIONS,
    fold3_text,
)

# --------------------------------------------------
# 2. Recompute validation probabilities using the correct function signature
# --------------------------------------------------
VAL_LORMA_PROBABILITIES_AUDIT = predict_document_probabilities_lorma(
    windows=VAL_WINDOWS,
    loader=VAL_LOADER,
    model=S3_LORMA_MODEL,
)

VAL_PREDICTIONS_AUDIT = {}

for doc_id in VAL_DOC_IDS:
    scores = VAL_LORMA_PROBABILITIES_AUDIT[doc_id]

    VAL_PREDICTIONS_AUDIT[doc_id] = decode_document_fast(
        start_scores=scores["start"],
        end_scores=scores["end"],
        label_thresholds=S3_LORMA_BEST_THRESHOLDS,
    )

S3_VAL_SPAN_RECORDS = build_span_audit_records(
    VAL_PREDICTIONS_AUDIT,
    ALL_TEXTS,
)

# --------------------------------------------------
# 3. Report span length distributions and long-span examples
# --------------------------------------------------
def print_span_length_audit(name, records):
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)
    print("Total spans:", len(records))

    if not records:
        print("No spans to audit.")
        return

    lengths = np.asarray(
        [record["length"] for record in records],
        dtype=np.int64,
    )

    print("Median span length:", float(np.median(lengths)))
    print("90th percentile:", float(np.percentile(lengths, 90)))
    print("95th percentile:", float(np.percentile(lengths, 95)))
    print("Maximum span length:", int(lengths.max()))

    for limit in [30, 50, 100, 200]:
        count = int(np.sum(lengths > limit))
        print(
            f"Spans longer than {limit:3d} characters: "
            f"{count:5d} ({100 * count / len(lengths):.2f}%)"
        )

    by_label = defaultdict(list)

    for record in records:
        by_label[record["label"]].append(record["length"])

    print("\nBy-label length summary:")
    print(
        f"{'Label':12s} {'Count':>7s} {'Median':>8s} "
        f"{'Max':>8s} {'>50 chars':>10s}"
    )

    for label in LABELS:
        values = by_label[label]

        if values:
            print(
                f"{label:12s} "
                f"{len(values):7d} "
                f"{np.median(values):8.1f} "
                f"{max(values):8d} "
                f"{sum(length > 50 for length in values):10d}"
            )

    print("\nLongest 15 predicted spans:")
    longest_records = sorted(
        records,
        key=lambda record: record["length"],
        reverse=True,
    )[:15]

    for record in longest_records:
        excerpt = " ".join(record["text"].split())

        if len(excerpt) > 240:
            excerpt = excerpt[:240] + "..."

        print(
            f"doc={record['doc_id']} "
            f"label={record['label']} "
            f"span=({record['start']},{record['end']}) "
            f"length={record['length']} "
            f"text={excerpt!r}"
        )


# --------------------------------------------------
# 4. Compare validation and test distributions
# --------------------------------------------------
print_span_length_audit(
    "VALIDATION PREDICTION AUDIT",
    S3_VAL_SPAN_RECORDS,
)

print_span_length_audit(
    "FOLD 3 TEST PREDICTION AUDIT",
    S3_TEST_SPAN_RECORDS,
)

print("\nCell 28 completed. Predictions and model weights were not modified.")


VALIDATION PREDICTION AUDIT
Total spans: 2380
Median span length: 16.0
90th percentile: 244.0
95th percentile: 366.0499999999997
Maximum span length: 512
Spans longer than  30 characters:   713 (29.96%)
Spans longer than  50 characters:   575 (24.16%)
Spans longer than 100 characters:   412 (17.31%)
Spans longer than 200 characters:   275 (11.55%)

By-label length summary:
Label          Count   Median      Max  >50 chars
APP              118     15.0      509         35
RESP              89     19.0      511         28
A.COUNSEL         89     13.0      316          5
R.COUNSEL         90     13.0      511          6
JUDGE             61     13.0      260          8
WIT               60     16.5      501         23
AUTH             516     16.0      507        164
COURT            271     13.0      512         42
STAT             299     14.0      510         65
PREC             379     31.0      512        140
DATE             212     10.0      493         28
CASENO           196   

In [29]:
# Cell 29: Audit gold entity character-span lengths by split and label
# to establish evidence-based maximum span lengths for validation decoding.

from collections import defaultdict
import numpy as np

# --------------------------------------------------
# 1. Collect gold span lengths separately for train and validation
# --------------------------------------------------
def collect_gold_span_lengths(document_ids, split_name):
    lengths_by_label = defaultdict(list)
    all_lengths = []

    for doc_id in document_ids:
        text_value = ALL_TEXTS[doc_id]

        for annotation in ALL_GOLD[doc_id]:
            start, end, label = annotation[:3]

            if not (0 <= start < end <= len(text_value)):
                raise ValueError(
                    f"Invalid gold annotation in {doc_id}: {annotation}"
                )

            span_length = end - start
            lengths_by_label[label].append(span_length)
            all_lengths.append(span_length)

    print("\n" + "=" * 75)
    print(f"{split_name.upper()} GOLD ENTITY LENGTH DISTRIBUTION")
    print("=" * 75)
    print("Documents:", len(document_ids))
    print("Entities:", len(all_lengths))

    if not all_lengths:
        raise ValueError(f"No gold entities found for {split_name}.")

    all_lengths_array = np.asarray(all_lengths, dtype=np.int32)

    print("\nOverall:")
    print("Minimum:", int(all_lengths_array.min()))
    print("Median:", float(np.median(all_lengths_array)))
    print("90th percentile:", float(np.percentile(all_lengths_array, 90)))
    print("95th percentile:", float(np.percentile(all_lengths_array, 95)))
    print("99th percentile:", float(np.percentile(all_lengths_array, 99)))
    print("Maximum:", int(all_lengths_array.max()))

    print("\nBy label:")
    print(
        f"{'Label':12s} {'Count':>7s} {'Median':>8s} "
        f"{'P95':>8s} {'P99':>8s} {'Maximum':>8s}"
    )

    for label in LABELS:
        values = lengths_by_label[label]

        if not values:
            print(f"{label:12s} {'0':>7s}")
            continue

        values_array = np.asarray(values, dtype=np.int32)

        print(
            f"{label:12s} "
            f"{len(values):7d} "
            f"{np.median(values_array):8.1f} "
            f"{np.percentile(values_array, 95):8.1f} "
            f"{np.percentile(values_array, 99):8.1f} "
            f"{values_array.max():8d}"
        )

    print("\nGold spans exceeding selected lengths:")
    for limit in [30, 50, 80, 100, 150, 200, 300, 512]:
        count = int(np.sum(all_lengths_array > limit))
        print(
            f"  > {limit:3d} characters: "
            f"{count:5d} ({100 * count / len(all_lengths_array):.2f}%)"
        )

    return {
        "all_lengths": all_lengths_array,
        "lengths_by_label": dict(lengths_by_label),
    }


# --------------------------------------------------
# 2. Run the audit on training and validation annotations
# --------------------------------------------------
S3_TRAIN_GOLD_LENGTH_AUDIT = collect_gold_span_lengths(
    TRAIN_DOC_IDS,
    "training",
)

S3_VAL_GOLD_LENGTH_AUDIT = collect_gold_span_lengths(
    VAL_DOC_IDS,
    "validation",
)

print("\nCell 29 completed. No model weights or predictions were changed.")


TRAINING GOLD ENTITY LENGTH DISTRIBUTION
Documents: 56
Entities: 6532

Overall:
Minimum: 3
Median: 13.0
90th percentile: 32.0
95th percentile: 50.0
99th percentile: 82.6899999999996
Maximum: 233

By label:
Label          Count   Median      P95      P99  Maximum
APP              310     13.0     29.0     41.4       46
RESP             313     14.0     29.0     37.0       42
A.COUNSEL        152     12.0     21.0     25.5       26
R.COUNSEL        125     12.0     18.8     23.3       29
JUDGE            199     13.0     22.0     25.1       32
WIT              154     12.0     20.3     22.5       23
AUTH             775     13.0     29.0     40.3       89
COURT            769     10.0     29.0     34.3       59
STAT             955     15.0     38.3     73.0      233
PREC             912     28.0     86.4    106.0      195
DATE            1343     10.0     19.0     23.0       28
CASENO           525     20.0     38.0     47.0       91

Gold spans exceeding selected lengths:
  >  30 char

In [30]:
# Cell 30: Calibrate per-label maximum span lengths and boundary thresholds
# using validation Strict Span Macro-F1, then save the decoder configuration.

import json
import math
from pathlib import Path
import numpy as np

# --------------------------------------------------
# 1. Validate required inputs
# --------------------------------------------------
if "VAL_LORMA_PROBABILITIES_AUDIT" not in globals():
    raise RuntimeError(
        "Run the corrected Cell 28 first to compute validation probabilities."
    )

if set(VAL_LORMA_PROBABILITIES_AUDIT) != set(VAL_DOC_IDS):
    raise RuntimeError("Validation probability document IDs do not match.")

if not S3_VAL_GOLD_LENGTH_AUDIT["lengths_by_label"]:
    raise RuntimeError("Validation gold-length audit is unavailable.")

# Save the thresholds selected during training before recalibration.
S3_LORMA_TRAINING_THRESHOLDS = list(S3_LORMA_BEST_THRESHOLDS)

# --------------------------------------------------
# 2. Define an evaluation function for one label
# --------------------------------------------------
def score_one_label_on_validation(
    label_id,
    threshold,
    max_span_chars,
):
    label = ID_TO_LABEL[label_id]

    gold_label = {
        span for span in VAL_GOLD_SPANS
        if span[3] == label
    }

    predicted_label = set()

    for doc_id, scores in VAL_LORMA_PROBABILITIES_AUDIT.items():
        decoded = decode_one_label_fast(
            start_scores=scores["start"],
            end_scores=scores["end"],
            label_id=label_id,
            threshold=float(threshold),
            max_span_chars=int(max_span_chars),
        )

        predicted_label.update(
            (doc_id, start, end, predicted_label_name)
            for start, end, predicted_label_name in decoded
        )

    tp = len(gold_label & predicted_label)
    fp = len(predicted_label - gold_label)
    fn = len(gold_label - predicted_label)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    return {
        "label": label,
        "threshold": float(threshold),
        "max_span_chars": int(max_span_chars),
        "gold": len(gold_label),
        "predicted": len(predicted_label),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


# --------------------------------------------------
# 3. Propose label-specific span caps from gold lengths
# --------------------------------------------------
S3_SPAN_CAP_CANDIDATES = {}

for label_id, label in enumerate(LABELS):
    train_lengths = np.asarray(
        S3_TRAIN_GOLD_LENGTH_AUDIT["lengths_by_label"][label],
        dtype=np.int32,
    )
    val_lengths = np.asarray(
        S3_VAL_GOLD_LENGTH_AUDIT["lengths_by_label"][label],
        dtype=np.int32,
    )

    if len(train_lengths) == 0 or len(val_lengths) == 0:
        raise ValueError(f"Missing gold-length observations for {label}.")

    train_max = int(train_lengths.max())
    val_max = int(val_lengths.max())

    # Candidate limits are derived from observed gold spans.
    p95_cap = max(
        20,
        math.ceil(float(np.percentile(val_lengths, 95)) * 1.25),
    )
    p99_cap = max(
        25,
        math.ceil(float(np.percentile(val_lengths, 99)) * 1.25),
    )
    observed_max_cap = min(
        256,
        math.ceil(max(train_max, val_max) * 1.10),
    )

    candidates = sorted(set([
        p95_cap,
        p99_cap,
        observed_max_cap,
        MAX_SPAN_CHARS,
    ]))

    S3_SPAN_CAP_CANDIDATES[label] = candidates

print("=" * 70)
print("S3: PROPOSED PER-LABEL SPAN CAPS")
print("=" * 70)

for label in LABELS:
    print(f"{label:12s}: {S3_SPAN_CAP_CANDIDATES[label]}")

# --------------------------------------------------
# 4. Select the best span cap for each label at its
# existing training-selected threshold
# --------------------------------------------------
S3_CALIBRATED_SPAN_CAPS = {}
S3_SPAN_CAP_SEARCH_RESULTS = {}

for label_id, label in enumerate(LABELS):
    training_threshold = S3_LORMA_TRAINING_THRESHOLDS[label_id]

    candidate_results = []

    for cap in S3_SPAN_CAP_CANDIDATES[label]:
        result = score_one_label_on_validation(
            label_id=label_id,
            threshold=training_threshold,
            max_span_chars=cap,
        )
        candidate_results.append(result)

    # Maximize F1; prefer higher precision and then a shorter cap on ties.
    best = max(
        candidate_results,
        key=lambda result: (
            result["f1"],
            result["precision"],
            -result["max_span_chars"],
        ),
    )

    S3_CALIBRATED_SPAN_CAPS[label] = best["max_span_chars"]
    S3_SPAN_CAP_SEARCH_RESULTS[label] = candidate_results

# --------------------------------------------------
# 5. Retune each label's threshold using its selected cap
# --------------------------------------------------
S3_CALIBRATION_THRESHOLD_GRID = [
    0.05, 0.08, 0.10, 0.15,
    0.20, 0.30, 0.40, 0.50,
    0.60, 0.70, 0.80, 0.90,
]

S3_CALIBRATED_THRESHOLDS = []
S3_THRESHOLD_SEARCH_RESULTS = {}

for label_id, label in enumerate(LABELS):
    cap = S3_CALIBRATED_SPAN_CAPS[label]
    candidate_results = []

    for threshold in S3_CALIBRATION_THRESHOLD_GRID:
        result = score_one_label_on_validation(
            label_id=label_id,
            threshold=threshold,
            max_span_chars=cap,
        )
        candidate_results.append(result)

    best = max(
        candidate_results,
        key=lambda result: (
            result["f1"],
            result["precision"],
            -result["predicted"],
            result["threshold"],
        ),
    )

    S3_CALIBRATED_THRESHOLDS.append(best["threshold"])
    S3_THRESHOLD_SEARCH_RESULTS[label] = candidate_results

# --------------------------------------------------
# 6. Evaluate the combined calibrated decoder
# --------------------------------------------------
def decode_document_with_label_caps(
    start_scores,
    end_scores,
    thresholds,
    span_caps,
):
    if len(thresholds) != NUM_LABELS or len(LABELS) != NUM_LABELS:
        raise ValueError("Threshold or label count mismatch.")

    decoded = []

    for label_id, label in enumerate(LABELS):
        decoded.extend(
            decode_one_label_fast(
                start_scores=start_scores,
                end_scores=end_scores,
                label_id=label_id,
                threshold=thresholds[label_id],
                max_span_chars=span_caps[label],
            )
        )

    return sorted(decoded, key=lambda span: (span[0], span[1], span[2]))


S3_VALIDATION_PREDICTIONS_CALIBRATED = set()

for doc_id, scores in VAL_LORMA_PROBABILITIES_AUDIT.items():
    spans = decode_document_with_label_caps(
        scores["start"],
        scores["end"],
        S3_CALIBRATED_THRESHOLDS,
        S3_CALIBRATED_SPAN_CAPS,
    )

    S3_VALIDATION_PREDICTIONS_CALIBRATED.update(
        (doc_id, start, end, label)
        for start, end, label in spans
    )

S3_CALIBRATED_VALIDATION_METRICS = calculate_strict_span_metrics(
    VAL_GOLD_SPANS,
    S3_VALIDATION_PREDICTIONS_CALIBRATED,
)

# --------------------------------------------------
# 7. Compare against the previous decoder configuration
# --------------------------------------------------
S3_PREVIOUS_VALIDATION_SPANS = set()

for doc_id, scores in VAL_LORMA_PROBABILITIES_AUDIT.items():
    spans = decode_document_fast(
        scores["start"],
        scores["end"],
        S3_LORMA_TRAINING_THRESHOLDS,
    )

    S3_PREVIOUS_VALIDATION_SPANS.update(
        (doc_id, start, end, label)
        for start, end, label in spans
    )

S3_PREVIOUS_VALIDATION_METRICS = calculate_strict_span_metrics(
    VAL_GOLD_SPANS,
    S3_PREVIOUS_VALIDATION_SPANS,
)

print("\n" + "=" * 70)
print("S3: DECODER CALIBRATION RESULTS")
print("=" * 70)

print(
    "Previous validation Macro-F1:",
    f"{S3_PREVIOUS_VALIDATION_METRICS['macro_f1']:.6f}",
)
print(
    "Calibrated validation Macro-F1:",
    f"{S3_CALIBRATED_VALIDATION_METRICS['macro_f1']:.6f}",
)
print(
    "Previous predicted spans:",
    S3_PREVIOUS_VALIDATION_METRICS["predicted_count"],
)
print(
    "Calibrated predicted spans:",
    S3_CALIBRATED_VALIDATION_METRICS["predicted_count"],
)

print("\nPer-label calibrated results:")
print(
    f"{'Label':12s} {'P':>8s} {'R':>8s} {'F1':>8s} "
    f"{'TP':>6s} {'FP':>6s} {'FN':>6s} "
    f"{'Threshold':>10s} {'Cap':>6s}"
)

for label_id, label in enumerate(LABELS):
    result = S3_CALIBRATED_VALIDATION_METRICS["per_label"][label]

    print(
        f"{label:12s} "
        f"{result['precision']:8.4f} "
        f"{result['recall']:8.4f} "
        f"{result['f1']:8.4f} "
        f"{result['tp']:6d} "
        f"{result['fp']:6d} "
        f"{result['fn']:6d} "
        f"{S3_CALIBRATED_THRESHOLDS[label_id]:10.3f} "
        f"{S3_CALIBRATED_SPAN_CAPS[label]:6d}"
    )

# --------------------------------------------------
# 8. Save the decoder configuration separately from model weights
# --------------------------------------------------
S3_DECODER_CONFIG_PATH = (
    Path("/kaggle/working/S3_Architecture_C")
    / "S3_decoder_calibration.json"
)

decoder_config = {
    "experiment_id": "CS781_Phase2_S3_Architecture_C_True_LoRMA",
    "validation_strict_span_macro_f1_before_calibration": (
        S3_PREVIOUS_VALIDATION_METRICS["macro_f1"]
    ),
    "validation_strict_span_macro_f1_after_calibration": (
        S3_CALIBRATED_VALIDATION_METRICS["macro_f1"]
    ),
    "validation_predicted_spans_before": (
        S3_PREVIOUS_VALIDATION_METRICS["predicted_count"]
    ),
    "validation_predicted_spans_after": (
        S3_CALIBRATED_VALIDATION_METRICS["predicted_count"]
    ),
    "labels": LABELS,
    "thresholds": {
        label: float(S3_CALIBRATED_THRESHOLDS[i])
        for i, label in enumerate(LABELS)
    },
    "max_span_chars": S3_CALIBRATED_SPAN_CAPS,
    "cap_candidates": S3_SPAN_CAP_CANDIDATES,
}

with S3_DECODER_CONFIG_PATH.open("w", encoding="utf-8") as f:
    json.dump(decoder_config, f, ensure_ascii=False, indent=2)

print("\nSaved decoder configuration:", S3_DECODER_CONFIG_PATH)
print("Cell 30 completed. Model weights were not changed.")

S3: PROPOSED PER-LABEL SPAN CAPS
APP         : [44, 51, 512]
RESP        : [52, 92, 98, 512]
A.COUNSEL   : [24, 26, 29, 512]
R.COUNSEL   : [22, 25, 32, 512]
JUDGE       : [28, 29, 36, 512]
WIT         : [21, 25, 26, 512]
AUTH        : [38, 60, 98, 512]
COURT       : [29, 40, 65, 512]
STAT        : [47, 75, 256, 512]
PREC        : [117, 176, 215, 512]
DATE        : [23, 25, 31, 512]
CASENO      : [42, 52, 101, 512]

S3: DECODER CALIBRATION RESULTS
Previous validation Macro-F1: 0.391665
Calibrated validation Macro-F1: 0.494281
Previous predicted spans: 2380
Calibrated predicted spans: 2204

Per-label calibrated results:
Label               P        R       F1     TP     FP     FN  Threshold    Cap
APP            0.2500   0.2474   0.2487     24     72     73      0.600     44
RESP           0.2530   0.3088   0.2781     21     62     47      0.400     52
A.COUNSEL      0.3696   0.7907   0.5037     34     58      9      0.500     24
R.COUNSEL      0.5063   0.6780   0.5797     40     39     

In [31]:
# Cell 31: Apply the validation-calibrated decoder to Fold 3, verify
# the calibrated validation score, audit prediction offsets, and save prediction.json.

import json
from collections import Counter
from pathlib import Path
import numpy as np

# --------------------------------------------------
# 1. Verify that calibrated settings and test probabilities exist
# --------------------------------------------------
required_objects = [
    "S3_CALIBRATED_THRESHOLDS",
    "S3_CALIBRATED_SPAN_CAPS",
    "S3_TEST_PROBABILITIES",
    "S3_CALIBRATED_VALIDATION_METRICS",
    "TEST_DOC_IDS",
    "S3_EMPTY_TEST_DOC_IDS",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        f"Required objects are missing. Run the preceding cells first: "
        f"{missing_objects}"
    )

if len(S3_CALIBRATED_THRESHOLDS) != NUM_LABELS:
    raise ValueError("Calibrated threshold count does not match label count.")

if set(S3_CALIBRATED_SPAN_CAPS) != set(LABELS):
    raise ValueError("Calibrated span caps do not cover the full label inventory.")

expected_nonempty_ids = set(TEST_DOC_IDS) - set(S3_EMPTY_TEST_DOC_IDS)

if set(S3_TEST_PROBABILITIES) != expected_nonempty_ids:
    raise RuntimeError(
        "Test probability IDs do not match non-empty Fold 3 documents."
    )

# --------------------------------------------------
# 2. Recheck calibrated validation performance
# --------------------------------------------------
validation_predictions = set()

for doc_id, scores in VAL_LORMA_PROBABILITIES_AUDIT.items():
    decoded = decode_document_with_label_caps(
        start_scores=scores["start"],
        end_scores=scores["end"],
        thresholds=S3_CALIBRATED_THRESHOLDS,
        span_caps=S3_CALIBRATED_SPAN_CAPS,
    )

    validation_predictions.update(
        (doc_id, start, end, label)
        for start, end, label in decoded
    )

validation_metrics_final = calculate_strict_span_metrics(
    VAL_GOLD_SPANS,
    validation_predictions,
)

expected_f1 = float(
    S3_CALIBRATED_VALIDATION_METRICS["macro_f1"]
)

if not np.isclose(
    validation_metrics_final["macro_f1"],
    expected_f1,
    atol=1e-10,
):
    raise RuntimeError(
        "The calibrated validation score did not reproduce: "
        f"expected {expected_f1:.10f}, "
        f"got {validation_metrics_final['macro_f1']:.10f}."
    )

print("=" * 70)
print("S3: CALIBRATED VALIDATION RECHECK")
print("=" * 70)
print("Validation documents:", len(VAL_DOC_IDS))
print("Gold spans:", validation_metrics_final["gold_count"])
print("Predicted spans:", validation_metrics_final["predicted_count"])
print("Exact true positives:", validation_metrics_final["exact_true_positives"])
print(
    "Strict Span Macro-F1:",
    f"{validation_metrics_final['macro_f1']:.6f}",
)

# --------------------------------------------------
# 3. Decode Fold 3 with calibrated thresholds and label-specific caps
# --------------------------------------------------
S3_FOLD3_PREDICTIONS_CALIBRATED = {}

for doc_id in TEST_DOC_IDS:
    # Preserve every required document ID, including zero-token documents.
    if doc_id in S3_EMPTY_TEST_DOC_IDS:
        S3_FOLD3_PREDICTIONS_CALIBRATED[doc_id] = []
        continue

    if doc_id not in S3_TEST_PROBABILITIES:
        raise KeyError(f"No test probabilities found for document {doc_id}.")

    text_value = fold3_text[doc_id]
    scores = S3_TEST_PROBABILITIES[doc_id]

    if scores["start"].shape != (len(text_value), NUM_LABELS):
        raise ValueError(
            f"Start-probability shape mismatch for document {doc_id}."
        )

    if scores["end"].shape != (len(text_value), NUM_LABELS):
        raise ValueError(
            f"End-probability shape mismatch for document {doc_id}."
        )

    decoded = decode_document_with_label_caps(
        start_scores=scores["start"],
        end_scores=scores["end"],
        thresholds=S3_CALIBRATED_THRESHOLDS,
        span_caps=S3_CALIBRATED_SPAN_CAPS,
    )

    verified_spans = []

    for start, end, label in decoded:
        if not (0 <= start < end <= len(text_value)):
            raise ValueError(
                f"Invalid span in {doc_id}: {(start, end, label)}."
            )

        if label not in LABEL_TO_ID:
            raise ValueError(f"Unknown label {label!r}.")

        if end - start > S3_CALIBRATED_SPAN_CAPS[label]:
            raise ValueError(
                f"Span exceeds configured cap for {label} in {doc_id}."
            )

        if not text_value[start:end].strip():
            raise ValueError(
                f"Whitespace-only entity predicted in document {doc_id}."
            )

        verified_spans.append([int(start), int(end), label])

    S3_FOLD3_PREDICTIONS_CALIBRATED[doc_id] = verified_spans

# --------------------------------------------------
# 4. Validate the final output structure and summarize predictions
# --------------------------------------------------
if set(S3_FOLD3_PREDICTIONS_CALIBRATED) != set(fold3_text):
    missing_ids = set(fold3_text) - set(S3_FOLD3_PREDICTIONS_CALIBRATED)
    extra_ids = set(S3_FOLD3_PREDICTIONS_CALIBRATED) - set(fold3_text)

    raise RuntimeError(
        f"Final document-ID mismatch. Missing={list(missing_ids)[:5]}, "
        f"extra={list(extra_ids)[:5]}"
    )

prediction_count_by_label = Counter()
span_lengths = []
total_predictions = 0

for doc_id, spans in S3_FOLD3_PREDICTIONS_CALIBRATED.items():
    if not isinstance(spans, list):
        raise TypeError(f"Prediction list for {doc_id} is invalid.")

    seen = set()

    for span in spans:
        if not isinstance(span, list) or len(span) != 3:
            raise ValueError(
                f"Prediction must be [start, end, label], got {span!r}."
            )

        start, end, label = span
        key = (start, end, label)

        if key in seen:
            raise ValueError(f"Duplicate prediction in document {doc_id}: {key}")

        seen.add(key)
        prediction_count_by_label[label] += 1
        span_lengths.append(end - start)
        total_predictions += 1

print("\n" + "=" * 70)
print("S3: CALIBRATED FOLD 3 PREDICTION AUDIT")
print("=" * 70)
print("Test documents:", len(TEST_DOC_IDS))
print("Documents in output:", len(S3_FOLD3_PREDICTIONS_CALIBRATED))
print("Total predicted spans:", total_predictions)
print("Empty documents preserved:", len(S3_EMPTY_TEST_DOC_IDS))

print("\nPredictions per label:")
for label in LABELS:
    print(f"  {label:12s}: {prediction_count_by_label[label]}")

if span_lengths:
    span_lengths_array = np.asarray(span_lengths)
    print("\nPredicted span lengths:")
    print("Median:", float(np.median(span_lengths_array)))
    print("95th percentile:", float(np.percentile(span_lengths_array, 95)))
    print("Maximum:", int(span_lengths_array.max()))

# --------------------------------------------------
# 5. Save the final prediction JSON
# --------------------------------------------------
S3_PREDICTION_PATH = (
    Path("/kaggle/working/S3_Architecture_C")
    / "prediction.json"
)

S3_PREDICTION_PATH.parent.mkdir(parents=True, exist_ok=True)

with S3_PREDICTION_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        S3_FOLD3_PREDICTIONS_CALIBRATED,
        f,
        ensure_ascii=False,
        separators=(",", ":"),
    )

# Read back the saved file and verify it matches the in-memory object.
with S3_PREDICTION_PATH.open("r", encoding="utf-8") as f:
    reloaded_predictions = json.load(f)

if reloaded_predictions != S3_FOLD3_PREDICTIONS_CALIBRATED:
    raise RuntimeError("Saved prediction JSON differs from the in-memory output.")

print("\nPrediction JSON saved:", S3_PREDICTION_PATH)
print("File size:", f"{S3_PREDICTION_PATH.stat().st_size:,} bytes")
print("JSON round-trip verification: PASS")
print("Cell 31 completed. No test gold labels were used.")

S3: CALIBRATED VALIDATION RECHECK
Validation documents: 14
Gold spans: 1861
Predicted spans: 2204
Exact true positives: 1062
Strict Span Macro-F1: 0.494281

S3: CALIBRATED FOLD 3 PREDICTION AUDIT
Test documents: 35
Documents in output: 35
Total predicted spans: 4969
Empty documents preserved: 1

Predictions per label:
  APP         : 159
  RESP        : 192
  A.COUNSEL   : 232
  R.COUNSEL   : 193
  JUDGE       : 107
  WIT         : 197
  AUTH        : 936
  COURT       : 519
  STAT        : 866
  PREC        : 589
  DATE        : 697
  CASENO      : 282

Predicted span lengths:
Median: 12.0
95th percentile: 48.0
Maximum: 117

Prediction JSON saved: /kaggle/working/S3_Architecture_C/prediction.json
File size: 102,746 bytes
JSON round-trip verification: PASS
Cell 31 completed. No test gold labels were used.


In [32]:
# Cell 32: Package the S3 submission and reproducibility artifacts.

import json
import zipfile
import hashlib
from pathlib import Path

S3_OUTPUT_DIR = Path("/kaggle/working/S3_Architecture_C")
S3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

S3_PREDICTION_PATH = S3_OUTPUT_DIR / "prediction.json"
S3_SUBMISSION_ZIP = Path(
    "/kaggle/working/CS781_P2_S3_Architecture_C_Submission.zip"
)
S3_ARTIFACTS_ZIP = Path(
    "/kaggle/working/CS781_P2_S3_Architecture_C_Artifacts.zip"
)

# --------------------------------------------------
# 1. Verify the prediction file
# --------------------------------------------------
if not S3_PREDICTION_PATH.exists():
    raise FileNotFoundError(
        f"Prediction JSON not found: {S3_PREDICTION_PATH}"
    )

with S3_PREDICTION_PATH.open("r", encoding="utf-8") as f:
    saved_predictions = json.load(f)

if set(saved_predictions) != set(fold3_text):
    raise ValueError("Prediction file does not contain exactly the Fold 3 IDs.")

if len(saved_predictions) != 35:
    raise ValueError(
        f"Expected 35 Fold 3 documents, found {len(saved_predictions)}."
    )

prediction_hash = hashlib.sha256(
    S3_PREDICTION_PATH.read_bytes()
).hexdigest()

# --------------------------------------------------
# 2. Create the submission ZIP with prediction.json at its root
# --------------------------------------------------
if S3_SUBMISSION_ZIP.exists():
    S3_SUBMISSION_ZIP.unlink()

with zipfile.ZipFile(
    S3_SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    zf.write(S3_PREDICTION_PATH, arcname="prediction.json")

with zipfile.ZipFile(S3_SUBMISSION_ZIP, "r") as zf:
    zip_names = zf.namelist()

    if zip_names != ["prediction.json"]:
        raise RuntimeError(
            f"Unexpected submission ZIP structure: {zip_names}"
        )

    zipped_predictions = json.loads(
        zf.read("prediction.json").decode("utf-8")
    )

    if zipped_predictions != saved_predictions:
        raise RuntimeError("ZIP prediction content failed verification.")

# --------------------------------------------------
# 3. Collect available S3 reproducibility files
# --------------------------------------------------
artifact_files = [
    S3_PREDICTION_PATH,
    S3_OUTPUT_DIR / "S3_decoder_calibration.json",
    S3_OUTPUT_DIR / "S3_LoRMA_multiplicative_history.json",
    S3_OUTPUT_DIR / "S3_additive_adapter_fallback.pt",
]

# Check common output locations for the best true multiplicative LoRMA model.
checkpoint_candidates = list(
    Path("/kaggle/working").glob(
        "**/S3_LoRMA_multiplicative_best.pt"
    )
)

# Also include a checkpoint if the notebook has recorded its path.
for variable_name in [
    "S3_LORMA_BEST_CHECKPOINT_PATH",
    "S3_LORMA_CHECKPOINT_PATH",
]:
    candidate = globals().get(variable_name)
    if candidate is not None:
        candidate_path = Path(str(candidate))
        if candidate_path.exists():
            checkpoint_candidates.append(candidate_path)

# Deduplicate candidates while preserving order.
seen_paths = set()
checkpoint_candidates = [
    p for p in checkpoint_candidates
    if not (str(p.resolve()) in seen_paths or seen_paths.add(str(p.resolve())))
]

if checkpoint_candidates:
    artifact_files.append(checkpoint_candidates[0])

# Keep only files that exist; do not invent missing artifacts.
artifact_files = list(dict.fromkeys(
    p.resolve() for p in artifact_files if p.exists()
))

# Record the settings and exact source prediction hash.
def make_json_safe(value):
    if isinstance(value, dict):
        return {str(k): make_json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(v) for v in value]
    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            pass
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)

S3_RUN_MANIFEST = {
    "experiment": "S3_Architecture_C",
    "method": (
        "Frozen Phase 1 Architecture C LSTM with multiplicative "
        "low-rank recurrent-weight adaptation and character-level "
        "start/end boundary heads"
    ),
    "validation_strict_span_macro_f1": float(
        S3_CALIBRATED_VALIDATION_METRICS["macro_f1"]
    ),
    "validation_documents": len(VAL_DOC_IDS),
    "test_documents": len(saved_predictions),
    "test_predicted_spans": sum(
        len(spans) for spans in saved_predictions.values()
    ),
    "labels": list(LABELS),
    "calibrated_thresholds": make_json_safe(S3_CALIBRATED_THRESHOLDS),
    "calibrated_span_caps": make_json_safe(S3_CALIBRATED_SPAN_CAPS),
    "prediction_sha256": prediction_hash,
    "phase1_backbone_checkpoint": str(ARCH_C_FINAL_CHECKPOINT),
    "notes": [
        "Fold 3 gold labels were not used.",
        "Validation calibration used the 14-document validation split.",
        "The validation score is not an official test or leaderboard score.",
        "The submission ZIP contains prediction.json only."
    ],
}

S3_MANIFEST_PATH = S3_OUTPUT_DIR / "S3_run_manifest.json"
with S3_MANIFEST_PATH.open("w", encoding="utf-8") as f:
    json.dump(S3_RUN_MANIFEST, f, ensure_ascii=False, indent=2)

artifact_files.append(S3_MANIFEST_PATH)

# Add the true multiplicative adapter checkpoint if found.
if not checkpoint_candidates:
    print(
        "Note: a separate saved multiplicative LoRMA checkpoint was not "
        "found automatically; the trained checkpoint may need to be saved "
        "explicitly from the notebook before producing a fully reproducible run."
    )

# --------------------------------------------------
# 4. Create the separate reproducibility archive
# --------------------------------------------------
if S3_ARTIFACTS_ZIP.exists():
    S3_ARTIFACTS_ZIP.unlink()

with zipfile.ZipFile(
    S3_ARTIFACTS_ZIP, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    for path in artifact_files:
        # Use unique basenames to keep the archive easy to inspect.
        zf.write(path, arcname=path.name)

with zipfile.ZipFile(S3_ARTIFACTS_ZIP, "r") as zf:
    artifact_names = zf.namelist()

    if "prediction.json" not in artifact_names:
        raise RuntimeError("Reproducibility archive is missing prediction.json.")

    if "S3_run_manifest.json" not in artifact_names:
        raise RuntimeError("Reproducibility archive is missing its manifest.")

# --------------------------------------------------
# 5. Report verified outputs
# --------------------------------------------------
print("=" * 70)
print("S3 PACKAGING COMPLETE")
print("=" * 70)
print("Submission ZIP:", S3_SUBMISSION_ZIP)
print("Submission ZIP size:", f"{S3_SUBMISSION_ZIP.stat().st_size:,} bytes")
print("Submission ZIP contents:", zip_names)
print()
print("Artifacts ZIP:", S3_ARTIFACTS_ZIP)
print("Artifacts ZIP size:", f"{S3_ARTIFACTS_ZIP.stat().st_size:,} bytes")
print("Artifacts included:")
for name in artifact_names:
    print(" -", name)

print("\nPrediction SHA-256:", prediction_hash)
print("Submission archive verification: PASS")
print("Artifacts archive verification: PASS")
print("Cell 32 completed.")

S3 PACKAGING COMPLETE
Submission ZIP: /kaggle/working/CS781_P2_S3_Architecture_C_Submission.zip
Submission ZIP size: 31,679 bytes
Submission ZIP contents: ['prediction.json']

Artifacts ZIP: /kaggle/working/CS781_P2_S3_Architecture_C_Artifacts.zip
Artifacts ZIP size: 5,070,252 bytes
Artifacts included:
 - prediction.json
 - S3_decoder_calibration.json
 - S3_LoRMA_multiplicative_history.json
 - S3_additive_adapter_fallback.pt
 - S3_LoRMA_multiplicative_best.pt
 - S3_run_manifest.json

Prediction SHA-256: 9a6cf6e6c335755e44b13bc907fdf3c5291f9f6122cf89a82bcdd431692779c3
Submission archive verification: PASS
Artifacts archive verification: PASS
Cell 32 completed.
